# Точный Session6 и блендинг

Подключите распакованный `Session6_exact_20261111.zip`, исходный output `blend-best-2` и прежние два датасета Session6 и XGB rich. Включите GPU для остальных баз бленда.

Сначала исполняется проверочный рецепт из архива. Он восстанавливает финальные CatBoost из raw-данных и весов, модели остатков, LightGBM и формулу Session6. Локальные и конкурсные прогнозы сравниваются построчно с каноническими. Часть оценочных CatBoost-весов отсутствует, поэтому локальное воспроизведение использует сохранённые OOS-прогнозы этих баз. Это воспроизведение сохранённого решения, а не повторение всего обучения с нуля. Public 0.326 сообщён автором, повторная отправка автоматически не выполняется.

После проверки строится бленд по прежнему фиксированному cat_d4__ridge. Для локального test используется проверенный точный Session6. Для конкурсного прогноза используется точный финальный Session6 из архива. Исторический рецепт замораживает LightGBM и переобучает CatBoost и модели остатков. Этот режим явно запрошен для точного воспроизведения.

Два CSV бленда сохраняют прежние имена train_val и full_history для совместимости. Эти названия теперь описывают обучение остальных баз и метамодели. В обоих конкурсных CSV Session6 использует свой канонический финальный комплект. Поэтому первый CSV представляет гибрид с финальным Session6, а его локальная оценка использует исторический оценочный комплект Session6. Оценка test получена до полного переобучения остальных компонентов. Период test уже изучался.

В рабочую папку копируется комплект архива, чтобы исходные датасеты оставались доступными для проверки контрольных сумм. Потребуется место под распакованный архив. Ошибка любого построчного сравнения останавливает сборку.


In [ ]:
from pathlib import Path
import shutil
import os
EXACT_DATA_DIR = None
candidates = [Path(EXACT_DATA_DIR)] if EXACT_DATA_DIR else [p.parent for p in Path('/kaggle/input').rglob('ANSWERS_RU.md') if (p.parent / 'session6_bundle/manifest.json').is_file()]
if len(candidates) != 1:
    raise ValueError(f'Укажите EXACT_DATA_DIR, найдено {candidates}')
exact_work = Path('/kaggle/working/session6_exact_replay')
shutil.copytree(candidates[0], exact_work, dirs_exist_ok=True)
verification_scope = {'__name__': '__session6_verification__'}


In [ ]:
exec(compile("from pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport sys\nimport zipfile\nBASE_DIR = None\nREBUILD_BASE_PREDICTIONS = True\nVERIFY_VALIDATION = True\nREFIT_CALIBRATORS = False\nBASE = Path('/kaggle/working/session6_exact_replay')\nBUNDLE = BASE / 'stepa_checkpoint_0328_bundle'\nOUT = BASE / 'outputs/reproduced_0328'\nOUT.mkdir(parents=True, exist_ok=True)\nDATA = BUNDLE / 'data'\nmanifest = json.loads((BUNDLE / 'bundle_manifest.json').read_text())\n\ndef sha256(path):\n    with Path(path).open('rb') as stream:\n        return hashlib.file_digest(stream, 'sha256').hexdigest()\nmissing = [name for name in ['numpy', 'pandas', 'pyarrow', 'catboost', 'scipy'] if importlib.util.find_spec(name) is None]\nif missing:\n    raise RuntimeError('В ядре отсутствуют ' + ', '.join(missing) + '. Выберите ядро проекта или установите requirements-reproduce.txt из комплекта.')\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor\nfrom IPython.display import display\nfor relative, expected in manifest['files'].items():\n    path = BUNDLE / relative\n    if not path.is_file() or sha256(path) != expected:\n        raise ValueError('Файл отсутствует или повреждён: ' + relative)\nif not sha256(BUNDLE / 'reference/submission_0328.csv') == manifest['original_submission_sha256']:\n    raise ValueError('Session6 verification failed')\nprint('Комплект проверен:', BUNDLE)\nprint('Чекпоинт:', manifest['checkpoint'], '| Public:', manifest['public_score_reported_by_user'])\noriginal_environment = json.loads((BUNDLE / 'environment.json').read_text())\ndisplay(pd.DataFrame([{'пакет': p, 'версия сервера': v, 'текущее ядро': importlib.metadata.version(p)} for p, v in original_environment['packages'].items() if importlib.util.find_spec(p) is not None]))", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("sys.path.insert(0, str(BUNDLE / 'code'))\nimport winning_strategy as strategy\nhistory = pd.read_parquet(BUNDLE / 'inputs/selected_evaluation.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\nif not (len(history) == 224000 and history.cutoff.nunique() == 28):\n    raise ValueError('Session6 verification failed')\nif not not history.duplicated(['cutoff', 'horizon', 'route_id']).any():\n    raise ValueError('Session6 verification failed')\nif not (history.groupby(['cutoff', 'horizon']).size() == 1000).all():\n    raise ValueError('Session6 verification failed')\ntuning = history.loc[history.split.eq('tuning')].copy()\nholdout = history.loc[history.split.eq('holdout')].copy()\nif not tuning.timestamp.max() + pd.Timedelta(hours=4) < holdout.cutoff.min():\n    raise ValueError('Session6 verification failed')\n\ndef load_calibrator(phase):\n    folder = BUNDLE / 'models' / f'{phase}_calibrator'\n    info = json.loads((folder / 'metadata.json').read_text())\n    if not info['combination_weights'] == {'shallow': 0.5, 'deep': 0.5}:\n        raise ValueError('Session6 verification failed')\n    known = tuning if phase == 'evaluation' else history\n    model = {}\n    for name, metadata in info['components'].items():\n        estimator = CatBoostRegressor()\n        estimator.load_model(str(folder / f'{name}.cbm'))\n        if not estimator.feature_names_ == metadata['feature_names']:\n            raise ValueError('Session6 verification failed')\n        model[name] = dict(model=estimator, tuning=known.copy(), initial_offset=metadata['initial_offset'], fit_through=metadata['fit_through'])\n    return model\nevaluation_calibrator = load_calibrator('evaluation')\nfinal_calibrator = load_calibrator('final')\ndisplay(pd.DataFrame([{'этап': phase, 'модель': name, 'деревья': item['model'].tree_count_, 'последняя метка fit': item['fit_through']} for phase, model in [('оценка', evaluation_calibrator), ('submission', final_calibrator)] for name, item in model.items()]))", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile('future = pd.read_parquet(BUNDLE / \'inputs/final_predictions.parquet\')\nkeys = pd.read_parquet(DATA / \'test_solo_track.parquet\')\nif not (len(future) == 8000 and future.id.is_unique):\n    raise ValueError(\'Session6 verification failed\')\nif REBUILD_BASE_PREDICTIONS:\n    for name, expected in manifest[\'data_sha256\'].items():\n        if sha256(DATA / name) != expected:\n            raise ValueError(\'Не совпал исходный parquet: \' + name)\n    base_work = OUT / \'base_inference\'\n    base_work.mkdir(exist_ok=True)\n    scope = {\'__name__\': \'__baseline_reproduction__\'}\n    exec(compile((BUNDLE / \'code/baseline_definitions.py\').read_text(), \'baseline_definitions.py\', \'exec\'), scope)\n    scope.update(ROOT=base_work, TRAIN_PATH=DATA / \'train_solo_track.parquet\', TEST_PATH=DATA / \'test_solo_track.parquet\')\n    scope[\'load_panel\']()\n    for name in [\'baseline_common_features.py\', \'baseline_correlations.py\', \'baseline_horizon.py\']:\n        exec(compile((BUNDLE / \'code\' / name).read_text(), name, \'exec\'), scope)\n    scope[\'validation_lag_table\'] = pd.read_csv(BUNDLE / \'inputs/status6_lags_final.csv\')\n    rebuilt = []\n    origin = np.array([len(scope[\'times\']) - 1])\n    for h in range(1, 9):\n        folder = BUNDLE / \'models/baseline\' / f\'h{h}\'\n        metadata = json.loads((folder / \'complete.json\').read_text())\n        features = scope[\'add_pilot_features\'](scope[\'horizon_dataset\'](h, origin), h, origin)\n        requests = scope[\'test\'].loc[scope[\'test\'].timestamp.eq(scope[\'end\'] + h * scope[\'STEP\'])]\n        features = requests.merge(features, on=[\'route_id\', \'timestamp\'], validate=\'one_to_one\', sort=False)\n        estimator = CatBoostRegressor()\n        estimator.load_model(str(folder / \'model.cbm\'))\n        if not estimator.feature_names_ == metadata[\'features\']:\n            raise ValueError(\'Session6 verification failed\')\n        part = features[[\'id\', \'route_id\', \'timestamp\', \'profile28\']].copy()\n        part[\'model_prediction\'] = np.maximum(estimator.predict(features[metadata[\'features\']]) * 100000.0, 0)\n        rebuilt.append(part)\n        print(f"h{h}: {len(part)} маршрутов, {metadata[\'trees\']} деревьев")\n    rebuilt = pd.concat(rebuilt, ignore_index=True)\n    checked = future[[\'id\', \'model_prediction\', \'profile28\']].merge(rebuilt[[\'id\', \'model_prediction\', \'profile28\']], on=\'id\', validate=\'one_to_one\', suffixes=(\'_saved\', \'_new\'))\n    if not len(checked) == 8000:\n        raise ValueError(\'Session6 verification failed\')\n    np.testing.assert_allclose(checked.model_prediction_saved, checked.model_prediction_new, rtol=1e-10, atol=1e-06)\n    np.testing.assert_allclose(checked.profile28_saved, checked.profile28_new, rtol=1e-07, atol=0.05)\n    print(\'Максимальное отличие базового прогноза:\', float(np.abs(checked.model_prediction_saved - checked.model_prediction_new).max()))\n    lookup = rebuilt.set_index(\'id\').reindex(future.id)\n    future[\'model_prediction\'] = lookup.model_prediction.to_numpy()\n    future[\'profile28\'] = lookup.profile28.to_numpy()\n    future[\'raw_blend\'] = np.where(future.horizon.eq(1), future.model_prediction, 0.7 * future.model_prediction + 0.3 * future.profile28)\n    del scope, rebuilt, lookup, features, estimator\n    import gc\n    gc.collect()\nelse:\n    print(\'Используются сохранённые финальные базовые прогнозы из проверенного комплекта.\')', 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("def metric(frame, prediction):\n    y = frame.y.to_numpy(dtype=np.float64)\n    p = np.asarray(prediction, dtype=np.float64)\n    if not (y.shape == p.shape and np.isfinite(p).all() and (p >= 0).all()):\n        raise ValueError('Session6 verification failed')\n    if y.sum() <= 0:\n        raise ValueError('Нулевой знаменатель')\n    error = p - y\n    daily = pd.DataFrame({'day': frame.cutoff.to_numpy(), 'y': y, 'absolute': np.abs(error), 'signed': error}).groupby('day').sum()\n    if (daily.y <= 0).any():\n        raise ValueError('Нулевой знаменатель дневной метрики')\n    return dict(WAPE=float(np.abs(error).sum() / y.sum()), Rbias=float(error.sum() / y.sum()), score=float((np.abs(error).sum() + abs(error.sum())) / y.sum()), daily_score=float(((daily.absolute + daily.signed.abs()) / daily.y).mean()))\nif VERIFY_VALIDATION:\n    evaluation_prediction = np.maximum(strategy.predict(evaluation_calibrator, history, BUNDLE), 0)\n    reference = pd.read_parquet(BUNDLE / 'reference/evaluation_prediction_values.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\n    if not np.array_equal(reference[['cutoff', 'horizon', 'route_id']].to_numpy(), history[['cutoff', 'horizon', 'route_id']].to_numpy()):\n        raise ValueError('Session6 verification failed')\n    np.testing.assert_allclose(evaluation_prediction, reference.calibrated_prediction, rtol=1e-12, atol=1e-06)\n    rows = []\n    mask = history.split.eq('holdout').to_numpy()\n    for label, days in [('holdout', np.sort(holdout.cutoff.unique())), ('неделя 1', np.sort(holdout.cutoff.unique())[:7]), ('неделя 2', np.sort(holdout.cutoff.unique())[7:])]:\n        selected = history.cutoff.isin(days).to_numpy()\n        rows.append({'период': label, 'горизонт': 0, **metric(history.loc[selected], evaluation_prediction[selected])})\n    for h in range(1, 9):\n        selected = mask & history.horizon.eq(h).to_numpy()\n        rows.append({'период': 'holdout', 'горизонт': h, **metric(history.loc[selected], evaluation_prediction[selected])})\n    validation_metrics = pd.DataFrame(rows)\n    np.testing.assert_allclose(validation_metrics.iloc[0]['score'], manifest['local_pooled_score'], rtol=0, atol=1e-12)\n    np.testing.assert_allclose(validation_metrics.iloc[0]['daily_score'], manifest['local_daily_score'], rtol=0, atol=1e-12)\n    display(validation_metrics)\n    validation_metrics.to_csv(OUT / 'validation_metrics_reproduced.csv', index=False)\n    cutoff = pd.Timestamp('2025-10-25 10:30')\n    prefix = history.loc[history.cutoff.le(cutoff)].copy()\n    prefix.loc[prefix.cutoff.eq(cutoff), 'y'] = np.nan\n    prefix_prediction = strategy.predict(evaluation_calibrator, prefix, BUNDLE)\n    np.testing.assert_allclose(prefix_prediction[prefix.cutoff.eq(cutoff)], evaluation_prediction[history.cutoff.eq(cutoff)], rtol=1e-12, atol=1e-06)\n    print('Проверка неизвестных текущих и будущих меток пройдена.')", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("future['split'] = 'submission'\nif not history.timestamp.max() + pd.Timedelta(hours=4) < future.cutoff.min():\n    raise ValueError('Session6 verification failed')\nprediction = np.maximum(strategy.predict(final_calibrator, future, BUNDLE), 0)\ncandidate = future[['id']].copy()\ncandidate['y_pred'] = prediction\nsubmission = keys[['id']].merge(candidate, on='id', how='left', sort=False, validate='one_to_one')\nif not (len(submission) == 8000 and submission.id.is_unique):\n    raise ValueError('Session6 verification failed')\nif not submission.id.equals(keys.id.reset_index(drop=True)):\n    raise ValueError('Session6 verification failed')\nif not (np.isfinite(submission.y_pred).all() and (submission.y_pred >= 0).all()):\n    raise ValueError('Session6 verification failed')\nreference_submission = pd.read_csv(BUNDLE / 'reference/submission_0328.csv')\nif not submission.id.equals(reference_submission.id):\n    raise ValueError('Session6 verification failed')\nnp.testing.assert_allclose(submission.y_pred, reference_submission.y_pred, rtol=1e-12, atol=1e-06)\ndestination = OUT / 'submission_reproduced.csv'\nsubmission.to_csv(destination, index=False)\nverification = dict(checkpoint=manifest['checkpoint'], public_score_reported_by_user=0.328, rows=len(submission), numeric_predictions_match=True, max_absolute_difference=float(np.abs(submission.y_pred - reference_submission.y_pred).max()), original_sha256=manifest['original_submission_sha256'], reproduced_sha256=sha256(destination))\nverification['byte_identical'] = verification['original_sha256'] == verification['reproduced_sha256']\n(OUT / 'verification.json').write_text(json.dumps(verification, ensure_ascii=False, indent=2))\ndisplay(submission.head())\nprint('Сохранено:', destination)\nprint(json.dumps(verification, ensure_ascii=False, indent=2))", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("import joblib, importlib.util, gc\nfeature_path = BASE / 'prepared_features/features_cb101246dc27_starts5.parquet'\ncutoffs = pd.read_parquet(feature_path, columns=['cutoff'])\nchoose = cutoffs.cutoff.ge('2025-10-04') & cutoffs.cutoff.lt('2025-11-01')\nd = pd.read_parquet(feature_path, filters=[('cutoff', '>=', pd.Timestamp('2025-10-04')), ('cutoff', '<', pd.Timestamp('2025-11-01'))])\nextra = pd.read_parquet(BASE / 'prepared_features/extra_long_5_0_f3390bfbe3_6a081832b4f4.parquet').loc[choose].reset_index(drop=True)\nif not len(d) == len(extra) == 224000:\n    raise ValueError('Session6 verification failed')\nd = pd.concat([d.reset_index(drop=True), extra], axis=1)\nspec = importlib.util.spec_from_file_location('verify_robust', BASE / 'session6_bundle/code/core_robust.py')\nrobust = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(robust)\ncfg = json.loads((BASE / 'server_output/lightgbm86/config.json').read_text())\nd = robust.extend_robust(d, cfg)\npred = np.empty(len(d))\nfor h in range(1, 9):\n    item = joblib.load(BASE / f'server_output/lightgbm86/h{h}/model_bundle.joblib')\n    use = d.horizon.eq(h).to_numpy()\n    pred[use] = np.maximum(item['model'].predict(d.loc[use, item['columns']]) * d.loc[use, 'scale'].to_numpy(), 0)\nnew = d[['cutoff', 'horizon', 'route_id']].copy()\nnew['core_reproduced'] = pred\nsaved_core = pd.read_parquet(BASE / 'server_output/lightgbm86/predictions.parquet')\ncomp = saved_core.merge(new, on=['cutoff', 'horizon', 'route_id'], validate='one_to_one')\nif not len(comp) == 224000:\n    raise ValueError('Session6 verification failed')\nnp.testing.assert_allclose(comp.core_prediction, comp.core_reproduced, rtol=1e-12, atol=1e-06)\nchecks = {'lightgbm_local_rows': len(comp), 'lightgbm_local_max_error': float(abs(comp.core_prediction - comp.core_reproduced).max()), 'catboost_evaluation': {}}\nfor h in [1, 3, 7]:\n    folder = BASE / 'server_output/original_catboost_run'\n    f = pd.read_parquet(folder / f'features/h{h}.parquet', filters=[('split', '!=', 'train')])\n    meta = json.loads((folder / f'evaluation/h{h}/complete.json').read_text())\n    model = CatBoostRegressor()\n    model.load_model(str(folder / f'evaluation/h{h}/model.cbm'))\n    f['reproduced'] = np.maximum(model.predict(f[meta['features']]) * 100000.0, 0)\n    original = pd.read_parquet(folder / f'evaluation/h{h}/predictions.parquet')\n    z = original.merge(f[['cutoff', 'route_id', 'reproduced']], on=['cutoff', 'route_id'], validate='one_to_one')\n    np.testing.assert_allclose(z.prediction, z.reproduced, rtol=1e-12, atol=1e-06)\n    checks['catboost_evaluation'][str(h)] = {'rows': len(z), 'max_error': float(abs(z.prediction - z.reproduced).max())}\n(BASE / 'reports/base_models_local_reproduction.json').write_text(json.dumps(checks, indent=2))\nprint(json.dumps(checks, indent=2))\ndel d, extra, cutoffs, new, comp, f, z, model, item\n_ = gc.collect()", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("from scipy.optimize import minimize_scalar\ncfg = json.loads((BASE / 'session6_bundle/manifest.json').read_text())\nweight = cfg['shape_weight']\nlight = pd.read_parquet(BASE / 'server_output/lightgbm86/predictions.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\nold = pd.read_parquet(BASE / 'server_output/baseline_iteration10/predictions.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\ncross = pd.read_parquet(BASE / 'prepared_features/reference_crossfit_4to10.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\nsaved = pd.read_parquet(BASE / 'server_output/session6_iteration06/predictions.parquet').sort_values(['cutoff', 'horizon', 'route_id']).reset_index(drop=True)\nfor other in [old, cross, saved]:\n    np.testing.assert_array_equal(light[['cutoff', 'horizon', 'route_id', 'y']], other[['cutoff', 'horizon', 'route_id', 'y']])\nnp.testing.assert_allclose(evaluation_prediction, old.calibrated_prediction, rtol=1e-12, atol=1e-06)\nbase = old.calibrated_prediction.to_numpy(dtype=float).copy()\ntune = history.split.eq('tuning').to_numpy()\nbase[tune] = cross.loc[tune, 'calibrated_prediction'].to_numpy(dtype=float)\nshape = light.core_prediction.to_numpy(dtype=float).copy()\nfor _, ix in light.groupby(['cutoff', 'horizon']).indices.items():\n    shape[ix] *= base[ix].sum() / shape[ix].sum()\nlocal = (1 - weight) * base + weight * shape\nnp.testing.assert_allclose(local, saved.session_prediction, rtol=0, atol=1e-08)\ncal = light.cutoff.between('2025-10-11', '2025-10-17 23:59').to_numpy()\n\ndef objective(w):\n    return metric(light.loc[cal], ((1 - w) * base + w * shape)[cal])['daily_score'] + 0.002 * w * w\nopt = minimize_scalar(objective, bounds=(0, 0.75), method='bounded')\nreplayed = min([0.0, 0.75, float(opt.x)], key=objective)\nif not abs(replayed - weight) < 1e-10:\n    raise ValueError('Session6 verification failed')\narchive = pd.read_parquet(BASE / 'validation_predictions.parquet')\nkey = ['cutoff', 'horizon', 'route_id']\nprint('Archive columns:', list(archive))\ncolumn = 'session_prediction' if 'session_prediction' in archive else 'prediction'\nactual = light[key].copy()\nactual['reproduced_prediction'] = local\njoined = archive.merge(actual, on=key, validate='one_to_one', how='left')\nif not (len(joined) == 112000 and joined.reproduced_prediction.notna().all()):\n    raise ValueError('Session6 verification failed')\nnp.testing.assert_allclose(joined[column], joined.reproduced_prediction, rtol=0, atol=1e-08)\nmeasured = metric(joined, joined.reproduced_prediction)\nif not abs(measured['score'] - 0.3191901999528053) < 1e-12:\n    raise ValueError('Session6 verification failed')\njoined['absolute_difference'] = abs(joined[column] - joined.reproduced_prediction)\njoined.to_parquet(BASE / 'outputs/local_test_row_comparison.parquet', index=False)\nreport = {'rows': len(joined), 'local_score': measured, 'weight': weight, 'replayed_weight': replayed, 'max_absolute_error': float(joined.absolute_difference.max()), 'archive_sha256': sha256(BASE / 'validation_predictions.parquet')}\n(BASE / 'reports/local_test_reproduction.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))\nprint(json.dumps(report, ensure_ascii=False, indent=2))\ndisplay(joined.head())", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exec(compile("import importlib.util\nspec = importlib.util.spec_from_file_location('session6_replay', BASE / 'session6_bundle/predict_session.py')\nruntime = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(runtime)\nfinal = runtime.predict(BASE / 'session6_bundle', DATA, BASE / 'outputs/lightgbm_cache')\ncanonical = pd.read_csv(BASE / 'submission_session6.csv')\nnp.testing.assert_array_equal(final.id, canonical.id)\nnp.testing.assert_allclose(final.y_pred, canonical.y_pred, rtol=1e-12, atol=1e-06)\nif not (len(final) == 8000 and final.id.is_unique and np.isfinite(final.y_pred).all() and final.y_pred.ge(0).all()):\n    raise ValueError('Session6 verification failed')\nfinal.to_csv(BASE / 'outputs/submission_session6_reproduced.csv', index=False)\ncheck = {'rows': len(final), 'max_absolute_error': float(abs(final.y_pred - canonical.y_pred).max()), 'canonical_sha256': sha256(BASE / 'submission_session6.csv'), 'local_test_passed': True, 'base0328_rebuilt_from_raw_and_weights': True, 'new_model_training': False}\n(BASE / 'reports/submission_reproduction.json').write_text(json.dumps(check, ensure_ascii=False, indent=2))\nprint(json.dumps(check, ensure_ascii=False, indent=2))\ndisplay(final.head())", 'session6_verify', 'exec'), verification_scope)


In [ ]:
exact_local = verification_scope['BASE'] / 'validation_predictions.parquet'
exact_future = verification_scope['BASE'] / 'outputs/submission_session6_reproduced.csv'
print('Session6 проверен. Локальный test и конкурсный прогноз совпали с архивом.')


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display
HORIZONS = tuple(range(1, 9))
STEP = pd.Timedelta(minutes=30)
OOF_START = pd.Timestamp('2025-09-20')
LOCAL_TEST_START = pd.Timestamp('2025-10-18')
LOCAL_TEST_END = pd.Timestamp('2025-11-01')
OOF_WINDOWS = list(zip(pd.date_range(OOF_START, periods=4, freq='7D'), pd.date_range(OOF_START + pd.Timedelta(days=7), periods=4, freq='7D')))
KEYS = ['cutoff', 'horizon', 'route_id']
GROUP_KEYS = KEYS[:2]
BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
SEED = 42
THREADS = 4
SESSION_FILES = ('comparison_meta.json', 'train_solo_track.parquet', 'test_solo_track.parquet', 'validation_predictions.parquet')
XGB_STAGES = ('oof_week_0', 'oof_week_1', 'oof_week_2', 'calibration_select', 'train_val')
XGB_FILES = ('manifest.json', 'protocol.json', 'evaluation_complete.json', *(f'{stage}_h{h}_xgb_k8.parquet' for stage in XGB_STAGES for h in HORIZONS))

def find_dataset(label, path, marker, required):
    if path is None:
        root = Path('/kaggle/input')
        candidates = sorted({file.parent for file in root.rglob(marker)})
        complete = [folder for folder in candidates if all(((folder / name).is_file() for name in required))]
        if len(complete) == 1:
            return complete[0]
        if not complete and len(candidates) == 1:
            path = candidates[0]
        else:
            raise FileNotFoundError(f'{label}: нужен один полный набор. Укажите путь к папке в ячейке настроек. Найденные полные наборы: {complete}; папки с {marker}: {candidates}')
    path = Path(path)
    missing = [name for name in required if not (path / name).is_file()]
    if missing:
        raise FileNotFoundError(f'{label}: в {path} отсутствуют файлы {missing}')
    return path

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 ** 2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, data):
    temp = path.with_suffix('.partial')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)

def save_parquet(path, data):
    temp = path.with_suffix('.partial')
    data.to_parquet(temp, index=False)
    temp.replace(path)

def initialize(session_path, xgb_path):
    global ARTIFACTS, TIMES, ROUTES, VALUES, SESSION_META, SESSION_DATA, XGB_DATA
    SESSION_DATA = find_dataset('Session6', session_path, 'comparison_meta.json', SESSION_FILES)
    XGB_DATA = find_dataset('XGBoost rich k8', xgb_path, 'oof_week_0_h1_xgb_k8.parquet', XGB_FILES)
    train_path = SESSION_DATA / 'train_solo_track.parquet'
    SESSION_META = json.loads((SESSION_DATA / 'comparison_meta.json').read_text())
    xgb_manifest = json.loads((XGB_DATA / 'manifest.json').read_text())
    XGB_PROTOCOL = json.loads((XGB_DATA / 'protocol.json').read_text())
    xgb_evaluation = json.loads((XGB_DATA / 'evaluation_complete.json').read_text())
    contract = dict(code=NOTEBOOK_HASH, seed=SEED, session_sha256={name: digest_file(SESSION_DATA / name) for name in SESSION_FILES}, xgb_sha256={name: digest_file(XGB_DATA / name) for name in XGB_FILES})
    if xgb_manifest['train_hash'] != contract['session_sha256']['train_solo_track.parquet'] or xgb_manifest['test_hash'] != contract['session_sha256']['test_solo_track.parquet']:
        raise ValueError('Датасеты XGBoost rich и Session6 используют разные входные файлы')
    if pd.Timestamp(XGB_PROTOCOL['val_start']) != OOF_START or pd.Timestamp(XGB_PROTOCOL['calibration_split']) != OOF_WINDOWS[-1][0] or pd.Timestamp(XGB_PROTOCOL['test_start']) != LOCAL_TEST_START or (pd.Timestamp(xgb_evaluation['train_val_model_target_end_exclusive']) != LOCAL_TEST_START) or (XGB_PROTOCOL['model_clusters'] != [8]) or XGB_PROTOCOL['test_used_for_selection'] or (not xgb_evaluation['test_evaluated_before_full_refit']):
        raise ValueError('Границы OOF или протокол архива XGB rich отличаются')
    run_id = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:12]
    ARTIFACTS = Path('/kaggle/working') / f'best_blend_{run_id}'
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACTS / 'inputs.json', dict(contract, session_path=str(SESSION_DATA), xgb_path=str(XGB_DATA)))
    print('Session6:', SESSION_DATA)
    print('XGBoost rich k8:', XGB_DATA)
    history = pd.read_parquet(train_path, columns=['timestamp', 'route_id', TARGET, *STATUSES])
    history = history.sort_values(['timestamp', 'route_id'])
    TIMES = pd.DatetimeIndex(history.timestamp.unique())
    ROUTES = np.sort(history.route_id.unique())
    if len(history) != len(TIMES) * len(ROUTES) or history.isna().any().any() or history.duplicated(['timestamp', 'route_id']).any() or (not TIMES.equals(pd.date_range(TIMES[0], TIMES[-1], freq=STEP))):
        raise ValueError('Требуется полная получасовая панель')
    VALUES = {name: history[name].to_numpy(dtype=np.float32).reshape(len(TIMES), len(ROUTES)) for name in (TARGET, *STATUSES)}
    if not all((np.isfinite(array).all() for array in VALUES.values())) or np.any(VALUES[TARGET] < 0):
        raise ValueError('Некорректные наблюдения исходной панели')
    display(pd.DataFrame([dict(split=f'oof_{i}', start=start, end_exclusive=stop) for i, (start, stop) in enumerate(OOF_WINDOWS, 1)] + [dict(split='test', start=LOCAL_TEST_START, end_exclusive=LOCAL_TEST_END)]))

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or (not np.isfinite(predicted).all()):
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_report(model, boundary, horizon, group, split, train_y, train_pred, query_y, query_pred):
    report = pd.DataFrame([dict(model=model, fit_before=boundary, horizon=horizon, group=group, period=period, **metric(y, pred)) for period, y, pred in [('train', train_y, train_pred), (split, query_y, query_pred)]])
    display(report)
    path = ARTIFACTS / f'fits_{model}_{boundary:%Y%m%d}_h{horizon}_{group}.csv'
    report.to_csv(path, index=False)

def read_xgb(stage):
    if stage not in XGB_STAGES:
        raise ValueError(f'Неизвестный этап XGBoost rich: {stage}')
    frames = [pd.read_parquet(XGB_DATA / f'{stage}_h{h}_xgb_k8.parquet', columns=['cutoff', 'timestamp', 'route_id', 'h', 'y', 'y_pred']) for h in HORIZONS]
    result = pd.concat(frames, ignore_index=True).rename(columns={'h': 'horizon', 'y_pred': 'xgb_rich'})
    if not result.timestamp.eq(result.cutoff + result.horizon * STEP).all():
        raise ValueError('Неверный timestamp в XGB rich')
    return result

def read_session6_test():
    frame = pd.read_parquet(SESSION_DATA / 'validation_predictions.parquet', columns=KEYS + ['y', 'session_prediction'])
    return frame.rename(columns={'session_prediction': 'session6'})

def context(start, stop):
    origins = np.flatnonzero((TIMES >= start) & (TIMES < stop) & (TIMES.hour == 10) & (TIMES.minute == 30))
    expected_days = (stop - start).days
    if len(origins) != expected_days or origins[-1] + max(HORIZONS) >= len(TIMES):
        raise ValueError('Не хватает дней или целевых горизонтов')
    y = VALUES[TARGET]
    windows = np.arange(-47, 1)
    level24 = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64) for pos in origins])
    seasonal_origins = origins[:, None] - 48 * np.arange(1, 29)
    if seasonal_origins.min() + windows.min() < 0:
        raise ValueError('Не хватает наблюдений для profile28')
    seasonal_level = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64) for pos in seasonal_origins.ravel()]).reshape(*seasonal_origins.shape, len(ROUTES))
    ratio = level24[:, None, :] / np.maximum(seasonal_level, 1)
    route_count = len(ROUTES)
    common = dict(cutoff=TIMES[origins].repeat(route_count), route_id=np.tile(ROUTES, len(origins)), level24=level24.ravel(), current=y[origins].ravel(), delta1h=(y[origins] - y[origins - 2]).ravel(), delta2h=(y[origins] - y[origins - 4]).ravel(), network_current=np.repeat(y[origins].sum(axis=1, dtype=np.float64), route_count))
    return pd.concat([pd.DataFrame(dict(common, horizon=h, y=y[origins + h].ravel(), profile28=(y[seasonal_origins + h] * ratio).mean(axis=1).ravel())) for h in HORIZONS], ignore_index=True).sort_values(KEYS).reset_index(drop=True)

def join_base(frame, incoming, column):
    relevant = incoming[KEYS + ['y', column]]
    merged = frame.merge(relevant, on=KEYS, how='left', validate='one_to_one', suffixes=('', '_incoming'))
    if len(relevant) != len(frame) or not np.array_equal(merged.y, merged.y_incoming):
        raise ValueError(f'Не совпали ключи или факты для {column}')
    if not np.isfinite(merged[column]).all() or merged[column].lt(0).any():
        raise ValueError(f'Некорректные прогнозы {column}')
    return merged.drop(columns='y_incoming')

def assemble(start, stop, session, rf, cat, xgb, xgb_clean):
    frame = context(start, stop)
    for source, name in [(session, 'session6'), (rf, 'rf'), (cat, 'cat_rich'), (xgb, 'xgb_rich'), (xgb_clean, 'xgb_clean')]:
        part = source.loc[source.cutoff.ge(start) & source.cutoff.lt(stop)]
        frame = join_base(frame, part, name)
    return frame

def archive_report():
    path = ARTIFACTS.with_suffix('.zip')
    temp = path.with_suffix('.partial')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for file in sorted(ARTIFACTS.rglob('*')):
            if file.is_file() and (not file.name.endswith('.partial')):
                archive.write(file, file.relative_to(ARTIFACTS.parent))
    temp.replace(path)
    print('Архив результатов', path)


In [ ]:
def locate_evaluation(path=None):
    required = ['inputs.json', 'base_oof.parquet', 'test_predictions.parquet', 'train_test_metrics.csv', 'frozen_selection.json']
    if path is None:
        candidates = set()
        for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
            candidates.update((p.parent for p in root.rglob('frozen_selection.json') if all(((p.parent / name).is_file() for name in required))))
        if len(candidates) != 1:
            raise ValueError(f'Укажите EVALUATION_DIR: найдено {sorted(map(str, candidates))}')
        path = candidates.pop()
    path = Path(path)
    if not all(((path / name).is_file() for name in required)):
        raise FileNotFoundError(f'Неполный output блендинга: {path}')
    return path

def initialize_submission(evaluation_path=None, session_path=None, xgb_path=None):
    global EVALUATION, ARTIFACTS, FROZEN, HISTORICAL_METRICS, FULL_BOUNDARY, REQUESTS, META_HISTORY
    global FUTURE_ORIGIN, FUTURE_CUTOFF, SUBMISSION_ROOT, ALL_META_HISTORY
    EVALUATION = locate_evaluation(evaluation_path)
    contract = json.loads((EVALUATION / 'inputs.json').read_text())
    globals()['NOTEBOOK_HASH'] = contract['code']
    initialize(session_path, xgb_path)
    for key in ['session_sha256', 'xgb_sha256']:
        actual = json.loads((ARTIFACTS / 'inputs.json').read_text())[key]
        if actual != contract[key]:
            raise ValueError(f'Входы отличаются от завершённого запуска: {key}')
    FROZEN = json.loads((EVALUATION / 'frozen_selection.json').read_text())
    if FROZEN['variant'] != 'cat_d4__ridge' or FROZEN['test_used_for_selection']:
        raise ValueError('Ожидался зафиксированный до test cat_d4__ridge')
    digest = hashlib.sha256((SUBMIT_CODE_HASH + json.dumps(contract, sort_keys=True) + digest_file(EVALUATION / 'base_oof.parquet') + digest_file(EVALUATION / 'test_predictions.parquet')).encode()).hexdigest()[:12]
    ARTIFACTS = Path('/kaggle/working') / f'blend_submit_{digest}'
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACTS / 'frozen_selection.json', FROZEN)
    SUBMISSION_ROOT = ARTIFACTS
    FULL_BOUNDARY = TIMES[-1] + STEP
    FUTURE_ORIGIN = np.array([len(TIMES) - 1])
    FUTURE_CUTOFF = TIMES[-1]
    REQUESTS = pd.read_parquet(SESSION_DATA / 'test_solo_track.parquet', columns=['id', 'route_id', 'timestamp'])
    REQUESTS['timestamp'] = pd.to_datetime(REQUESTS.timestamp)
    expected = pd.MultiIndex.from_product([ROUTES, [FUTURE_CUTOFF + h * STEP for h in HORIZONS]])
    actual = pd.MultiIndex.from_frame(REQUESTS[['route_id', 'timestamp']])
    if not REQUESTS.id.is_unique or actual.has_duplicates or len(actual) != len(expected) or len(expected.difference(actual)):
        raise ValueError('Конкурсный test не соответствует восьми будущим горизонтам')
    columns = KEYS + ['y', *BASES, 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
    META_HISTORY = pd.concat([pd.read_parquet(EVALUATION / name)[columns] for name in ['base_oof.parquet', 'test_predictions.parquet']], ignore_index=True)
    origins = TIMES.get_indexer(META_HISTORY.cutoff)
    routes = pd.Index(ROUTES).get_indexer(META_HISTORY.route_id)
    targets = origins + META_HISTORY.horizon.to_numpy()
    if META_HISTORY.duplicated(KEYS).any() or (origins < 0).any() or (routes < 0).any() or (targets >= len(TIMES)).any() or (not np.array_equal(META_HISTORY.y, VALUES[TARGET][targets, routes])):
        raise ValueError('Некорректные исторические прогнозы для финальной метамодели')
    if not np.isfinite(META_HISTORY[columns[3:]].to_numpy(dtype=float)).all():
        raise ValueError('Пропуски в исторических прогнозах')
    ALL_META_HISTORY = META_HISTORY.copy()
    HISTORICAL_METRICS = pd.read_csv(EVALUATION / 'train_test_metrics.csv')
    HISTORICAL_METRICS.to_csv(ARTIFACTS / 'metrics_before_full_refit.csv', index=False)
    display(HISTORICAL_METRICS.loc[HISTORICAL_METRICS.variant.eq(FROZEN['variant'])])
    print('Это оценка до полного переобучения. После включения local test в fit новой независимой оценки нет.')

def enter_submission_scope(scope):
    global ARTIFACTS, FULL_BOUNDARY, META_HISTORY, SUBMISSION_SCOPE, QUERY_ORIGINS
    if scope not in ('train_val', 'full_history'):
        raise ValueError(f'Неизвестный режим: {scope}')
    if scope == 'full_history':
        marker = SUBMISSION_ROOT / 'train_val' / 'evaluation_complete.json'
        if not marker.is_file() or not (SUBMISSION_ROOT / 'train_val' / 'submission.csv').is_file():
            raise RuntimeError('Сначала завершите новую оценку train_val на test и первый сабмит')
        audit = json.loads(marker.read_text())
        if audit['code'] != SUBMIT_CODE_HASH or audit['boundary'] != str(LOCAL_TEST_START):
            raise ValueError('Оценка test относится к другому коду или границе')
    SUBMISSION_SCOPE = scope
    FULL_BOUNDARY = LOCAL_TEST_START if scope == 'train_val' else TIMES[-1] + STEP
    test_origins = np.flatnonzero((TIMES >= LOCAL_TEST_START) & (TIMES < LOCAL_TEST_END) & (TIMES.hour == 10) & (TIMES.minute == 30))
    QUERY_ORIGINS = np.concatenate([test_origins, FUTURE_ORIGIN]) if scope == 'train_val' else FUTURE_ORIGIN.copy()
    ARTIFACTS = SUBMISSION_ROOT / scope
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    source_history = ALL_META_HISTORY
    if scope == 'full_history':
        recent = pd.read_parquet(SUBMISSION_ROOT / 'train_val' / 'test_predictions.parquet')
        early = ALL_META_HISTORY.loc[(ALL_META_HISTORY.cutoff + ALL_META_HISTORY.horizon * STEP).lt(LOCAL_TEST_START)]
        source_history = pd.concat([early, recent[early.columns]], ignore_index=True)
    target_times = source_history.cutoff + source_history.horizon * STEP
    META_HISTORY = source_history.loc[target_times.lt(FULL_BOUNDARY)].reset_index(drop=True)
    if META_HISTORY.empty or not (META_HISTORY.cutoff + META_HISTORY.horizon * STEP).lt(FULL_BOUNDARY).all():
        raise ValueError('Некорректная граница обучения метамодели')
    write_json(ARTIFACTS / 'frozen_selection.json', FROZEN)
    HISTORICAL_METRICS.to_csv(ARTIFACTS / 'metrics_before_full_refit.csv', index=False)
    write_json(ARTIFACTS / 'submission_protocol.json', dict(code=SUBMIT_CODE_HASH, evaluation=str(EVALUATION), frozen=FROZEN, scope=scope, training_target_end_exclusive=str(FULL_BOUNDARY), feature_cutoff=str(FUTURE_CUTOFF), local_test_used_for_fit=scope == 'full_history', meta_rows=len(META_HISTORY), meta_last_target=str((META_HISTORY.cutoff + META_HISTORY.horizon * STEP).max()), historical_observations_used_for_future_features=True, test_evaluated_before_full_refit=True, new_independent_test_score=scope == 'train_val', session6_refit='bases and residuals restricted to training boundary; EWMA uses only pre-boundary residuals', session6_cat_training_starts='eligible[len(eligible) // 4:] as in original recipe', train_val_authorization='explicit user request for comparison submission'))
    print('Режим:', scope, 'обучение на целях строго до', FULL_BOUNDARY, 'признаки на момент', FUTURE_CUTOFF, 'OOF-строк', len(META_HISTORY), flush=True)

def refit_report(model, boundary, horizon, group, split, train_y, train_pred, query_y, query_pred):
    row = dict(model=model, horizon=horizon, group=group, period=f'{SUBMISSION_SCOPE}_train', **metric(train_y, train_pred))
    rows = [row]
    actual = np.asarray(query_y).ravel()
    predicted = np.asarray(query_pred).ravel()
    known = np.isfinite(actual)
    if SUBMISSION_SCOPE == 'train_val' and known.any():
        rows.append(dict(model=model, horizon=horizon, group=group, period='test', **metric(actual[known], predicted[known])))
    table = pd.DataFrame(rows)
    display(table)
    table.to_csv(ARTIFACTS / f'refit_{model}_h{horizon}_{group}.csv', index=False)

def future_context():
    y = VALUES[TARGET]
    pos = len(TIMES) - 1
    windows = np.arange(-47, 1)
    level = y[pos + windows].mean(axis=0, dtype=np.float64)
    seasonal = pos - 48 * np.arange(1, 29)
    historical_level = np.stack([y[p + windows].mean(axis=0, dtype=np.float64) for p in seasonal])
    ratio = level / np.maximum(historical_level, 1)
    return pd.concat([pd.DataFrame(dict(cutoff=FUTURE_CUTOFF, horizon=h, route_id=ROUTES, level24=level, current=y[pos], delta1h=y[pos] - y[pos - 2], delta2h=y[pos] - y[pos - 4], network_current=y[pos].sum(dtype=np.float64), profile28=(y[seasonal + h] * ratio).mean(axis=0))) for h in HORIZONS], ignore_index=True).sort_values(KEYS).reset_index(drop=True)

def observed_query(h):
    result = np.full((len(QUERY_ORIGINS), len(ROUTES)), np.nan)
    known = QUERY_ORIGINS + h < len(TIMES)
    result[known] = VALUES[TARGET][QUERY_ORIGINS[known] + h]
    return result

def join_predictions(frame, parts):
    for name, incoming in parts.items():
        selected = incoming.loc[incoming.cutoff.isin(frame.cutoff.unique()), KEYS + [name]]
        if selected.duplicated(KEYS).any() or len(selected) != len(frame):
            raise ValueError(f'Неверные ключи: {name}')
        frame = frame.merge(selected, on=KEYS, how='left', validate='one_to_one')
    if not np.isfinite(frame[BASES]).all().all() or frame[BASES].lt(0).any().any():
        raise ValueError('Некорректные базовые прогнозы')
    return frame

def finish_submission(parts):
    matrix = shape_matrix(META_HISTORY)
    joint = fit_weights(matrix, META_HISTORY.y)
    shape = fit_shape(FROZEN['shape'], META_HISTORY, matrix, joint)
    volumes = fit_volumes(META_HISTORY)

    def predict(frame):
        values = predict_shape(FROZEN['shape'], shape, frame, shape_matrix(frame), joint)
        return values * volume_multipliers(frame, volumes)[FROZEN['volume']]
    fit_prediction = predict(META_HISTORY)
    if SUBMISSION_SCOPE == 'train_val':
        test = join_predictions(context(LOCAL_TEST_START, LOCAL_TEST_END), parts)
        test_prediction = predict(test)
        refit_report(FROZEN['variant'], FULL_BOUNDARY, 'all', 'all', 'test', META_HISTORY.y, fit_prediction, test.y, test_prediction)
        rows = [dict(model=FROZEN['variant'], period=period, **metric(y, pred)) for period, y, pred in [('train_oof', META_HISTORY.y, fit_prediction), ('test', test.y, test_prediction)]]
        rows.extend((dict(model=name, period='test', **metric(test.y, test[name])) for name in BASES))
        table = pd.DataFrame(rows)
        table.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
        display(table)
        test['blend_prediction'] = test_prediction
        save_parquet(ARTIFACTS / 'test_predictions.parquet', test)
        daily = [dict(day=str(day), **metric(frame.y, frame.blend_prediction)) for day, frame in test.groupby(test.cutoff.dt.normalize())]
        pd.DataFrame(daily).to_csv(ARTIFACTS / 'test_daily_metrics.csv', index=False)
        write_json(ARTIFACTS / 'evaluation_complete.json', dict(code=SUBMIT_CODE_HASH, boundary=str(FULL_BOUNDARY), test_evaluated_before_full_refit=True, same_models_for_test_and_submission=True, test_used_for_selection=False, **metric(test.y, test_prediction)))
        print('Новая оценка test завершена до полного переобучения. Эти же модели делают первый сабмит.')
    else:
        refit_report(FROZEN['variant'], FULL_BOUNDARY, 'all', 'all', 'future', META_HISTORY.y, fit_prediction, [], [])
    joblib.dump(dict(frozen=FROZEN, bases=BASES, joint=joint, shape=shape, volumes=volumes), ARTIFACTS / 'submission_meta.joblib')
    frame = join_predictions(future_context(), parts)
    frame['timestamp'] = frame.cutoff + frame.horizon * STEP
    frame['y_pred'] = predict(frame)
    result = REQUESTS.merge(frame[['route_id', 'timestamp', 'y_pred']], on=['route_id', 'timestamp'], how='left', sort=False, validate='one_to_one')
    if not result.id.equals(REQUESTS.id) or not np.isfinite(result.y_pred).all() or result.y_pred.lt(0).any():
        raise ValueError('Неверный submission')
    result[['id', 'y_pred']].to_csv(ARTIFACTS / 'submission.csv', index=False)
    destination = SUBMISSION_ROOT.parent / f'submission_{SUBMISSION_SCOPE}.csv'
    result[['id', 'y_pred']].to_csv(destination, index=False)
    save_parquet(ARTIFACTS / 'future_predictions.parquet', frame)
    pd.DataFrame([dict(model=base, weight=float(w)) for base, w in zip(BASES, joint)]).to_csv(ARTIFACTS / 'full_meta_weights.csv', index=False)
    archive_report()
    print('Готово:', destination, result.shape)
    print('Независимая от полного переобучения оценка test получена до переобучения; период test ранее изучался.')

def session_full():
    global start_tune, start_test, eval_origins, out
    out = ARTIFACTS / 'session6_full'
    out.mkdir(exist_ok=True)
    cached = out / 'future.parquet'
    if cached.is_file():
        return pd.read_parquet(cached)
    start_tune = OOF_START - pd.Timedelta(days=7)
    prepare_history(values)
    origins = positions[(times >= start_tune) & (times.hour == 10) & (times.minute == 30) & (times + max(HORIZONS) * STEP < FULL_BOUNDARY)]
    historical = []
    for h in HORIZONS:
        path = EVALUATION / 'session6' / 'models' / 'early_anchor' / f'catboost_h{h}.cbm'
        if not path.is_file():
            raise FileNotFoundError(f'Подключите полный output, включая {path.name}')
        model = CatBoostRegressor()
        model.load_model(str(path))
        f = make_cat_frame(origins, h)
        columns = cat_columns + (SEASONAL_FEATURES if h in (5, 6, 8) else [])
        part = f[['cutoff', 'route_id', 'timestamp', 'profile28', *anchor_columns]].copy()
        part['horizon'] = h
        part['y'] = part['fit_y'] = target[origins + h].ravel()
        part['model_prediction'] = np.maximum(model.predict(f[columns]) * 100000, 0)
        part['raw_blend'] = part.model_prediction if h == 1 else 0.7 * part.model_prediction + 0.3 * part.profile28
        part['shape'] = 1.0
        historical.append(part)
        del model, f
    history = pd.concat(historical, ignore_index=True)
    save_parquet(out / 'residual_training.parquet', history)
    start_tune = FULL_BOUNDARY
    start_test = FULL_BOUNDARY if SUBMISSION_SCOPE == 'train_val' else FUTURE_CUTOFF
    eval_origins = QUERY_ORIGINS
    (out / 'models' / 'full').mkdir(parents=True, exist_ok=True)
    prepare_history(values)
    future = []
    for h in HORIZONS:
        path = out / f'base_h{h}.parquet'
        if not path.is_file():
            save_parquet(path, base_predictions('full', h))
        future.append(pd.read_parquet(path))
    combined = pd.concat([history, *future], ignore_index=True)
    result = finish_pipeline(combined, 'full', FULL_BOUNDARY)
    result = result.loc[result.cutoff.isin(TIMES[QUERY_ORIGINS]), keys + ['prediction']].rename(columns={'prediction': 'session6'})
    save_parquet(cached, result)
    return result

def session_refit_report(branch, stage, h, split, y, p):
    key = (stage, h)
    if split == 'train':
        SESSION_TRAIN_REPORTS[key] = (np.asarray(y), np.asarray(p))
    elif key in SESSION_TRAIN_REPORTS:
        train_y, train_pred = SESSION_TRAIN_REPORTS.pop(key)
        refit_report('session6', FULL_BOUNDARY, h, stage, split, train_y, train_pred, y, p)

def fit_rich_xgb_full():
    folder = ARTIFACTS / 'xgb_rich_full'
    folder.mkdir(exist_ok=True)
    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]
    if all((p.is_file() for p in outputs)):
        return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)
    with threadpool_limits(limits=THREADS):
        segmentation = segment_routes(FULL_BOUNDARY, 'full_xgb')
    manifest = json.loads((XGB_DATA / 'manifest.json').read_text())
    parameters = manifest['cat_configs'][0]['parameters']
    for h, path in zip(HORIZONS, outputs):
        if path.is_file():
            continue
        print('Full XGB rich', h, flush=True)
        origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < FULL_BOUNDARY)
        matrix, names = rich_matrix(h, origins, segmentation)
        query, query_names = rich_matrix(h, QUERY_ORIGINS, segmentation)
        if names != query_names:
            raise ValueError('Разные признаки XGB rich')
        train = dict(x=matrix.reshape(-1, len(names)), names=names, route_index=np.tile(np.arange(len(ROUTES)), len(origins)), y=VALUES[TARGET][origins + h].ravel())
        globals()['PARAMETERS'] = parameters
        globals()['ACTIVE'] = dict(parameter_set='selected_xgb', residual=False)
        globals()['ACTIVE_H'] = h
        bundle = fit_model(train, segmentation, 'xgb', 8, {})
        pred = predict_model(bundle, train, segmentation)
        future = predict_model(bundle, dict(x=query.reshape(-1, len(names)), route_index=np.tile(np.arange(len(ROUTES)), len(QUERY_ORIGINS))), segmentation)
        refit_report('xgb_rich', FULL_BOUNDARY, h, 'all', 'future', train['y'], pred, observed_query(h).ravel(), future)
        for group, item in bundle['models'].items():
            item['model'].save_model(str(folder / f'h{h}_group{group}.ubj'))
        joblib.dump(segmentation, folder / 'segmentation.joblib')
        save_parquet(path, pd.DataFrame(dict(cutoff=TIMES[QUERY_ORIGINS].repeat(len(ROUTES)), horizon=h, route_id=np.tile(ROUTES, len(QUERY_ORIGINS)), xgb_rich=future)))
        del matrix, query, train, pred, bundle
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
    return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)


In [ ]:
SUBMIT_CODE_HASH = 'b57ea6018119651ec410f30f48ccb16b6d91fb4361196d220b9e421db08283e2'
import cupy as cp
import lightgbm as lgb
import torch
import joblib
import xgboost as xgb
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
from threadpoolctl import threadpool_limits
EVALUATION_DIR = None
SESSION_DATA_DIR = None
XGB_DATA_DIR = None
if cp.cuda.runtime.getDeviceCount() < 1 or get_gpu_device_count() < 1:
    raise RuntimeError('Нужен GPU Kaggle с CuPy, cuML, CatBoost, XGBoost и LightGBM')
initialize_submission(EVALUATION_DIR, SESSION_DATA_DIR, XGB_DATA_DIR)
fit_report = refit_report


## Обучение на train + val

Отдельный комплект моделей и файл `submission_train_val.csv`. В train_val после обучения выполняется новая оценка test, затем сохраняется сабмит теми же моделями. Full_history запускается только после наличия оценки и первого CSV. Справочные метрики прошлого запуска сохранены отдельно.


In [ ]:
enter_submission_scope('train_val')


In [ ]:
local = pd.read_parquet(exact_local).rename(columns={'session_prediction': 'session6'})
future = pd.read_csv(exact_future)
future = REQUESTS.merge(future, on='id', validate='one_to_one')
future['cutoff'] = FUTURE_CUTOFF
future['horizon'] = ((future.timestamp - FUTURE_CUTOFF) / STEP).astype(int)
future = future.rename(columns={'y_pred': 'session6'})
if SUBMISSION_SCOPE == 'train_val':
    original = read_session6_test()
    comparison = original.merge(local[KEYS + ['session6']], on=KEYS, validate='one_to_one', suffixes=('_old', '_exact'))
    if len(comparison) != len(original) or not np.allclose(comparison.session6_old, comparison.session6_exact, rtol=1e-12, atol=1e-06):
        raise ValueError('Архив Session6 отличается от входа исходного бленда')
    session_future = pd.concat([local[KEYS + ['session6']], future[KEYS + ['session6']]], ignore_index=True)
else:
    session_future = future[KEYS + ['session6']]
write_json(ARTIFACTS / 'session6_exact_protocol.json', dict(local_source=str(exact_local), future_source=str(exact_future), contest_uses_canonical_final_session6=True, full_retraining_from_scratch=False))


In [ ]:
cat_runtime = dict(globals())
exec(compile('\n\nimport joblib\nfrom catboost import CatBoostRegressor\nfrom sklearn.cluster import KMeans\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nCLUSTERS = (1, 8)\nMIN_CLUSTER_ROUTES = 10\nRUN = ARTIFACTS / \'cat_rich\'\nRUN.mkdir(exist_ok=True)\nROLL = {w: pd.DataFrame(VALUES[TARGET]).rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32)\n        for w in (4, 12, 48, 336)}\n\n\nPARAMETERS = {\'selected_cat\': {\'1\': {\'params\': {\'depth\': 6, \'learning_rate\': 0.03694755509074355, \'l2_leaf_reg\': 1.5679933916723008, \'random_strength\': 2.052699079536471, \'bagging_temperature\': 0.8803049874792026}, \'trees\': 900}, \'2\': {\'params\': {\'depth\': 5, \'learning_rate\': 0.1358198535217987, \'l2_leaf_reg\': 29.10635913133069, \'random_strength\': 1.7959754525911098, \'bagging_temperature\': 0.31203728088487304}, \'trees\': 387}, \'3\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.0224831270535195, \'l2_leaf_reg\': 53.99484409787433, \'random_strength\': 1.8033450352296265, \'bagging_temperature\': 1.416145155592091}, \'trees\': 900}, \'4\': {\'params\': {\'depth\': 6, \'learning_rate\': 0.0299058938693663, \'l2_leaf_reg\': 10.677482709481351, \'random_strength\': 1.7772437065861273, \'bagging_temperature\': 0.09290082543999545}, \'trees\': 364}, \'5\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.0224831270535195, \'l2_leaf_reg\': 53.99484409787433, \'random_strength\': 1.8033450352296265, \'bagging_temperature\': 1.416145155592091}, \'trees\': 504}, \'6\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.1411758906434134, \'l2_leaf_reg\': 46.22589001020831, \'random_strength\': 0.6370173320348285, \'bagging_temperature\': 0.36364993441420124}, \'trees\': 125}, \'7\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.1411758906434134, \'l2_leaf_reg\': 46.22589001020831, \'random_strength\': 0.6370173320348285, \'bagging_temperature\': 0.36364993441420124}, \'trees\': 167}, \'8\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.03691990689655602, \'l2_leaf_reg\': 11.207606211860567, \'random_strength\': 1.2958350559263474, \'bagging_temperature\': 0.5824582803960838}, \'trees\': 900}}}\n\nRICH_SOURCE = "import pyarrow as pa\\nimport pyarrow.parquet as pq\\ntarget = values[TARGET]\\n\\ndef past(array, positions):\\n    positions = np.asarray(positions, dtype=np.int64)\\n    valid = (positions >= 0) & (positions < len(array))\\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\\n    result[~valid] = np.nan\\n    return result\\n\\ndef average_available(matrices):\\n    total = np.zeros_like(matrices[0], dtype=np.float32)\\n    count = np.zeros_like(total)\\n    for matrix in matrices:\\n        valid = np.isfinite(matrix)\\n        total += np.where(valid, matrix, 0)\\n        count += valid\\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\\n\\ndef rolling(array, steps):\\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\\nlevel24 = rolling(target, 48)\\npositions = np.arange(len(times))\\nextra_signals = {}\\nfor name in STATUSES[:3]:\\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\\nfor name in STATUSES[3:]:\\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\\nhub_signal = extra_signals[\'status_6_diff_hourly\']\\n\\ndef common_features(origins):\\n    common = {}\\n    for k in (0, 1, 2, 4):\\n        common[f\'target_back_{k}\'] = past(target, origins - k)\\n    for name, array in values.items():\\n        for steps in (4, 12):\\n            common[f\'{name}_mean_{steps}\'] = means[name, steps][origins]\\n        if name in STATUSES:\\n            common[f\'{name}_latest\'] = array[origins]\\n            if name in STATUSES[:3]:\\n                common[f\'{name}_change_1h\'] = array[origins] - past(array, origins - 2)\\n    for name, array in extra_signals.items():\\n        common[name] = array[origins]\\n    for lag in (48, 96):\\n        common[f\'target_1h_lag_{lag}\'] = past(target, origins - lag)\\n    common[\'warehouse_total_load_lag_48\'] = past(extra_signals[\'warehouse_total_load\'], origins - 48)\\n    common[\'status_6_diff_hourly_lag_48\'] = past(hub_signal, origins - 48)\\n    common[\'target_vs_mean_6h\'] = target[origins] - means[TARGET, 12][origins]\\n    recent = means[TARGET, 4][origins]\\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\\n    common[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\\n    return common\\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\\nFEATURE_GROUPS = {\'calendar\': [\'hour\', \'minute\', \'hour_float\', \'hour_sin\', \'hour_cos\', \'dayofweek\', \'is_weekend\', \'weekday_sin\', \'weekday_cos\'], \'local_statuses\': [\'status_1_hourly\', \'status_2_hourly\', \'status_3_hourly\', \'warehouse_total_load\', \'warehouse_processing_rate\'], \'hub_combinations\': [\'status_4_diff_hourly\', \'status_5_diff_hourly\', \'status_6_diff_hourly\', \'hub_buffer\', \'hub_gate_bottleneck\'], \'historical_lags\': [\'target_1h_lag_48\', \'target_1h_lag_96\', \'warehouse_total_load_lag_48\', \'status_6_diff_hourly_lag_48\'], \'target_change\': [\'target_acceleration\'], \'correlation_lag\': CORRELATION_COLUMNS, \'pilot_start_time\': [\'start_sin\', \'start_cos\'], \'pilot_peaks\': [\'peak7_mean_max\', \'peak7_mean_level\', \'peak7_share_-1\', \'peak7_share_0\', \'peak7_share_1\']}\\n\\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\\n    grid = pd.DatetimeIndex(timestamps)\\n    route_ids = np.asarray(route_ids)\\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\\n    if len(np.unique(route_ids)) != len(route_ids):\\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\\n    steps = float(max_lookback_hours) * 2\\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\\n    max_lag = int(steps)\\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\\n    anchors = np.arange(max_lag + 2, stop)\\n    best_corr = np.full(len(route_ids), np.nan)\\n    best_lag = np.full(len(route_ids), np.nan)\\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\\n    for start in range(0, len(route_ids), route_block_size):\\n        finish = min(start + route_block_size, len(route_ids))\\n        x = target_values[anchors, start:finish].astype(np.float64)\\n        for lag in range(2, max_lag + 1):\\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\\n            valid = np.isfinite(x) & np.isfinite(z)\\n            count = valid.sum(axis=0)\\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\\n            divisor = np.maximum(count, 1)\\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\\n            denominator = np.sqrt(vx * vz)\\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\\n            corr = np.clip(corr, -1, 1)\\n            previous = best_corr[start:finish]\\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\\n            best_corr[start:finish][better] = corr[better]\\n            best_lag[start:finish][better] = lag\\n            best_pairs[start:finish][better] = count[better]\\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\\n\\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\\n    if h not in range(1, 9) or chunk_rows < 1:\\n        raise ValueError(\'Некорректный горизонт или размер блока\')\\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\\n    route_index = pd.Index(routes)\\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\\n    for name in CORRELATION_COLUMNS:\\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\\n    for start in range(0, len(frame), chunk_rows):\\n        finish = min(start + chunk_rows, len(frame))\\n        block = frame.iloc[start:finish]\\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\\n        route = route_index.get_indexer(block.route_id)\\n        if (origin < 0).any() or (route < 0).any():\\n            raise ValueError(\'Строка признаков не сопоставлена с историей\')\\n        selected = lag[route]\\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\\n        available &= np.isfinite(signal)\\n        for name, array in quality.items():\\n            frame.iloc[start:finish, column_positions[name]] = array[route]\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal\']] = np.where(available, signal, np.nan)\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal_available\']] = available.astype(np.float32)\\n    return frame\\n\\ndef horizon_dataset(h, origins):\\n    slot = origins + h\\n    dates = times[origins] + h * STEP\\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins))})\\n    for name, matrix in common_features(origins).items():\\n        frame[name] = matrix.ravel()\\n    calendar = {\'hour\': dates.hour, \'minute\': dates.minute, \'hour_float\': dates.hour + dates.minute / 60.0, \'dayofweek\': dates.dayofweek, \'is_weekend\': (dates.dayofweek >= 5).astype(np.int8)}\\n    calendar[\'hour_sin\'] = np.sin(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'hour_cos\'] = np.cos(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\\n    for name, array in calendar.items():\\n        frame[name] = np.repeat(array, len(routes))\\n    frame = add_correlation_features(frame, h, validation_lag_table)\\n    for days in (1, 2, 7):\\n        frame[f\'slot_days_back_{days}\'] = past(target, slot - days * 48).ravel()\\n    for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\\n    profile = np.zeros((len(origins), len(routes)))\\n    counts = np.zeros_like(profile)\\n    for day in range(1, 29):\\n        level = past(level24, origins - day * 48)\\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\\n        profile += np.nan_to_num(adjusted)\\n        counts += np.isfinite(adjusted)\\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\\n    if h == 1:\\n        lower = np.zeros_like(target[origins])\\n        upper = past(target, origins - 2)\\n        for position in (origins - 1, origins):\\n            total = past(target, position)\\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\\n    else:\\n        frame[\'overlap\'] = np.nan\\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\\n    return frame.astype({name: \'float32\' for name in numeric})"\n\ndef segment_routes(boundary, stage):\n    mask = TIMES < boundary\n    y = VALUES[TARGET][mask].astype(np.float64)\n    scale = np.maximum(y.mean(axis=0), 1.0)\n    normalized = y / scale\n    grid = TIMES[mask]\n    descriptors = {}\n    for slot in range(48):\n        select = grid.hour * 2 + grid.minute // 30 == slot\n        descriptors[f\'slot_{slot:02}\'] = normalized[select].mean(axis=0)\n    for day in range(7):\n        descriptors[f\'weekday_{day}\'] = normalized[grid.dayofweek == day].mean(axis=0)\n    descriptors[\'log_cv\'] = np.log1p(normalized.std(axis=0))\n    descriptors[\'zero_share\'] = (y == 0).mean(axis=0)\n    descriptors[\'jump_share\'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)\n    descriptors[\'log_q95\'] = np.log1p(np.quantile(normalized, 0.95, axis=0))\n    frame = pd.DataFrame(descriptors, index=ROUTES)\n    if not np.isfinite(frame.to_numpy()).all():\n        raise ValueError(\'Недостаточно данных для профилей маршрутов\')\n    scaler = StandardScaler().fit(frame)\n    z = scaler.transform(frame)\n    labels, estimators = ({1: np.zeros(len(ROUTES), dtype=int)}, {})\n    for k in CLUSTERS:\n        if k == 1:\n            continue\n        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)\n        groups = model.labels_.copy()\n        counts = np.bincount(groups, minlength=k)\n        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1\n        labels[k], estimators[k] = (groups, model)\n    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels, descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))\n    joblib.dump(artifact, RUN / f\'segments_{stage}.joblib\')\n    frame.insert(0, \'route_id\', ROUTES)\n    frame[\'target_scale\'] = scale\n    for k, groups in labels.items():\n        frame[f\'cluster_{k}\'] = groups\n    frame.to_csv(RUN / f\'route_profiles_{stage}.csv\', index=False)\n    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum())) for k, groups in labels.items() for g in np.unique(groups)]))\n    return artifact\n\ndef features(h, origins, segmentation):\n    scale = segmentation[\'scale\'].astype(np.float32)\n    columns, names = ([], [])\n    shape = (len(origins), len(ROUTES))\n\n    def add(name, array):\n        names.append(name)\n        columns.append(np.broadcast_to(array, shape).reshape(-1))\n\n    def lag(name, array, indices):\n        valid = indices >= 0\n        data = array[np.maximum(indices, 0)] / scale\n        add(name, np.where(valid[:, None], data, 0))\n        add(name + \'_available\', valid[:, None])\n    for delay in (0, 1, 2, 4, 12):\n        lag(f\'target_back_{delay}\', VALUES[TARGET], origins - delay)\n    for status in STATUSES:\n        lag(status, VALUES[status], origins)\n        add(status + \'_delta_1h\', (VALUES[status][origins] - VALUES[status][np.maximum(origins - 2, 0)]) / scale)\n    for window, array in ROLL.items():\n        lag(f\'target_mean_{window}\', array, origins)\n    for days in (1, 7, 14, 28):\n        lag(f\'target_season_{days}\', VALUES[TARGET], origins + h - 48 * days)\n    dates = TIMES[origins] + h * STEP\n    for label, phase in [(\'weekday\', dates.dayofweek.to_numpy() / 7), (\'slot\', (dates.hour * 2 + dates.minute / 30).to_numpy() / 48)]:\n        add(label + \'_sin\', np.sin(2 * np.pi * phase)[:, None])\n        add(label + \'_cos\', np.cos(2 * np.pi * phase)[:, None])\n    add(\'log_route_scale\', np.log1p(scale)[None, :])\n    matrix = np.column_stack(columns).astype(np.float32)\n    if not np.isfinite(matrix).all():\n        raise ValueError(\'Неконечные признаки\')\n    return (matrix, names)\n\noriginal_features_rich = features\noriginal_segment_rich = segment_routes\n\ndef segment_routes(boundary, stage):\n    global FEATURE_NS\n    result = original_segment_rich(boundary, stage)\n    FEATURE_NS = dict(np=np, pd=pd, values=VALUES, TARGET=TARGET, STATUSES=STATUSES, times=TIMES, routes=ROUTES, STEP=STEP, WINDOW_OFFSETS=tuple(range(-4, 5)))\n    exec(RICH_SOURCE, FEATURE_NS)\n    lag_table = FEATURE_NS[\'calculate_dynamic_shoulder\'](VALUES[TARGET], FEATURE_NS[\'hub_signal\'], TIMES, ROUTES, fit_before=boundary)\n    FEATURE_NS[\'validation_lag_table\'] = lag_table\n    FEATURE_NS[\'network_values\'] = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32) for name, array in VALUES.items()}\n    lag_table.to_csv(RUN / f\'rich_status6_lags_{stage}.csv\', index=False)\n    return result\n\ndef features(h, origins, segmentation):\n    base, names = original_features_rich(h, origins, segmentation)\n    frame = FEATURE_NS[\'horizon_dataset\'](h, origins)\n    cols = [c for c in frame if c not in (\'route_id\', \'cutoff\', \'timestamp\', \'y\', \'id\', \'overlap\')]\n    values = frame[cols].to_numpy(dtype=np.float32)\n    scale = np.tile(segmentation[\'scale\'], len(origins)).astype(np.float32)\n    dimensional = [i for i, c in enumerate(cols) if c not in (\'hour\', \'minute\', \'hour_float\', \'dayofweek\', \'is_weekend\') and (not any((v in c for v in (\'ratio\', \'rate\', \'correlation\', \'pair_count\', \'lag_steps\', \'valid\', \'available\', \'_sin\', \'_cos\'))))]\n    for i in dimensional:\n        values[:, i] /= scale\n    available = np.isfinite(values).astype(np.float32)\n    values = np.where(np.isfinite(values), values, 0)\n    shoulder_columns, shoulder_names = ([], [])\n    for label, days in [(\'daily7\', range(1, 8)), (\'weekly4\', (7, 14, 21, 28))]:\n        profiles = []\n        for offset in (-4, -2, 0, 2, 4):\n            total = np.zeros((len(origins), len(ROUTES)), dtype=np.float32)\n            count = np.zeros((len(origins), 1), dtype=np.float32)\n            for day in days:\n                positions = origins + h - day * 48 + offset\n                valid = positions >= 0\n                total += np.where(valid[:, None], VALUES[TARGET][np.maximum(positions, 0)], 0)\n                count += valid[:, None]\n            profile = np.divide(total, count, out=np.zeros_like(total), where=count > 0) / segmentation[\'scale\'][None, :]\n            profiles.append(profile.ravel())\n            shoulder_columns.append(profile.ravel())\n            shoulder_names.append(f\'rich_{label}_shoulder_{offset}\')\n        window = np.column_stack(profiles)\n        shoulder_columns.extend([window.max(axis=1) - window.min(axis=1), np.argmax(window, axis=1) * 60 - 120])\n        shoulder_names.extend([f\'rich_{label}_shoulder_range\', f\'rich_{label}_peak_minutes\'])\n    for signal, array in FEATURE_NS[\'network_values\'].items():\n        for lag in (0, 2):\n            signal_values = np.where(origins >= lag, array[np.maximum(origins - lag, 0)], 0) / max(float(segmentation[\'scale\'].sum()), 1.0)\n            shoulder_columns.append(np.repeat(signal_values, len(ROUTES)))\n            shoulder_names.append(f\'rich_network_{signal}_back_{lag}\')\n        change = (array[origins] - array[np.maximum(origins - 2, 0)]) / max(float(segmentation[\'scale\'].sum()), 1.0)\n        shoulder_columns.append(np.repeat(change, len(ROUTES)))\n        shoulder_names.append(f\'rich_network_{signal}_change_1h\')\n    shoulder_columns.append(np.tile(np.arange(len(ROUTES), dtype=np.float32), len(origins)))\n    shoulder_names.append(\'rich_route_index\')\n    share = VALUES[TARGET][origins] / np.maximum(FEATURE_NS[\'network_values\'][TARGET][origins, None], 1)\n    shoulder_columns.append(share.ravel())\n    shoulder_names.append(\'rich_route_target_network_share\')\n    matrix = np.column_stack([base, values, available, *[np.asarray(c, dtype=np.float32) for c in shoulder_columns]]).astype(np.float32, copy=False)\n    names = names + [\'rich_\' + c for c in cols] + [\'rich_available__\' + c for c in cols] + shoulder_names\n    if not np.isfinite(matrix).all():\n        raise ValueError(\'Неконечные расширенные признаки\')\n    return (matrix, names)\n\ndef model_frame(x, names, algorithm):\n    frame = pd.DataFrame(x, columns=names)\n    route_codes = x[:, names.index(\'rich_route_index\')].astype(np.int64)\n    frame[\'route_code\'] = pd.Categorical(route_codes, categories=np.arange(len(ROUTES))) if algorithm == \'xgb\' else route_codes.astype(str)\n    return frame\n\ndef fit_rich_cat(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        with threadpool_limits(limits=THREADS):\n            segmentation = segment_routes(start, start.strftime(\'%Y%m%d\'))\n        route_groups = segmentation[\'labels\'][8]\n        route_scale = segmentation[\'scale\']\n        origins = np.arange(len(TIMES) - 8)\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            train_origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < start)\n            query_origins = QUERY_ORIGINS\n            x, columns = rich_matrix(h, train_origins, segmentation)\n            query, query_columns = rich_matrix(h, query_origins, segmentation)\n            if columns != query_columns:\n                raise ValueError(\'Расходятся колонки CatBoost rich\')\n            y = VALUES[TARGET][train_origins + h]\n            actual = observed_query(h)\n            predictions = np.empty_like(actual, dtype=np.float64)\n            spec = PARAMETERS[\'selected_cat\'][str(h)]\n            for group in np.unique(route_groups):\n                query_routes = np.flatnonzero(route_groups == group)\n                train_routes = np.arange(len(ROUTES)) if group == -1 else query_routes\n                train_x = model_frame(x[:, train_routes].reshape(-1, len(columns)), columns, \'cat\')\n                train_y = y[:, train_routes].reshape(-1)\n                train_scale = np.tile(route_scale[train_routes], len(train_origins))\n                query_x = model_frame(query[:, query_routes].reshape(-1, len(columns)), columns, \'cat\')\n                model = CatBoostRegressor(**spec[\'params\'], iterations=spec[\'trees\'], loss_function=\'MAE\', task_type=\'GPU\', devices=\'0\', thread_count=THREADS, random_seed=SEED, verbose=False, allow_writing_files=False, allow_const_label=True, bootstrap_type=\'Bayesian\', cat_features=[\'route_code\'])\n                model.fit(train_x, train_y / train_scale, sample_weight=train_scale / route_scale.mean())\n                train_pred = np.maximum(model.predict(train_x) * train_scale, 0)\n                query_pred = np.maximum(model.predict(query_x).reshape(len(query_origins), -1) * route_scale[query_routes], 0)\n                predictions[:, query_routes] = query_pred\n                fit_report(\'cat_rich\', start, h, int(group), split, train_y, train_pred, actual[:, query_routes], query_pred)\n                model.save_model(str(folder / f\'h{h}_group{group}.cbm\'))\n                joblib.dump(segmentation, folder / \'segmentation.joblib\')\n                del model, train_x, query_x, train_pred\n                gc.collect()\n            frame = pd.DataFrame(dict(cutoff=TIMES[query_origins].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query_origins)), horizon=h, y=actual.ravel(), cat_rich=predictions.ravel()))\n            save_parquet(path, frame)\n            del x, query\n            gc.collect()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n\ndef rich_matrix(h, origins, segmentation):\n    if not len(origins):\n        raise ValueError(\'Пустой диапазон CatBoost rich\')\n    routes_count = len(ROUTES)\n    matrix = None\n    for start in range(0, len(origins), 32):\n        batch = origins[start:start + 32]\n        raw, names = features(h, batch, segmentation)\n        if matrix is None:\n            columns = names\n            matrix = np.empty((len(origins), routes_count, len(columns)), dtype=np.float32)\n        if names != columns:\n            raise ValueError(\'Расходятся колонки батчей CatBoost rich\')\n        matrix[start:start + len(batch)] = raw.reshape(len(batch), routes_count, len(columns))\n    return (matrix, columns)\n', 'cat_runtime', 'exec'), cat_runtime)


In [ ]:
cat_future = cat_runtime['fit_rich_cat'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
xgb_rich_runtime = dict(cat_runtime)
exec(compile("import time\nimport xgboost as xgb\nimport cupy as cp\nGPU_BATCH_ROWS = 65536\nGPU_MAX_BIN = 256\nGPU_QUANTILE_BATCHES = 8\nclass GPUTrainingIterator(xgb.DataIter):\n\n    def __init__(self, train, indices, segmentation):\n        self.train = train\n        self.indices = indices\n        self.segmentation = segmentation\n        self.position = 0\n        self.device_batch = None\n        super().__init__(release_data=True)\n\n    def reset(self):\n        self.position = 0\n        self.device_batch = None\n\n    def next(self, input_data):\n        if self.position >= len(self.indices):\n            return False\n        rows = self.indices[self.position:self.position + GPU_BATCH_ROWS]\n        self.position += len(rows)\n        x = gpu_features(self.train['x'][rows], self.train['names'])\n        scales = self.segmentation['scale'][self.train['route_index'][rows]]\n        y = cp.asarray(self.train['y'][rows] / scales, dtype=cp.float32)\n        weight = cp.asarray(scales / self.weight_scale, dtype=cp.float32)\n        self.device_batch = (x, y, weight)\n        input_data(data=x, label=y, weight=weight, feature_names=self.feature_names, feature_types=self.feature_types)\n        return True\n\ndef gpu_features(x, names):\n    values = cp.asarray(x, dtype=cp.float32)\n    route = values[:, names.index('rich_route_index')].reshape(-1, 1)\n    return cp.concatenate([values, route], axis=1)\n\ndef fit_model(train, segmentation, family, k, settings):\n    labels = segmentation['labels'][k][train['route_index']]\n    specification = PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]\n    feature_names = list(train['names']) + ['route_code']\n    feature_types = ['float'] * len(train['names']) + ['c']\n    params = dict(specification['params'], objective='reg:absoluteerror', eval_metric='mae', device='cuda:0', tree_method='hist', max_bin=GPU_MAX_BIN, max_cat_to_onehot=1, nthread=THREADS, seed=SEED)\n    models = {}\n    for group in np.unique(labels):\n        indices = np.arange(len(labels)) if group == -1 else np.flatnonzero(labels == group)\n        iterator = GPUTrainingIterator(train, indices, segmentation)\n        iterator.weight_scale = float(segmentation['scale'][train['route_index']].mean())\n        iterator.feature_names, iterator.feature_types = (feature_names, feature_types)\n        started = time.monotonic()\n        try:\n            matrix = xgb.QuantileDMatrix(iterator, max_bin=GPU_MAX_BIN, max_quantile_batches=GPU_QUANTILE_BATCHES, enable_categorical=True)\n            model = xgb.train(params, matrix, num_boost_round=specification['trees'])\n        except Exception as error:\n            raise RuntimeError('Ошибка CUDA-обучения. При нехватке VRAM уменьшите GPU_BATCH_ROWS/GPU_MAX_BIN или оставьте MODEL_CLUSTERS=(8,). CPU fallback не используется.') from error\n        if not json.loads(model.save_config())['learner']['generic_param']['device'].startswith('cuda'):\n            raise RuntimeError('XGBoost переключился на CPU')\n        models[int(group)] = dict(model=model, scaler=None)\n        del matrix, iterator\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n        print('GPU h', ACTIVE_H, 'k', k, 'group', int(group), 'rows', len(indices), 'seconds', round(time.monotonic() - started, 1), flush=True)\n    return dict(models=models, k=k, residual=False, feature_names=train['names'], experiment=ACTIVE)\n\ndef predict_model(bundle, data, segmentation):\n    labels = segmentation['labels'][bundle['k']][data['route_index']]\n    prediction = np.empty(len(labels), dtype=np.float64)\n    for group, item in bundle['models'].items():\n        indices = np.flatnonzero(labels == group)\n        item['model'].set_param(dict(device='cuda:0'))\n        for start in range(0, len(indices), GPU_BATCH_ROWS):\n            rows = indices[start:start + GPU_BATCH_ROWS]\n            x = gpu_features(data['x'][rows], bundle['feature_names'])\n            result = item['model'].inplace_predict(x, validate_features=False)\n            if not isinstance(result, cp.ndarray):\n                raise RuntimeError('Предсказание не вернуло CUDA-массив')\n            prediction[rows] = cp.asnumpy(result)\n            del x, result\n    return np.maximum(prediction * segmentation['scale'][data['route_index']], 0)\ndef fit_rich_xgb_full():\n    folder = ARTIFACTS / 'xgb_rich_full'\n    folder.mkdir(exist_ok=True)\n    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]\n    if all((p.is_file() for p in outputs)):\n        return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)\n    with threadpool_limits(limits=THREADS):\n        segmentation = segment_routes(FULL_BOUNDARY, 'full_xgb')\n    manifest = json.loads((XGB_DATA / 'manifest.json').read_text())\n    parameters = manifest['cat_configs'][0]['parameters']\n    for h, path in zip(HORIZONS, outputs):\n        if path.is_file():\n            continue\n        print('Full XGB rich', h, flush=True)\n        origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < FULL_BOUNDARY)\n        matrix, names = rich_matrix(h, origins, segmentation)\n        query, query_names = rich_matrix(h, QUERY_ORIGINS, segmentation)\n        if names != query_names:\n            raise ValueError('Разные признаки XGB rich')\n        train = dict(x=matrix.reshape(-1, len(names)), names=names, route_index=np.tile(np.arange(len(ROUTES)), len(origins)), y=VALUES[TARGET][origins + h].ravel())\n        globals()['PARAMETERS'] = parameters\n        globals()['ACTIVE'] = dict(parameter_set='selected_xgb', residual=False)\n        globals()['ACTIVE_H'] = h\n        bundle = fit_model(train, segmentation, 'xgb', 8, {})\n        pred = predict_model(bundle, train, segmentation)\n        future = predict_model(bundle, dict(x=query.reshape(-1, len(names)), route_index=np.tile(np.arange(len(ROUTES)), len(QUERY_ORIGINS))), segmentation)\n        refit_report('xgb_rich', FULL_BOUNDARY, h, 'all', 'future', train['y'], pred, observed_query(h).ravel(), future)\n        for group, item in bundle['models'].items():\n            item['model'].save_model(str(folder / f'h{h}_group{group}.ubj'))\n        joblib.dump(segmentation, folder / 'segmentation.joblib')\n        save_parquet(path, pd.DataFrame(dict(cutoff=TIMES[QUERY_ORIGINS].repeat(len(ROUTES)), horizon=h, route_id=np.tile(ROUTES, len(QUERY_ORIGINS)), xgb_rich=future)))\n        del matrix, query, train, pred, bundle\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)", 'rich_xgb_full', 'exec'), xgb_rich_runtime)
xgb_future = xgb_rich_runtime['fit_rich_xgb_full']()


In [ ]:
rf_runtime = dict(globals())
exec(compile('\n\nimport re\nimport cupy as cp\nfrom cuml.ensemble import RandomForestRegressor\nfrom sklearn.impute import SimpleImputer\n\ntimes, routes, values = TIMES, ROUTES, VALUES\nRUN = ARTIFACTS / \'rf\'\nRUN.mkdir(exist_ok=True)\n\n\nBASE_PARAMS_BY_HORIZON = {1: {\'n_estimators\': 150, \'max_depth\': 16, \'min_samples_leaf\': 44, \'max_features\': 0.9154095569322449, \'max_samples\': 0.7361074625809747}, 2: {\'n_estimators\': 300, \'max_depth\': 8, \'min_samples_leaf\': 37, \'max_features\': 0.6600327095074597, \'max_samples\': 0.5924272277627636}, 3: {\'n_estimators\': 300, \'max_depth\': 8, \'min_samples_leaf\': 37, \'max_features\': 0.6600327095074597, \'max_samples\': 0.5924272277627636}, 4: {\'n_estimators\': 150, \'max_depth\': 12, \'min_samples_leaf\': 37, \'max_features\': 0.6943109266465318, \'max_samples\': 0.5232252063599989}, 5: {\'n_estimators\': 150, \'max_depth\': 16, \'min_samples_leaf\': 44, \'max_features\': 0.9154095569322449, \'max_samples\': 0.7361074625809747}, 6: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 23, \'max_features\': 0.5045012539746527, \'max_samples\': 0.6831809216468459}, 7: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 18, \'max_features\': 0.7252813963310067, \'max_samples\': 0.5852620618436457}, 8: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 18, \'max_features\': 0.7252813963310067, \'max_samples\': 0.5852620618436457}}\nN_STREAMS = 4\nPREDICTION_BATCH = 32768\nTRAIN_MINUTES = (570, 600, 630, 660, 690)\nWINDOW_OFFSETS = tuple(range(-4, 5))\nLAG_MAX_LOOKBACK_HOURS = 24\nLAG_MIN_PAIRS = 336\nLAG_MIN_CORRELATION = 0.05\nRESIDUAL = False\n\nbest_params = BASE_PARAMS_BY_HORIZON\nselected = pd.DataFrame(json.loads(\'[{"horizon":2,"trimmed":false,"route_median":true,"experiment":"thin_offsets","mean":0.3472783742,"std":0.0237257355,"max":0.3845543627},{"horizon":6,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":0.3505317232,"std":0.0160416261,"max":0.3713695576},{"horizon":1,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null},{"horizon":3,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":4,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":5,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":7,"trimmed":false,"route_median":false,"experiment":"baseline","mean":null,"std":null,"max":null},{"horizon":8,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null}]\')).set_index(\'horizon\')\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals[\'status_6_diff_hourly\']\nextra_signals[\'target_vs_mean_6h\'] = target - means[TARGET, 12]\nrecent = means[TARGET, 4]\nusual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])\nextra_signals[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n\ndef offset_name(base, offset):\n    if offset == 0:\n        return base\n    return f"{base}_offset_{(\'m\' if offset < 0 else \'p\')}{abs(offset)}"\n\ndef add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):\n    for offset in offsets:\n        common[offset_name(base, offset)] = past(array, anchor + offset)\n\ndef common_features(origins):\n    common = {}\n    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})\n    for lag in local_target_lags:\n        common[f\'target_back_{lag}\'] = past(target, origins - lag)\n    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))\n    for name, array in values.items():\n        for steps in (4, 12):\n            add_window(common, f\'{name}_mean_{steps}\', means[name, steps], origins, causal_offsets)\n        if name in STATUSES:\n            common[f\'{name}_latest\'] = array[origins]\n            for lag in range(1, 5):\n                common[f\'{name}_back_{lag}\'] = past(array, origins - lag)\n            if name in STATUSES[:3]:\n                change = array - past(array, positions - 2)\n                add_window(common, f\'{name}_change_1h\', change, origins, causal_offsets)\n    for name, array in extra_signals.items():\n        add_window(common, name, array, origins, causal_offsets)\n    for lag in (48, 96):\n        add_window(common, f\'target_1h_lag_{lag}\', target, origins - lag)\n    add_window(common, \'warehouse_total_load_lag_48\', extra_signals[\'warehouse_total_load\'], origins - 48)\n    add_window(common, \'status_6_diff_hourly_lag_48\', hub_signal, origins - 48)\n    return common\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\nfor name, series in values.items():\n    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]\n    for suffix, signal in [(\'latest\', network), (\'change_1h\', network - past(network, positions - 2))]:\n        extra_signals[f\'network_{name}_{suffix}\'] = np.broadcast_to(signal, target.shape)\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\n\ndef correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError(\'Некорректный горизонт или размер блока\')\n    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in (\'lagged_status6_signal\', \'lagged_status6_signal_available\')]\n    names = CORRELATION_COLUMNS + extra_columns\n    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)\n    if lag_table is None:\n        names = [prefix + name for name in names]\n        return pd.concat([frame.drop(columns=names, errors=\'ignore\'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError(\'Строка не сопоставлена с историей\')\n        selected = lag[route]\n        for j, name in enumerate(CORRELATION_COLUMNS[:4]):\n            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]\n        for offset in WINDOW_OFFSETS:\n            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset\n            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))\n            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n            available &= np.isfinite(signal)\n            result[start:finish, names.index(offset_name(\'lagged_status6_signal\', offset))] = np.where(available, signal, np.nan)\n            result[start:finish, names.index(offset_name(\'lagged_status6_signal_available\', offset))] = available.astype(np.float32)\n    names = [prefix + name for name in names]\n    frame = frame.drop(columns=names, errors=\'ignore\')\n    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)\n\ndef remove_seasonality(series):\n    typical = average_available([past(series, positions - 48 * d) for d in range(1, 8)])\n    return series - typical\nresidual_target = remove_seasonality(target)\nresidual_signal = remove_seasonality(hub_signal)\n\ndef fit_lags(boundary):\n    tables = []\n    for y, signal, prefix in ((target, hub_signal, \'\'), (residual_target, residual_signal, \'residual_\')):\n        table = calculate_dynamic_shoulder(y, signal, times, routes, fit_before=boundary, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)\n        tables.append(table.rename(columns={c: prefix + c for c in table.columns if c != \'route_id\'}))\n    return tables[0].merge(tables[1], on=\'route_id\', validate=\'one_to_one\')\n\ndef add_correlation_features(frame, h, lag_table):\n    frame = correlation_block(frame, h, lag_table, hub_signal, \'\')\n    if lag_table is None:\n        return correlation_block(frame, h, None, residual_signal, \'residual_\')\n    residual_table = lag_table[[\'route_id\', *[\'residual_\' + c for c in CORRELATION_COLUMNS[:4]]]].copy()\n    residual_table.columns = [\'route_id\', *CORRELATION_COLUMNS[:4]]\n    return correlation_block(frame, h, residual_table, residual_signal, \'residual_\')\n\ndef stable_ratio(numerator, denominator):\n    return numerator / np.maximum(np.abs(denominator), 1.0)\n\ndef available_statistics(window):\n    valid = np.isfinite(window)\n    count = valid.sum(axis=0)\n    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count, out=np.full(window.shape[1:], np.nan), where=count > 0)\n    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count, out=np.full_like(mean, np.nan), where=count > 0)\n    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)\n    return (mean, np.sqrt(variance), spread, count)\n\ndef add_experimental_features(frame, h, origins):\n    additions = {}\n    additions[\'x_time_cutoff_minutes\'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))\n    additions[\'x_profile_profile28\'] = frame.profile28.to_numpy()\n    additions[\'x_level_level24\'] = level24[origins].ravel()\n    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])\n    current = target[origins]\n    additions[\'x_anomaly_target_difference\'] = (current - typical).ravel()\n    additions[\'x_anomaly_target_ratio\'] = stable_ratio(current, typical).ravel()\n    for name in (TARGET, *STATUSES[:3]):\n        short, long = (means[name, 4][origins], means[name, 12][origins])\n        additions[f\'x_trend_{name}_difference\'] = (short - long).ravel()\n        additions[f\'x_trend_{name}_ratio\'] = stable_ratio(short, long).ravel()\n        for steps in (4, 12):\n            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])\n            mean, std, spread, count = available_statistics(window)\n            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]\n            additions[f\'x_volatility_{name}_std_{steps}\'] = std.ravel()\n            additions[f\'x_volatility_{name}_range_{steps}\'] = spread.ravel()\n            additions[f\'x_volatility_{name}_change_{steps}\'] = change.ravel()\n    for label, days in [(\'daily7\', range(1, 8)), (\'weekly4\', (7, 14, 21, 28))]:\n        window = np.stack([past(target, origins + h - day * 48) for day in days])\n        mean, std, spread, count = available_statistics(window)\n        additions[f\'x_dispersion_{label}_std\'] = std.ravel()\n        additions[f\'x_dispersion_{label}_relative_std\'] = stable_ratio(std, mean).ravel()\n        additions[f\'x_dispersion_{label}_count\'] = count.ravel()\n    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    common = common_features(origins)\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})\n    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, None)\n    seasonal_columns = {}\n    for offset in WINDOW_OFFSETS:\n        for day in (1, 2, 7):\n            seasonal_columns[offset_name(f\'slot_days_back_{day}\', offset)] = past(target, slot - day * 48 + offset).ravel()\n        for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\n            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()\n    for base in (\'slot_daily7\', \'slot_weekly4\'):\n        window = np.column_stack([seasonal_columns[offset_name(base, k)] for k in WINDOW_OFFSETS])\n        valid = np.isfinite(window)\n        maximum = np.where(valid, window, -np.inf).max(axis=1)\n        minimum = np.where(valid, window, np.inf).min(axis=1)\n        peak = valid & (window == maximum[:, None])\n        count = peak.sum(axis=1)\n        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window), np.nan), where=count > 0)\n        seasonal_columns[base + \'_peak_minutes\'] = shift * 30\n        seasonal_columns[base + \'_range\'] = np.where(count > 0, maximum - minimum, np.nan)\n        seasonal_columns[base + \'_right_minus_left\'] = window[:, 5:].mean(axis=1) - window[:, :4].mean(axis=1)\n    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame[\'overlap\'] = np.nan\n    raw_target_sources = []\n    for lag in range(9):\n        raw_target_sources.append((f\'target_back_{lag}\', -lag, lag in (0, 1, 2, 4)))\n    for lag in (48, 96):\n        for offset in WINDOW_OFFSETS:\n            raw_target_sources.append((offset_name(f\'target_1h_lag_{lag}\', offset), -lag + offset, offset == 0))\n    for day in (1, 2, 7):\n        for offset in WINDOW_OFFSETS:\n            raw_target_sources.append((offset_name(f\'slot_days_back_{day}\', offset), h - day * 48 + offset, offset == 0))\n    seen, redundant = (set(), [])\n    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):\n        if source in seen:\n            redundant.append(name)\n        else:\n            seen.add(source)\n    frame = frame.drop(columns=redundant)\n    frame = add_experimental_features(frame, h, origins)\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\n    return frame.astype({name: \'float32\' for name in numeric}).copy()\n\nTARGET_SCALE = 100000\n\ndef model_columns(data):\n    return data.columns.drop([\'timestamp\', \'cutoff\', \'y\', \'profile28\', \'overlap\']).tolist()\n\nEXPERIMENTS = {\'baseline\': dict(), \'drop_constant\': dict(drop_constant=True), \'compact_availability\': dict(compact_availability=True), \'calendar_dayofweek\': dict(calendar=True), \'thin_offsets\': dict(thin=True), \'drop_network\': dict(network=\'drop\'), \'thin_network\': dict(network=\'thin\'), \'add_time\': dict(add=[\'time\']), \'add_profile\': dict(add=[\'profile\']), \'add_level24\': dict(add=[\'level\']), \'add_anomaly\': dict(add=[\'anomaly\']), \'add_trends\': dict(add=[\'trend\']), \'add_volatility\': dict(add=[\'volatility\']), \'add_seasonal_dispersion\': dict(add=[\'dispersion\']), \'all_additions\': dict(add=[\'time\', \'profile\', \'level\', \'anomaly\', \'trend\', \'volatility\', \'dispersion\']), \'compact\': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network=\'thin\'), \'compact_with_additions\': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network=\'thin\', add=[\'time\', \'profile\', \'level\', \'anomaly\', \'trend\', \'volatility\', \'dispersion\'])}\n\ndef model_target(frame):\n    baseline = frame.profile28.to_numpy() if RESIDUAL else 0\n    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)\n\ndef transformed_frame(frame, columns):\n    result = frame[columns].drop(columns=\'route_id\').astype(np.float32)\n    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1\n    bits = {f\'route_bit_{i}\': (codes >> i & 1).astype(np.float32) for i in range(max(1, int(len(routes)).bit_length()))}\n    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()\n\ndef fit_forest(frame, matrix, params):\n    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type=\'numpy\', **params)\n    model.fit(matrix, model_target(frame))\n    cp.cuda.get_current_stream().synchronize()\n    return model\n\ndef predict_matrix(model, frame, matrix):\n    result = np.empty(len(frame), dtype=np.float64)\n    for start in range(0, len(frame), PREDICTION_BATCH):\n        stop = min(start + PREDICTION_BATCH, len(frame))\n        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)\n        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0\n        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)\n    if not np.isfinite(result).all():\n        raise ValueError(\'Некорректный прогноз\')\n    return result\n\ndef experiment_columns(names, spec):\n    result = []\n    for name in names:\n        if name.startswith(\'x_\'):\n            if any((name.startswith(f\'x_{group}_\') for group in spec.get(\'add\', []))):\n                result.append(name)\n            continue\n        if spec.get(\'drop_constant\') and name == \'selected_status6_pair_count\':\n            continue\n        if spec.get(\'compact_availability\') and \'signal_available\' in name:\n            continue\n        if spec.get(\'calendar\') and name in (\'weekday_sin\', \'weekday_cos\', \'is_weekend\'):\n            continue\n        if spec.get(\'network\') == \'drop\' and name.startswith(\'network_\'):\n            continue\n        if spec.get(\'network\') == \'thin\' and name.startswith(\'network_\') and (\'_offset_\' in name):\n            continue\n        if spec.get(\'thin\') and (not name.startswith(\'slot_\')) and re.search(\'_offset_m[134]$\', name):\n            continue\n        result.append(name)\n    if spec.get(\'compact_availability\'):\n        for name in (\'status6_lag_valid\', \'residual_status6_lag_valid\'):\n            if name in names and name not in result:\n                result.append(name)\n    return result\n\ndef rf_data(h, origins, lag_table, spec):\n    batches, parts = ([], [])\n    for start in range(0, len(origins), 24):\n        batch = origins[start:start + 24]\n        frame = horizon_dataset(h, batch)\n        frame[\'y\'] = past(target, batch + h).ravel()\n        frame = frame.loc[np.isfinite(frame.profile28)].reset_index(drop=True)\n        frame = add_correlation_features(frame, h, lag_table)\n        matrix = transformed_frame(frame, model_columns(frame))\n        columns = experiment_columns(list(matrix.columns), EXPERIMENTS[spec.experiment])\n        batches.append(matrix[columns].replace([np.inf, -np.inf], np.nan))\n        parts.append(frame[[\'cutoff\', \'timestamp\', \'route_id\', \'y\', \'profile28\']].copy())\n    return (pd.concat(parts, ignore_index=True), pd.concat(batches, ignore_index=True))\n\ndef fit_rf(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        lag_table = fit_lags(start)\n        minute = times.hour * 60 + times.minute\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            spec = selected.loc[h]\n            if bool(spec.trimmed):\n                raise ValueError(\'Для этого рецепта RF ожидается полная обучающая история\')\n            train_origins = np.flatnonzero(np.isin(minute, TRAIN_MINUTES) & (times + h * STEP < start))\n            query_origins = QUERY_ORIGINS\n            train, train_x = rf_data(h, train_origins, lag_table, spec)\n            query, query_x = rf_data(h, query_origins, lag_table, spec)\n            if bool(spec.route_median):\n                median = train_x.groupby(train.route_id).median()\n                train_x = train_x.fillna(median.reindex(train.route_id).set_axis(train_x.index))\n                query_x = query_x.fillna(median.reindex(query.route_id).set_axis(query_x.index))\n            imputer = SimpleImputer(strategy=\'median\', keep_empty_features=True)\n            train_matrix = imputer.fit_transform(train_x).astype(np.float32, copy=False)\n            query_matrix = imputer.transform(query_x).astype(np.float32, copy=False)\n            del train_x, query_x\n            model = fit_forest(train, train_matrix, best_params[h])\n            train_pred = predict_matrix(model, train, train_matrix)\n            query_pred = predict_matrix(model, query, query_matrix)\n            fit_report(\'rf\', start, h, \'all\', split, train.y, train_pred, query.y, query_pred)\n            save_parquet(path, query[[\'cutoff\', \'route_id\', \'y\']].assign(horizon=h, rf=query_pred))\n            joblib.dump(dict(model=model, imputer=imputer, median=median if bool(spec.route_median) else None, lag_table=lag_table, spec=spec), folder / f\'h{h}_model.joblib\')\n            del model, train_matrix, query_matrix, train, query, train_pred, query_pred\n            gc.collect()\n            cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n', 'rf_runtime', 'exec'), rf_runtime)


In [ ]:
rf_future = rf_runtime['fit_rf'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
clean_xgb_runtime = dict(globals())
exec(compile('\n\nimport xgboost as xgb\nimport cupy as cp\n\nRUN = ARTIFACTS / \'xgb_clean\'\nRUN.mkdir(exist_ok=True)\nCLEAN_XGB_CONTEXT = None\n\n\nTARGET_SCALE = 100000.0\nOBJECTIVE = \'reg:absoluteerror\'\nCLEANING = {\'window_days\': 28, \'min_days\': 14, \'z_threshold\': 4.0, \'log_scale_floor\': 0.25, \'mad_normalization\': 1.4826}\nLAG_MAX_LOOKBACK_HOURS = 24\nLAG_MIN_PAIRS = 336\nLAG_MIN_CORRELATION = 0.05\nBEST_PARAMS = {\'1\': {\'n_estimators\': 2530, \'max_depth\': 6, \'learning_rate\': 0.036760715868196514, \'min_child_weight\': 1.0809355847970479, \'subsample\': 0.9003563537429966, \'colsample_bytree\': 0.6639976846136139, \'reg_alpha\': 9.305271303855188e-05, \'reg_lambda\': 7.596418498005285}, \'2\': {\'n_estimators\': 1419, \'max_depth\': 4, \'learning_rate\': 0.10190438495585248, \'min_child_weight\': 1.8389812995413264, \'subsample\': 0.6817515181500764, \'colsample_bytree\': 0.6619351860505892, \'reg_alpha\': 0.0022965441587741537, \'reg_lambda\': 14.08596091970978}, \'3\': {\'n_estimators\': 1930, \'max_depth\': 3, \'learning_rate\': 0.07005359250881395, \'min_child_weight\': 2.2064897603579636, \'subsample\': 0.8779096005107611, \'colsample_bytree\': 0.7846752865583575, \'reg_alpha\': 0.018954349619526665, \'reg_lambda\': 1.043977626067855}, \'4\': {\'n_estimators\': 1827, \'max_depth\': 4, \'learning_rate\': 0.02925325002470326, \'min_child_weight\': 1.5841239345284504, \'subsample\': 0.8126045473410459, \'colsample_bytree\': 0.7066552862291448, \'reg_alpha\': 2.3192129195348187e-05, \'reg_lambda\': 0.22202924731915144}, \'5\': {\'n_estimators\': 1387, \'max_depth\': 3, \'learning_rate\': 0.03164978915749773, \'min_child_weight\': 1.9902815543275585, \'subsample\': 0.6844098448217815, \'colsample_bytree\': 0.7432566543439275, \'reg_alpha\': 0.010252186188681373, \'reg_lambda\': 0.5427730704771981}, \'6\': {\'n_estimators\': 1566, \'max_depth\': 3, \'learning_rate\': 0.023401679328514332, \'min_child_weight\': 3.1806793320017555, \'subsample\': 0.87035767368126, \'colsample_bytree\': 0.7980364720685608, \'reg_alpha\': 0.02123901077609503, \'reg_lambda\': 0.3328783942599951}, \'7\': {\'n_estimators\': 1136, \'max_depth\': 5, \'learning_rate\': 0.0330759240922086, \'min_child_weight\': 3.8692684642092345, \'subsample\': 0.6794525496919193, \'colsample_bytree\': 0.6799389555259473, \'reg_alpha\': 2.7314479492664567, \'reg_lambda\': 0.3796668381307971}, \'8\': {\'n_estimators\': 534, \'max_depth\': 4, \'learning_rate\': 0.09417045025581995, \'min_child_weight\': 13.418857993305474, \'subsample\': 0.689895840619612, \'colsample_bytree\': 0.6918995986906583, \'reg_alpha\': 0.3746751674810787, \'reg_lambda\': 1.8540896283895405}}\n\ndef clean_raw_array(array, timestamps, settings):\n    array = np.asarray(array, dtype=np.float32)\n    if not np.isfinite(array).all() or (array < 0).any():\n        raise ValueError(\'Для log1p нужны конечные неотрицательные исходные значения\')\n    cleaned = array.copy()\n    flagged = np.zeros(array.shape, dtype=bool)\n    slots = timestamps.hour * 2 + timestamps.minute // 30\n    for slot in range(48):\n        positions = np.flatnonzero(slots == slot)\n        logged = np.log1p(array[positions].astype(np.float64))\n        window_days = settings[\'window_days\']\n        min_days = settings[\'min_days\']\n        if not 1 <= min_days <= window_days:\n            raise ValueError(\'Нужно 1 <= min_days <= window_days\')\n        if len(positions) <= min_days:\n            continue\n        padded = np.pad(logged, ((window_days, 0), (0, 0)), constant_values=np.nan)\n        windows = np.lib.stride_tricks.sliding_window_view(padded, window_days, axis=0)[:-1]\n        eligible = np.arange(len(positions)) >= min_days\n        past = windows[eligible]\n        median = np.nanmedian(past, axis=-1)\n        mad = np.nanmedian(np.abs(past - median[..., None]), axis=-1)\n        scale = np.maximum(settings[\'mad_normalization\'] * mad, settings[\'log_scale_floor\'])\n        mask = np.abs(logged[eligible] - median) > settings[\'z_threshold\'] * scale\n        destination = positions[eligible]\n        cleaned[destination] = np.where(mask, np.expm1(median), array[destination]).astype(np.float32)\n        flagged[destination] = mask\n    return (cleaned, flagged)\n\nFEATURE_CODE = "import pyarrow as pa\\nimport pyarrow.parquet as pq\\ntarget = values[TARGET]\\n\\ndef past(array, positions):\\n    positions = np.asarray(positions, dtype=np.int64)\\n    valid = (positions >= 0) & (positions < len(array))\\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\\n    result[~valid] = np.nan\\n    return result\\n\\ndef average_available(matrices):\\n    total = np.zeros_like(matrices[0], dtype=np.float32)\\n    count = np.zeros_like(total)\\n    for matrix in matrices:\\n        valid = np.isfinite(matrix)\\n        total += np.where(valid, matrix, 0)\\n        count += valid\\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\\n\\ndef rolling(array, steps):\\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\\nlevel24 = rolling(target, 48)\\npositions = np.arange(len(times))\\nextra_signals = {}\\nfor name in STATUSES[:3]:\\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\\nfor name in STATUSES[3:]:\\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\\nhub_signal = extra_signals[\'status_6_diff_hourly\']\\n\\ndef common_features(origins):\\n    common = {}\\n    for k in (0, 1, 2, 4):\\n        common[f\'target_back_{k}\'] = past(target, origins - k)\\n    for name, array in values.items():\\n        for steps in (4, 12):\\n            common[f\'{name}_mean_{steps}\'] = means[name, steps][origins]\\n        if name in STATUSES:\\n            common[f\'{name}_latest\'] = array[origins]\\n            if name in STATUSES[:3]:\\n                common[f\'{name}_change_1h\'] = array[origins] - past(array, origins - 2)\\n    for name, array in extra_signals.items():\\n        common[name] = array[origins]\\n    for lag in (48, 96):\\n        common[f\'target_1h_lag_{lag}\'] = past(target, origins - lag)\\n    common[\'warehouse_total_load_lag_48\'] = past(extra_signals[\'warehouse_total_load\'], origins - 48)\\n    common[\'status_6_diff_hourly_lag_48\'] = past(hub_signal, origins - 48)\\n    common[\'target_vs_mean_6h\'] = target[origins] - means[TARGET, 12][origins]\\n    recent = means[TARGET, 4][origins]\\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\\n    common[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\\n    return common\\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\\nFEATURE_GROUPS = {\'calendar\': [\'hour\', \'minute\', \'hour_float\', \'hour_sin\', \'hour_cos\', \'dayofweek\', \'is_weekend\', \'weekday_sin\', \'weekday_cos\'], \'local_statuses\': [\'status_1_hourly\', \'status_2_hourly\', \'status_3_hourly\', \'warehouse_total_load\', \'warehouse_processing_rate\'], \'hub_combinations\': [\'status_4_diff_hourly\', \'status_5_diff_hourly\', \'status_6_diff_hourly\', \'hub_buffer\', \'hub_gate_bottleneck\'], \'historical_lags\': [\'target_1h_lag_48\', \'target_1h_lag_96\', \'warehouse_total_load_lag_48\', \'status_6_diff_hourly_lag_48\'], \'target_change\': [\'target_acceleration\'], \'correlation_lag\': CORRELATION_COLUMNS, \'pilot_start_time\': [\'start_sin\', \'start_cos\'], \'pilot_peaks\': [\'peak7_mean_max\', \'peak7_mean_level\', \'peak7_share_-1\', \'peak7_share_0\', \'peak7_share_1\']}\\n\\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\\n    grid = pd.DatetimeIndex(timestamps)\\n    route_ids = np.asarray(route_ids)\\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\\n    if len(np.unique(route_ids)) != len(route_ids):\\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\\n    steps = float(max_lookback_hours) * 2\\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\\n    max_lag = int(steps)\\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\\n    anchors = np.arange(max_lag + 2, stop)\\n    best_corr = np.full(len(route_ids), np.nan)\\n    best_lag = np.full(len(route_ids), np.nan)\\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\\n    for start in range(0, len(route_ids), route_block_size):\\n        finish = min(start + route_block_size, len(route_ids))\\n        x = target_values[anchors, start:finish].astype(np.float64)\\n        for lag in range(2, max_lag + 1):\\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\\n            valid = np.isfinite(x) & np.isfinite(z)\\n            count = valid.sum(axis=0)\\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\\n            divisor = np.maximum(count, 1)\\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\\n            denominator = np.sqrt(vx * vz)\\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\\n            corr = np.clip(corr, -1, 1)\\n            previous = best_corr[start:finish]\\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\\n            best_corr[start:finish][better] = corr[better]\\n            best_lag[start:finish][better] = lag\\n            best_pairs[start:finish][better] = count[better]\\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\\n\\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\\n    if h not in range(1, 9) or chunk_rows < 1:\\n        raise ValueError(\'Некорректный горизонт или размер блока\')\\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\\n    route_index = pd.Index(routes)\\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\\n    for name in CORRELATION_COLUMNS:\\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\\n    for start in range(0, len(frame), chunk_rows):\\n        finish = min(start + chunk_rows, len(frame))\\n        block = frame.iloc[start:finish]\\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\\n        route = route_index.get_indexer(block.route_id)\\n        if (origin < 0).any() or (route < 0).any():\\n            raise ValueError(\'Строка признаков не сопоставлена с историей\')\\n        selected = lag[route]\\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\\n        available &= np.isfinite(signal)\\n        for name, array in quality.items():\\n            frame.iloc[start:finish, column_positions[name]] = array[route]\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal\']] = np.where(available, signal, np.nan)\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal_available\']] = available.astype(np.float32)\\n    return frame\\n\\ndef horizon_dataset(h, origins):\\n    slot = origins + h\\n    dates = times[origins] + h * STEP\\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins))})\\n    for name, matrix in common_features(origins).items():\\n        frame[name] = matrix.ravel()\\n    calendar = {\'hour\': dates.hour, \'minute\': dates.minute, \'hour_float\': dates.hour + dates.minute / 60.0, \'dayofweek\': dates.dayofweek, \'is_weekend\': (dates.dayofweek >= 5).astype(np.int8)}\\n    calendar[\'hour_sin\'] = np.sin(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'hour_cos\'] = np.cos(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\\n    for name, array in calendar.items():\\n        frame[name] = np.repeat(array, len(routes))\\n    frame = add_correlation_features(frame, h, validation_lag_table)\\n    for days in (1, 2, 7):\\n        frame[f\'slot_days_back_{days}\'] = past(target, slot - days * 48).ravel()\\n    for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\\n    profile = np.zeros((len(origins), len(routes)))\\n    counts = np.zeros_like(profile)\\n    for day in range(1, 29):\\n        level = past(level24, origins - day * 48)\\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\\n        profile += np.nan_to_num(adjusted)\\n        counts += np.isfinite(adjusted)\\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\\n    if h == 1:\\n        lower = np.zeros_like(target[origins])\\n        upper = past(target, origins - 2)\\n        for position in (origins - 1, origins):\\n            total = past(target, position)\\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\\n    else:\\n        frame[\'overlap\'] = np.nan\\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\\n    return frame.astype({name: \'float32\' for name in numeric})\\nbase_horizon_dataset = horizon_dataset\\nnetwork_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}\\n\\ndef ratio_features(current, usual):\\n    denominator = np.maximum(usual, np.float32(1.0))\\n    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))\\n    return (ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32))\\n\\ndef horizon_dataset(h, origins):\\n    frame = base_horizon_dataset(h, origins)\\n    added = {}\\n    slot = origins + h\\n    profile = {}\\n    for offset in (-2, -1, 0, 1, 2):\\n        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])\\n        if offset != 0:\\n            added[f\'slot_daily7_neighbor_{offset:+d}\'] = profile[offset].ravel()\\n    added[\'slot_daily7_change_2h\'] = (profile[2] - profile[-2]).ravel()\\n    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])\\n    finite = np.isfinite(stack)\\n    masked = np.where(finite, stack, -np.inf)\\n    peak = masked.max(axis=0)\\n    available = finite.any(axis=0)\\n    added[\'slot_daily7_peak_value\'] = np.where(available, peak, np.nan).ravel()\\n    added[\'slot_daily7_peak_offset_minutes\'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()\\n    for name in STATUSES[:3]:\\n        for suffix, array in [(\'latest\', values[name]), (\'mean_4\', means[name, 4])]:\\n            current = array[origins]\\n            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\\n            ratio, small = ratio_features(current, usual)\\n            added[f\'{name}_{suffix}_vs_usual7\'] = (current - usual).ravel()\\n            added[f\'{name}_{suffix}_ratio_usual7\'] = ratio.ravel()\\n            added[f\'{name}_{suffix}_usual7_small\'] = small.ravel()\\n    for day in (14, 21, 28):\\n        added[f\'slot_days_back_{day}\'] = past(target, slot - day * 48).ravel()\\n    added[\'slot_week1_minus_week2\'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()\\n    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])\\n    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])\\n    added[\'slot_recent3_minus_previous4\'] = (recent - previous).ravel()\\n    for name, array in network_values.items():\\n        current = array[origins]\\n        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\\n        ratio, small = ratio_features(current, usual)\\n        features = {\'latest\': current, \'change_1h\': current - past(array, origins - 2), \'vs_usual7\': current - usual, \'ratio_usual7\': ratio, \'usual7_small\': small}\\n        for suffix, feature in features.items():\\n            added[f\'network_{name}_{suffix}\'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()\\n        local = values[name][origins]\\n        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)\\n        past_total = past(array, origins - 2)\\n        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)\\n        added[f\'network_{name}_route_share\'] = share.ravel()\\n        added[f\'network_{name}_route_share_change_1h\'] = (share - old_share).ravel()\\n    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)"\n\nclass CleanXgbBatches(xgb.DataIter):\n\n    def __init__(self, matrix, labels, columns):\n        self.matrix = matrix\n        self.labels = labels\n        self.columns = columns\n        self.types = [\'c\' if column == \'route_id\' else \'q\' for column in columns]\n        self.position = 0\n        super().__init__(release_data=True)\n\n    def reset(self):\n        self.position = 0\n\n    def next(self, input_data):\n        if self.position >= len(self.labels):\n            return False\n        start = self.position\n        stop = min(start + 65536, len(self.labels))\n        self.position = stop\n        input_data(data=cp.asarray(self.matrix[start:stop]), label=cp.asarray(self.labels[start:stop]), feature_names=self.columns, feature_types=self.types)\n        return True\n\ndef clean_xgb_matrix(h, origins, features):\n    route_count = len(ROUTES)\n    route_codes = np.arange(route_count, dtype=np.float32)\n    matrix = None\n    for start in range(0, len(origins), 24):\n        block = origins[start:start + 24]\n        frame = features(h, block)\n        columns = frame.columns.drop([\'cutoff\', \'timestamp\', \'overlap\']).tolist()\n        frame[\'route_id\'] = np.tile(route_codes, len(block))\n        if matrix is None:\n            names = columns\n            matrix = np.empty((len(origins) * route_count, len(names)), dtype=np.float32)\n        if columns != names:\n            raise ValueError(\'Расходятся колонки обычного XGBoost\')\n        matrix[start * route_count:(start + len(block)) * route_count] = frame[columns].to_numpy(dtype=np.float32)\n    if matrix is None:\n        raise ValueError(\'Пустая матрица обычного XGBoost\')\n    return (matrix, names)\n\ndef predict_clean_xgb(model, matrix):\n    result = np.empty(len(matrix), dtype=np.float64)\n    for start in range(0, len(matrix), 65536):\n        stop = min(start + 65536, len(matrix))\n        result[start:stop] = cp.asnumpy(model.inplace_predict(cp.asarray(matrix[start:stop]))) * TARGET_SCALE\n    return np.maximum(result, 0)\n\ndef clean_xgb_features():\n    global CLEAN_XGB_CONTEXT\n    if CLEAN_XGB_CONTEXT is None:\n        values = {}\n        rows = []\n        for name in [TARGET, *STATUSES]:\n            values[name], flagged = clean_raw_array(VALUES[name], TIMES, CLEANING)\n            rows.append(dict(column=name, replaced=int(flagged.sum()), rows=flagged.size))\n        pd.DataFrame(rows).to_csv(RUN / \'cleaning_counts.csv\', index=False)\n        CLEAN_XGB_CONTEXT = dict(np=np, pd=pd, TARGET=TARGET, STATUSES=STATUSES, STEP=STEP, values=values, times=TIMES, routes=ROUTES)\n        exec(FEATURE_CODE, CLEAN_XGB_CONTEXT)\n    return CLEAN_XGB_CONTEXT\n\ndef fit_clean_xgb(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        features = clean_xgb_features()\n        target = features[\'target\']\n        lag_table = features[\'calculate_dynamic_shoulder\'](target, features[\'hub_signal\'], TIMES, ROUTES, fit_before=start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)\n        features[\'validation_lag_table\'] = lag_table\n        lag_table.to_csv(folder / \'status6_lags.csv\', index=False)\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            origins = np.arange(len(TIMES) - h)\n            train = origins[TIMES[origins + h] < start]\n            query = QUERY_ORIGINS\n            print(\'XGBoost clean\', split, start.date(), f\'h{h}\', flush=True)\n            matrix, columns = clean_xgb_matrix(h, train, features[\'horizon_dataset\'])\n            labels = (target[train + h].ravel() / TARGET_SCALE).astype(np.float32)\n            batches = CleanXgbBatches(matrix, labels, columns)\n            quantile = xgb.QuantileDMatrix(batches, max_bin=256, max_quantile_batches=8, enable_categorical=True)\n            params = BEST_PARAMS[str(h)].copy()\n            trees = params.pop(\'n_estimators\')\n            params.update(objective=OBJECTIVE, eval_metric=\'mae\', tree_method=\'hist\', device=\'cuda:0\', max_bin=256, max_cat_to_onehot=1, seed=SEED, nthread=THREADS)\n            model = xgb.train(params, quantile, num_boost_round=trees)\n            device = json.loads(model.save_config())[\'learner\'][\'generic_param\'][\'device\']\n            if not device.startswith(\'cuda\') or model.feature_types != batches.types:\n                raise RuntimeError(\'XGBoost clean потерял CUDA или категориальный route_id\')\n            del quantile, batches\n            cp.get_default_memory_pool().free_all_blocks()\n            train_prediction = predict_clean_xgb(model, matrix)\n            del matrix\n            query_matrix, query_columns = clean_xgb_matrix(h, query, features[\'horizon_dataset\'])\n            if columns != query_columns:\n                raise ValueError(\'Расходятся колонки train и проверки обычного XGBoost\')\n            prediction = predict_clean_xgb(model, query_matrix)\n            actual = observed_query(h).ravel()\n            fit_report(\'xgb_clean\', start, h, \'all\', split, VALUES[TARGET][train + h].ravel(), train_prediction, actual, prediction)\n            frame = pd.DataFrame(dict(cutoff=TIMES[query].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query)), horizon=h, y=actual, xgb_clean=prediction))\n            save_parquet(path, frame)\n            model.save_model(str(folder / f\'h{h}_model.ubj\'))\n            del model, query_matrix, train_prediction\n            gc.collect()\n            cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n', 'clean_xgb_runtime', 'exec'), clean_xgb_runtime)


In [ ]:
clean_future = clean_xgb_runtime['fit_clean_xgb'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
import joblib
import torch
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
PAIRS = BASES[1:]
VOLUME_INPUTS = [name for name in PAIRS if name != 'xgb_clean']
SHAPES = ['session6', 'equal', 'joint', 'joint_without_xgb_clean', 'by_horizon', *[f'pair_{name}' for name in PAIRS], 'ridge_100', 'ridge_1000', 'cat_d2', 'cat_d4', 'xgb_d2', 'xgb_d3', 'gating_8', 'gating_16']
VOLUMES = ['fixed', 'ratio', 'ewma7', 'ridge']
META_DEVICE = 'cuda'

def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-05):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (0.2, 0.05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-09:
                        best = loss
                        choice = (source, destination, amount, trial)
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(log_level=np.log1p(level), current=frame.current.to_numpy() / level, delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level, log_network=np.log1p(frame.network_current.to_numpy()), horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(), shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return (context.clip(-50, 50).to_numpy(dtype=np.float32), level)

class GatingNetwork(torch.nn.Module):

    def __init__(self, width, experts, hidden, prior):
        super().__init__()
        self.layers = torch.nn.Sequential(torch.nn.Linear(width, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, experts))
        self.register_buffer('prior', torch.as_tensor(prior, dtype=torch.float32))
        torch.nn.init.zeros_(self.layers[-1].weight)
        torch.nn.init.zeros_(self.layers[-1].bias)

    def forward(self, context):
        return torch.softmax(self.prior.log() + self.layers(context), dim=1)

def fit_gating(x, experts, y, level, prior, hidden):
    torch.manual_seed(SEED)
    scaler = StandardScaler().fit(x)
    prior = (prior + 0.02) / (prior.sum() + 0.02 * len(prior))
    model = GatingNetwork(x.shape[1], experts.shape[1], hidden, prior).to(META_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    normalized = experts / level[:, None]
    target = np.asarray(y, dtype=np.float64) / level
    weights = level / level.mean()
    tensors = [torch.as_tensor(value, dtype=torch.float32, device=META_DEVICE) for value in (scaler.transform(x), normalized, target, weights)]
    tx, te, ty, tw = tensors
    for _ in range(15):
        permutation = torch.randperm(len(x), device=META_DEVICE)
        for rows in permutation.split(4096):
            gate = model(tx[rows])
            pred = (gate * te[rows]).sum(dim=1)
            loss = ((pred - ty[rows]).abs() * tw[rows]).mean() + 0.03 * (gate - model.prior).square().sum(dim=1).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return (scaler, model.cpu())

def fit_shape(name, frame, matrix, joint):
    if name == 'session6':
        return None
    if name == 'equal':
        return np.full(len(BASES), 1 / len(BASES))
    if name == 'joint':
        return joint
    if name == 'joint_without_xgb_clean':
        mask = np.array([base != 'xgb_clean' for base in BASES])
        weights = np.zeros(len(BASES))
        weights[mask] = fit_weights(matrix[:, mask], frame.y)
        return weights
    if name.startswith('pair_'):
        index = BASES.index(name.removeprefix('pair_'))
        weights = np.zeros(len(BASES))
        weights[[0, index]] = fit_weights(matrix[:, [0, index]], frame.y)
        return weights
    if name == 'by_horizon':
        weights = []
        for h in HORIZONS:
            mask = frame.horizon.eq(h).to_numpy()
            weights.append(0.5 * joint + 0.5 * fit_weights(matrix[mask], frame.y.to_numpy()[mask]))
        return np.stack(weights)
    baseline = matrix @ joint
    features, level = meta_features(frame, matrix, baseline)
    residual = (frame.y.to_numpy(dtype=np.float64) - baseline) / level
    sample_weight = level / level.mean()
    if name.startswith('gating_'):
        return fit_gating(features, matrix, frame.y, level, joint, int(name.split('_')[-1]))
    if name.startswith('ridge_'):
        prep = StandardScaler().fit(features)
        model = Ridge(alpha=float(name.split('_')[-1]))
        model.fit(prep.transform(features), residual, sample_weight=sample_weight)
        return (prep, model)
    depth = int(name[-1])
    if name.startswith('cat_'):
        model = CatBoostRegressor(depth=depth, iterations=220, loss_function='MAE', learning_rate=0.035, l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=THREADS)
    else:
        model = XGBRegressor(max_depth=depth, n_estimators=220, objective='reg:absoluteerror', tree_method='hist', learning_rate=0.035, reg_lambda=20, min_child_weight=30, random_state=SEED, n_jobs=THREADS)
    model.fit(features, residual, sample_weight=sample_weight)
    return (None, model)

def predict_shape(name, fitted, frame, matrix, joint):
    if name == 'session6':
        return matrix[:, 0]
    if isinstance(fitted, np.ndarray):
        if fitted.ndim == 1:
            return matrix @ fitted
        return np.sum(matrix * fitted[frame.horizon.to_numpy() - 1], axis=1)
    baseline = matrix @ joint
    prep, model = fitted
    features, level = meta_features(frame, matrix, baseline)
    if name.startswith('gating_'):
        weights = gate_weights(prep, model, features)
        return with_volume(frame, np.sum(matrix * weights, axis=1))
    encoded = features if prep is None else prep.transform(features)
    correction = np.clip(model.predict(encoded), -0.5, 0.5) * level
    return with_volume(frame, baseline + correction)

def gate_weights(prep, model, features):
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(prep.transform(features), dtype=torch.float32)).numpy()

def volume_table(frame, with_y=False):
    aggregations = {name: (name, 'sum') for name in BASES}
    aggregations['network_current'] = ('network_current', 'first')
    if with_y:
        aggregations['y'] = ('y', 'sum')
    return frame.groupby(GROUP_KEYS, as_index=False, sort=True).agg(**aggregations)

def volume_features(table):
    anchor = np.maximum(table.session6.to_numpy(), 1)
    features = np.column_stack([np.log1p(anchor), np.log1p(table.network_current), table.horizon, table.cutoff.dt.day_of_week, *[np.log((table[name].to_numpy() + 1) / (anchor + 1)) for name in VOLUME_INPUTS]])
    return features

def fit_volumes(frame):
    table = volume_table(frame, with_y=True)
    if table.session6.sum() <= 0:
        raise ValueError('Нулевой объём Session6')
    ages = (table.cutoff.max() - table.cutoff).dt.total_seconds().to_numpy() / 86400
    w = np.exp2(-ages / 7)
    ratio = table.y.sum() / table.session6.sum()
    recent_ratio = np.dot(table.y, w) / np.dot(table.session6, w)
    x = volume_features(table)
    prep = StandardScaler().fit(x)
    model = Ridge(alpha=100).fit(prep.transform(x), np.log((table.y + 1) / (table.session6 + 1)))
    return dict(fixed=1.0, ratio=float(1 + 0.5 * (ratio - 1)), ewma7=float(1 + 0.5 * (recent_ratio - 1)), ridge=(prep, model))

def volume_multipliers(frame, fitted):
    table = volume_table(frame)
    prep, model = fitted['ridge']
    fitted = dict(fitted, ridge=1 + 0.5 * np.expm1(model.predict(prep.transform(volume_features(table)))))
    return {name: frame[GROUP_KEYS].merge(table[GROUP_KEYS].assign(factor=np.clip(factor, 0.95, 1.05)), on=GROUP_KEYS, how='left', validate='many_to_one').factor.to_numpy() for name, factor in fitted.items()}

def fit_meta(train, query, phase, split):
    train_matrix, query_matrix = (shape_matrix(train), shape_matrix(query))
    joint = fit_weights(train_matrix, train.y)
    volume_models = fit_volumes(train)
    train_factors = volume_multipliers(train, volume_models)
    query_factors = volume_multipliers(query, volume_models)
    rows, fitted, predictions = ([], {}, {})
    for name in SHAPES:
        model = fit_shape(name, train, train_matrix, joint)
        fitted[name] = model
        train_prediction = predict_shape(name, model, train, train_matrix, joint)
        query_prediction = predict_shape(name, model, query, query_matrix, joint)
        local = []
        for volume in VOLUMES:
            label = f'{name}__{volume}'
            train_values = train_prediction * train_factors[volume]
            query_values = query_prediction * query_factors[volume]
            predictions[label] = query_values
            local.extend([dict(phase=phase, variant=label, shape=name, volume=volume, period=period, **metric(y, p)) for period, y, p in [('train_oof', train.y, train_values), (split, query.y, query_values)]])
        display(pd.DataFrame(local))
        rows.extend(local)
    return (pd.DataFrame(rows), dict(joint=joint, shapes=fitted, volumes=volume_models), predictions)

def fit_selection(oof):
    blocks = []
    for fold, (start, stop) in enumerate(OOF_WINDOWS[2:], 3):
        train = oof.loc[oof.cutoff.lt(start)]
        validation = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)]
        rows, _, _ = fit_meta(train, validation, f'val_week_{fold}', 'val')
        blocks.append(rows)
    metrics = pd.concat(blocks, ignore_index=True)
    metrics.to_csv(ARTIFACTS / 'selection_metrics.csv', index=False)
    ranking = metrics.loc[metrics.period.eq('val')].groupby(['variant', 'shape', 'volume'], as_index=False).agg(mean_val_score=('score', 'mean'), worst_val_score=('score', 'max'), weeks=('score', 'count'))
    ranking['order'] = ranking['shape'].map({name: i for i, name in enumerate(SHAPES)})
    ranking = ranking.sort_values(['mean_val_score', 'worst_val_score', 'order', 'variant'])
    if not ranking.weeks.eq(2).all():
        raise ValueError('В подборе должны присутствовать обе недели val')
    ranking.to_csv(ARTIFACTS / 'selection_ranking.csv', index=False)
    display(ranking.drop(columns='order').head(20))
    selected = ranking.iloc[0]
    frozen = dict(variant=selected.variant, shape=selected['shape'], volume=selected.volume, mean_val_score=float(selected.mean_val_score), test_used_for_selection=False, meta_fit_weeks=4, local_test_before_full_refit=True)
    write_json(ARTIFACTS / 'frozen_selection.json', frozen)
    return frozen

def evaluate(oof, test, frozen):
    rows, models, predictions = fit_meta(oof, test, 'frozen_local_test', 'test')
    base_rows = [dict(phase='base', variant=column, shape=column, volume='raw', period=period, **metric(frame.y, frame[column])) for period, frame in [('train_oof', oof), ('test', test)] for column in BASES]
    table = pd.concat([pd.DataFrame(base_rows), rows], ignore_index=True)
    table['selected_on_val'] = table.variant.eq(frozen['variant'])
    table.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
    predictions = pd.concat([test.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
    save_parquet(ARTIFACTS / 'test_predictions.parquet', predictions)
    columns = [*BASES, *[f'{name}__fixed' for name in SHAPES]]
    if frozen['variant'] not in columns:
        columns.append(frozen['variant'])
    slices = []
    for kind, key in [('day', predictions.cutoff), ('horizon', predictions.horizon), ('week', (predictions.cutoff.dt.normalize() - LOCAL_TEST_START).dt.days // 7 + 1)]:
        for value, group in predictions.groupby(key):
            slices.extend((dict(slice=kind, value=str(value), variant=name, **metric(group.y, group[name])) for name in columns))
    pd.DataFrame(slices).to_csv(ARTIFACTS / 'test_slices.csv', index=False)
    weights = []
    matrix = shape_matrix(test)
    features, _ = meta_features(test, matrix, matrix @ models['joint'])
    for name, model in models['shapes'].items():
        if isinstance(model, np.ndarray):
            if model.ndim == 1:
                weights.append(dict(variant=name, horizon='all', **dict(zip(BASES, model))))
            else:
                weights.extend((dict(variant=name, horizon=h, **dict(zip(BASES, row))) for h, row in zip(HORIZONS, model)))
        elif name.startswith('gating_'):
            prep, network = model
            frame = pd.DataFrame(gate_weights(prep, network, features), columns=BASES)
            means = frame.groupby(test.horizon.to_numpy()).mean()
            weights.extend((dict(variant=name, horizon=h, **row.to_dict()) for h, row in means.iterrows()))
            models['shapes'][name] = dict(prep=prep, state=network.state_dict(), width=features.shape[1], hidden=int(name.split('_')[-1]))
    pd.DataFrame(weights).to_csv(ARTIFACTS / 'blend_weights.csv', index=False)
    artifact = ARTIFACTS / 'evaluation_meta.joblib'
    temporary = artifact.with_suffix('.partial')
    joblib.dump(dict(frozen=frozen, bases=BASES, models=models), temporary)
    temporary.replace(artifact)
    comparison = ['joint__fixed', 'joint_without_xgb_clean__fixed', 'pair_xgb_clean__fixed']
    display(table.loc[table.selected_on_val | table.variant.isin([*BASES, *comparison])])
    note = 'Независимая от полного переобучения оценка test получена до переобучения. Локальный период 18–31 октября уже изучался. Выбор сделан на двух более ранних неделях. Новые метамодели обучены только на OOF до test. Session6 на OOF восстановлен по рецепту, а на test взят из архива решения 0.326. Полное переобучение и конкурсный сабмит не выполняются.'
    (ARTIFACTS / 'evaluation_note.txt').write_text(note)
    print(note)


In [ ]:
finish_submission(dict(session6=session_future, rf=rf_future, cat_rich=cat_future, xgb_rich=xgb_future, xgb_clean=clean_future))


In [ ]:
for name in ['session_runtime', 'cat_runtime', 'rf_runtime', 'clean_xgb_runtime', 'xgb_rich_runtime']:
    globals().pop(name, None)
gc.collect()
cp.get_default_memory_pool().free_all_blocks()


## Обучение на всей истории

Отдельный комплект моделей и файл `submission_full_history.csv`. В train_val после обучения выполняется новая оценка test, затем сохраняется сабмит теми же моделями. Full_history запускается только после наличия оценки и первого CSV. Справочные метрики прошлого запуска сохранены отдельно.


In [ ]:
enter_submission_scope('full_history')


In [ ]:
local = pd.read_parquet(exact_local).rename(columns={'session_prediction': 'session6'})
future = pd.read_csv(exact_future)
future = REQUESTS.merge(future, on='id', validate='one_to_one')
future['cutoff'] = FUTURE_CUTOFF
future['horizon'] = ((future.timestamp - FUTURE_CUTOFF) / STEP).astype(int)
future = future.rename(columns={'y_pred': 'session6'})
if SUBMISSION_SCOPE == 'train_val':
    original = read_session6_test()
    comparison = original.merge(local[KEYS + ['session6']], on=KEYS, validate='one_to_one', suffixes=('_old', '_exact'))
    if len(comparison) != len(original) or not np.allclose(comparison.session6_old, comparison.session6_exact, rtol=1e-12, atol=1e-06):
        raise ValueError('Архив Session6 отличается от входа исходного бленда')
    session_future = pd.concat([local[KEYS + ['session6']], future[KEYS + ['session6']]], ignore_index=True)
else:
    session_future = future[KEYS + ['session6']]
write_json(ARTIFACTS / 'session6_exact_protocol.json', dict(local_source=str(exact_local), future_source=str(exact_future), contest_uses_canonical_final_session6=True, full_retraining_from_scratch=False))


In [ ]:
cat_runtime = dict(globals())
exec(compile('\n\nimport joblib\nfrom catboost import CatBoostRegressor\nfrom sklearn.cluster import KMeans\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nCLUSTERS = (1, 8)\nMIN_CLUSTER_ROUTES = 10\nRUN = ARTIFACTS / \'cat_rich\'\nRUN.mkdir(exist_ok=True)\nROLL = {w: pd.DataFrame(VALUES[TARGET]).rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32)\n        for w in (4, 12, 48, 336)}\n\n\nPARAMETERS = {\'selected_cat\': {\'1\': {\'params\': {\'depth\': 6, \'learning_rate\': 0.03694755509074355, \'l2_leaf_reg\': 1.5679933916723008, \'random_strength\': 2.052699079536471, \'bagging_temperature\': 0.8803049874792026}, \'trees\': 900}, \'2\': {\'params\': {\'depth\': 5, \'learning_rate\': 0.1358198535217987, \'l2_leaf_reg\': 29.10635913133069, \'random_strength\': 1.7959754525911098, \'bagging_temperature\': 0.31203728088487304}, \'trees\': 387}, \'3\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.0224831270535195, \'l2_leaf_reg\': 53.99484409787433, \'random_strength\': 1.8033450352296265, \'bagging_temperature\': 1.416145155592091}, \'trees\': 900}, \'4\': {\'params\': {\'depth\': 6, \'learning_rate\': 0.0299058938693663, \'l2_leaf_reg\': 10.677482709481351, \'random_strength\': 1.7772437065861273, \'bagging_temperature\': 0.09290082543999545}, \'trees\': 364}, \'5\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.0224831270535195, \'l2_leaf_reg\': 53.99484409787433, \'random_strength\': 1.8033450352296265, \'bagging_temperature\': 1.416145155592091}, \'trees\': 504}, \'6\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.1411758906434134, \'l2_leaf_reg\': 46.22589001020831, \'random_strength\': 0.6370173320348285, \'bagging_temperature\': 0.36364993441420124}, \'trees\': 125}, \'7\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.1411758906434134, \'l2_leaf_reg\': 46.22589001020831, \'random_strength\': 0.6370173320348285, \'bagging_temperature\': 0.36364993441420124}, \'trees\': 167}, \'8\': {\'params\': {\'depth\': 4, \'learning_rate\': 0.03691990689655602, \'l2_leaf_reg\': 11.207606211860567, \'random_strength\': 1.2958350559263474, \'bagging_temperature\': 0.5824582803960838}, \'trees\': 900}}}\n\nRICH_SOURCE = "import pyarrow as pa\\nimport pyarrow.parquet as pq\\ntarget = values[TARGET]\\n\\ndef past(array, positions):\\n    positions = np.asarray(positions, dtype=np.int64)\\n    valid = (positions >= 0) & (positions < len(array))\\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\\n    result[~valid] = np.nan\\n    return result\\n\\ndef average_available(matrices):\\n    total = np.zeros_like(matrices[0], dtype=np.float32)\\n    count = np.zeros_like(total)\\n    for matrix in matrices:\\n        valid = np.isfinite(matrix)\\n        total += np.where(valid, matrix, 0)\\n        count += valid\\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\\n\\ndef rolling(array, steps):\\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\\nlevel24 = rolling(target, 48)\\npositions = np.arange(len(times))\\nextra_signals = {}\\nfor name in STATUSES[:3]:\\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\\nfor name in STATUSES[3:]:\\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\\nhub_signal = extra_signals[\'status_6_diff_hourly\']\\n\\ndef common_features(origins):\\n    common = {}\\n    for k in (0, 1, 2, 4):\\n        common[f\'target_back_{k}\'] = past(target, origins - k)\\n    for name, array in values.items():\\n        for steps in (4, 12):\\n            common[f\'{name}_mean_{steps}\'] = means[name, steps][origins]\\n        if name in STATUSES:\\n            common[f\'{name}_latest\'] = array[origins]\\n            if name in STATUSES[:3]:\\n                common[f\'{name}_change_1h\'] = array[origins] - past(array, origins - 2)\\n    for name, array in extra_signals.items():\\n        common[name] = array[origins]\\n    for lag in (48, 96):\\n        common[f\'target_1h_lag_{lag}\'] = past(target, origins - lag)\\n    common[\'warehouse_total_load_lag_48\'] = past(extra_signals[\'warehouse_total_load\'], origins - 48)\\n    common[\'status_6_diff_hourly_lag_48\'] = past(hub_signal, origins - 48)\\n    common[\'target_vs_mean_6h\'] = target[origins] - means[TARGET, 12][origins]\\n    recent = means[TARGET, 4][origins]\\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\\n    common[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\\n    return common\\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\\nFEATURE_GROUPS = {\'calendar\': [\'hour\', \'minute\', \'hour_float\', \'hour_sin\', \'hour_cos\', \'dayofweek\', \'is_weekend\', \'weekday_sin\', \'weekday_cos\'], \'local_statuses\': [\'status_1_hourly\', \'status_2_hourly\', \'status_3_hourly\', \'warehouse_total_load\', \'warehouse_processing_rate\'], \'hub_combinations\': [\'status_4_diff_hourly\', \'status_5_diff_hourly\', \'status_6_diff_hourly\', \'hub_buffer\', \'hub_gate_bottleneck\'], \'historical_lags\': [\'target_1h_lag_48\', \'target_1h_lag_96\', \'warehouse_total_load_lag_48\', \'status_6_diff_hourly_lag_48\'], \'target_change\': [\'target_acceleration\'], \'correlation_lag\': CORRELATION_COLUMNS, \'pilot_start_time\': [\'start_sin\', \'start_cos\'], \'pilot_peaks\': [\'peak7_mean_max\', \'peak7_mean_level\', \'peak7_share_-1\', \'peak7_share_0\', \'peak7_share_1\']}\\n\\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\\n    grid = pd.DatetimeIndex(timestamps)\\n    route_ids = np.asarray(route_ids)\\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\\n    if len(np.unique(route_ids)) != len(route_ids):\\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\\n    steps = float(max_lookback_hours) * 2\\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\\n    max_lag = int(steps)\\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\\n    anchors = np.arange(max_lag + 2, stop)\\n    best_corr = np.full(len(route_ids), np.nan)\\n    best_lag = np.full(len(route_ids), np.nan)\\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\\n    for start in range(0, len(route_ids), route_block_size):\\n        finish = min(start + route_block_size, len(route_ids))\\n        x = target_values[anchors, start:finish].astype(np.float64)\\n        for lag in range(2, max_lag + 1):\\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\\n            valid = np.isfinite(x) & np.isfinite(z)\\n            count = valid.sum(axis=0)\\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\\n            divisor = np.maximum(count, 1)\\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\\n            denominator = np.sqrt(vx * vz)\\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\\n            corr = np.clip(corr, -1, 1)\\n            previous = best_corr[start:finish]\\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\\n            best_corr[start:finish][better] = corr[better]\\n            best_lag[start:finish][better] = lag\\n            best_pairs[start:finish][better] = count[better]\\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\\n\\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\\n    if h not in range(1, 9) or chunk_rows < 1:\\n        raise ValueError(\'Некорректный горизонт или размер блока\')\\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\\n    route_index = pd.Index(routes)\\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\\n    for name in CORRELATION_COLUMNS:\\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\\n    for start in range(0, len(frame), chunk_rows):\\n        finish = min(start + chunk_rows, len(frame))\\n        block = frame.iloc[start:finish]\\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\\n        route = route_index.get_indexer(block.route_id)\\n        if (origin < 0).any() or (route < 0).any():\\n            raise ValueError(\'Строка признаков не сопоставлена с историей\')\\n        selected = lag[route]\\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\\n        available &= np.isfinite(signal)\\n        for name, array in quality.items():\\n            frame.iloc[start:finish, column_positions[name]] = array[route]\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal\']] = np.where(available, signal, np.nan)\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal_available\']] = available.astype(np.float32)\\n    return frame\\n\\ndef horizon_dataset(h, origins):\\n    slot = origins + h\\n    dates = times[origins] + h * STEP\\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins))})\\n    for name, matrix in common_features(origins).items():\\n        frame[name] = matrix.ravel()\\n    calendar = {\'hour\': dates.hour, \'minute\': dates.minute, \'hour_float\': dates.hour + dates.minute / 60.0, \'dayofweek\': dates.dayofweek, \'is_weekend\': (dates.dayofweek >= 5).astype(np.int8)}\\n    calendar[\'hour_sin\'] = np.sin(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'hour_cos\'] = np.cos(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\\n    for name, array in calendar.items():\\n        frame[name] = np.repeat(array, len(routes))\\n    frame = add_correlation_features(frame, h, validation_lag_table)\\n    for days in (1, 2, 7):\\n        frame[f\'slot_days_back_{days}\'] = past(target, slot - days * 48).ravel()\\n    for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\\n    profile = np.zeros((len(origins), len(routes)))\\n    counts = np.zeros_like(profile)\\n    for day in range(1, 29):\\n        level = past(level24, origins - day * 48)\\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\\n        profile += np.nan_to_num(adjusted)\\n        counts += np.isfinite(adjusted)\\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\\n    if h == 1:\\n        lower = np.zeros_like(target[origins])\\n        upper = past(target, origins - 2)\\n        for position in (origins - 1, origins):\\n            total = past(target, position)\\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\\n    else:\\n        frame[\'overlap\'] = np.nan\\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\\n    return frame.astype({name: \'float32\' for name in numeric})"\n\ndef segment_routes(boundary, stage):\n    mask = TIMES < boundary\n    y = VALUES[TARGET][mask].astype(np.float64)\n    scale = np.maximum(y.mean(axis=0), 1.0)\n    normalized = y / scale\n    grid = TIMES[mask]\n    descriptors = {}\n    for slot in range(48):\n        select = grid.hour * 2 + grid.minute // 30 == slot\n        descriptors[f\'slot_{slot:02}\'] = normalized[select].mean(axis=0)\n    for day in range(7):\n        descriptors[f\'weekday_{day}\'] = normalized[grid.dayofweek == day].mean(axis=0)\n    descriptors[\'log_cv\'] = np.log1p(normalized.std(axis=0))\n    descriptors[\'zero_share\'] = (y == 0).mean(axis=0)\n    descriptors[\'jump_share\'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)\n    descriptors[\'log_q95\'] = np.log1p(np.quantile(normalized, 0.95, axis=0))\n    frame = pd.DataFrame(descriptors, index=ROUTES)\n    if not np.isfinite(frame.to_numpy()).all():\n        raise ValueError(\'Недостаточно данных для профилей маршрутов\')\n    scaler = StandardScaler().fit(frame)\n    z = scaler.transform(frame)\n    labels, estimators = ({1: np.zeros(len(ROUTES), dtype=int)}, {})\n    for k in CLUSTERS:\n        if k == 1:\n            continue\n        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)\n        groups = model.labels_.copy()\n        counts = np.bincount(groups, minlength=k)\n        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1\n        labels[k], estimators[k] = (groups, model)\n    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels, descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))\n    joblib.dump(artifact, RUN / f\'segments_{stage}.joblib\')\n    frame.insert(0, \'route_id\', ROUTES)\n    frame[\'target_scale\'] = scale\n    for k, groups in labels.items():\n        frame[f\'cluster_{k}\'] = groups\n    frame.to_csv(RUN / f\'route_profiles_{stage}.csv\', index=False)\n    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum())) for k, groups in labels.items() for g in np.unique(groups)]))\n    return artifact\n\ndef features(h, origins, segmentation):\n    scale = segmentation[\'scale\'].astype(np.float32)\n    columns, names = ([], [])\n    shape = (len(origins), len(ROUTES))\n\n    def add(name, array):\n        names.append(name)\n        columns.append(np.broadcast_to(array, shape).reshape(-1))\n\n    def lag(name, array, indices):\n        valid = indices >= 0\n        data = array[np.maximum(indices, 0)] / scale\n        add(name, np.where(valid[:, None], data, 0))\n        add(name + \'_available\', valid[:, None])\n    for delay in (0, 1, 2, 4, 12):\n        lag(f\'target_back_{delay}\', VALUES[TARGET], origins - delay)\n    for status in STATUSES:\n        lag(status, VALUES[status], origins)\n        add(status + \'_delta_1h\', (VALUES[status][origins] - VALUES[status][np.maximum(origins - 2, 0)]) / scale)\n    for window, array in ROLL.items():\n        lag(f\'target_mean_{window}\', array, origins)\n    for days in (1, 7, 14, 28):\n        lag(f\'target_season_{days}\', VALUES[TARGET], origins + h - 48 * days)\n    dates = TIMES[origins] + h * STEP\n    for label, phase in [(\'weekday\', dates.dayofweek.to_numpy() / 7), (\'slot\', (dates.hour * 2 + dates.minute / 30).to_numpy() / 48)]:\n        add(label + \'_sin\', np.sin(2 * np.pi * phase)[:, None])\n        add(label + \'_cos\', np.cos(2 * np.pi * phase)[:, None])\n    add(\'log_route_scale\', np.log1p(scale)[None, :])\n    matrix = np.column_stack(columns).astype(np.float32)\n    if not np.isfinite(matrix).all():\n        raise ValueError(\'Неконечные признаки\')\n    return (matrix, names)\n\noriginal_features_rich = features\noriginal_segment_rich = segment_routes\n\ndef segment_routes(boundary, stage):\n    global FEATURE_NS\n    result = original_segment_rich(boundary, stage)\n    FEATURE_NS = dict(np=np, pd=pd, values=VALUES, TARGET=TARGET, STATUSES=STATUSES, times=TIMES, routes=ROUTES, STEP=STEP, WINDOW_OFFSETS=tuple(range(-4, 5)))\n    exec(RICH_SOURCE, FEATURE_NS)\n    lag_table = FEATURE_NS[\'calculate_dynamic_shoulder\'](VALUES[TARGET], FEATURE_NS[\'hub_signal\'], TIMES, ROUTES, fit_before=boundary)\n    FEATURE_NS[\'validation_lag_table\'] = lag_table\n    FEATURE_NS[\'network_values\'] = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32) for name, array in VALUES.items()}\n    lag_table.to_csv(RUN / f\'rich_status6_lags_{stage}.csv\', index=False)\n    return result\n\ndef features(h, origins, segmentation):\n    base, names = original_features_rich(h, origins, segmentation)\n    frame = FEATURE_NS[\'horizon_dataset\'](h, origins)\n    cols = [c for c in frame if c not in (\'route_id\', \'cutoff\', \'timestamp\', \'y\', \'id\', \'overlap\')]\n    values = frame[cols].to_numpy(dtype=np.float32)\n    scale = np.tile(segmentation[\'scale\'], len(origins)).astype(np.float32)\n    dimensional = [i for i, c in enumerate(cols) if c not in (\'hour\', \'minute\', \'hour_float\', \'dayofweek\', \'is_weekend\') and (not any((v in c for v in (\'ratio\', \'rate\', \'correlation\', \'pair_count\', \'lag_steps\', \'valid\', \'available\', \'_sin\', \'_cos\'))))]\n    for i in dimensional:\n        values[:, i] /= scale\n    available = np.isfinite(values).astype(np.float32)\n    values = np.where(np.isfinite(values), values, 0)\n    shoulder_columns, shoulder_names = ([], [])\n    for label, days in [(\'daily7\', range(1, 8)), (\'weekly4\', (7, 14, 21, 28))]:\n        profiles = []\n        for offset in (-4, -2, 0, 2, 4):\n            total = np.zeros((len(origins), len(ROUTES)), dtype=np.float32)\n            count = np.zeros((len(origins), 1), dtype=np.float32)\n            for day in days:\n                positions = origins + h - day * 48 + offset\n                valid = positions >= 0\n                total += np.where(valid[:, None], VALUES[TARGET][np.maximum(positions, 0)], 0)\n                count += valid[:, None]\n            profile = np.divide(total, count, out=np.zeros_like(total), where=count > 0) / segmentation[\'scale\'][None, :]\n            profiles.append(profile.ravel())\n            shoulder_columns.append(profile.ravel())\n            shoulder_names.append(f\'rich_{label}_shoulder_{offset}\')\n        window = np.column_stack(profiles)\n        shoulder_columns.extend([window.max(axis=1) - window.min(axis=1), np.argmax(window, axis=1) * 60 - 120])\n        shoulder_names.extend([f\'rich_{label}_shoulder_range\', f\'rich_{label}_peak_minutes\'])\n    for signal, array in FEATURE_NS[\'network_values\'].items():\n        for lag in (0, 2):\n            signal_values = np.where(origins >= lag, array[np.maximum(origins - lag, 0)], 0) / max(float(segmentation[\'scale\'].sum()), 1.0)\n            shoulder_columns.append(np.repeat(signal_values, len(ROUTES)))\n            shoulder_names.append(f\'rich_network_{signal}_back_{lag}\')\n        change = (array[origins] - array[np.maximum(origins - 2, 0)]) / max(float(segmentation[\'scale\'].sum()), 1.0)\n        shoulder_columns.append(np.repeat(change, len(ROUTES)))\n        shoulder_names.append(f\'rich_network_{signal}_change_1h\')\n    shoulder_columns.append(np.tile(np.arange(len(ROUTES), dtype=np.float32), len(origins)))\n    shoulder_names.append(\'rich_route_index\')\n    share = VALUES[TARGET][origins] / np.maximum(FEATURE_NS[\'network_values\'][TARGET][origins, None], 1)\n    shoulder_columns.append(share.ravel())\n    shoulder_names.append(\'rich_route_target_network_share\')\n    matrix = np.column_stack([base, values, available, *[np.asarray(c, dtype=np.float32) for c in shoulder_columns]]).astype(np.float32, copy=False)\n    names = names + [\'rich_\' + c for c in cols] + [\'rich_available__\' + c for c in cols] + shoulder_names\n    if not np.isfinite(matrix).all():\n        raise ValueError(\'Неконечные расширенные признаки\')\n    return (matrix, names)\n\ndef model_frame(x, names, algorithm):\n    frame = pd.DataFrame(x, columns=names)\n    route_codes = x[:, names.index(\'rich_route_index\')].astype(np.int64)\n    frame[\'route_code\'] = pd.Categorical(route_codes, categories=np.arange(len(ROUTES))) if algorithm == \'xgb\' else route_codes.astype(str)\n    return frame\n\ndef fit_rich_cat(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        with threadpool_limits(limits=THREADS):\n            segmentation = segment_routes(start, start.strftime(\'%Y%m%d\'))\n        route_groups = segmentation[\'labels\'][8]\n        route_scale = segmentation[\'scale\']\n        origins = np.arange(len(TIMES) - 8)\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            train_origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < start)\n            query_origins = QUERY_ORIGINS\n            x, columns = rich_matrix(h, train_origins, segmentation)\n            query, query_columns = rich_matrix(h, query_origins, segmentation)\n            if columns != query_columns:\n                raise ValueError(\'Расходятся колонки CatBoost rich\')\n            y = VALUES[TARGET][train_origins + h]\n            actual = observed_query(h)\n            predictions = np.empty_like(actual, dtype=np.float64)\n            spec = PARAMETERS[\'selected_cat\'][str(h)]\n            for group in np.unique(route_groups):\n                query_routes = np.flatnonzero(route_groups == group)\n                train_routes = np.arange(len(ROUTES)) if group == -1 else query_routes\n                train_x = model_frame(x[:, train_routes].reshape(-1, len(columns)), columns, \'cat\')\n                train_y = y[:, train_routes].reshape(-1)\n                train_scale = np.tile(route_scale[train_routes], len(train_origins))\n                query_x = model_frame(query[:, query_routes].reshape(-1, len(columns)), columns, \'cat\')\n                model = CatBoostRegressor(**spec[\'params\'], iterations=spec[\'trees\'], loss_function=\'MAE\', task_type=\'GPU\', devices=\'0\', thread_count=THREADS, random_seed=SEED, verbose=False, allow_writing_files=False, allow_const_label=True, bootstrap_type=\'Bayesian\', cat_features=[\'route_code\'])\n                model.fit(train_x, train_y / train_scale, sample_weight=train_scale / route_scale.mean())\n                train_pred = np.maximum(model.predict(train_x) * train_scale, 0)\n                query_pred = np.maximum(model.predict(query_x).reshape(len(query_origins), -1) * route_scale[query_routes], 0)\n                predictions[:, query_routes] = query_pred\n                fit_report(\'cat_rich\', start, h, int(group), split, train_y, train_pred, actual[:, query_routes], query_pred)\n                model.save_model(str(folder / f\'h{h}_group{group}.cbm\'))\n                joblib.dump(segmentation, folder / \'segmentation.joblib\')\n                del model, train_x, query_x, train_pred\n                gc.collect()\n            frame = pd.DataFrame(dict(cutoff=TIMES[query_origins].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query_origins)), horizon=h, y=actual.ravel(), cat_rich=predictions.ravel()))\n            save_parquet(path, frame)\n            del x, query\n            gc.collect()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n\ndef rich_matrix(h, origins, segmentation):\n    if not len(origins):\n        raise ValueError(\'Пустой диапазон CatBoost rich\')\n    routes_count = len(ROUTES)\n    matrix = None\n    for start in range(0, len(origins), 32):\n        batch = origins[start:start + 32]\n        raw, names = features(h, batch, segmentation)\n        if matrix is None:\n            columns = names\n            matrix = np.empty((len(origins), routes_count, len(columns)), dtype=np.float32)\n        if names != columns:\n            raise ValueError(\'Расходятся колонки батчей CatBoost rich\')\n        matrix[start:start + len(batch)] = raw.reshape(len(batch), routes_count, len(columns))\n    return (matrix, columns)\n', 'cat_runtime', 'exec'), cat_runtime)


In [ ]:
cat_future = cat_runtime['fit_rich_cat'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
xgb_rich_runtime = dict(cat_runtime)
exec(compile("import time\nimport xgboost as xgb\nimport cupy as cp\nGPU_BATCH_ROWS = 65536\nGPU_MAX_BIN = 256\nGPU_QUANTILE_BATCHES = 8\nclass GPUTrainingIterator(xgb.DataIter):\n\n    def __init__(self, train, indices, segmentation):\n        self.train = train\n        self.indices = indices\n        self.segmentation = segmentation\n        self.position = 0\n        self.device_batch = None\n        super().__init__(release_data=True)\n\n    def reset(self):\n        self.position = 0\n        self.device_batch = None\n\n    def next(self, input_data):\n        if self.position >= len(self.indices):\n            return False\n        rows = self.indices[self.position:self.position + GPU_BATCH_ROWS]\n        self.position += len(rows)\n        x = gpu_features(self.train['x'][rows], self.train['names'])\n        scales = self.segmentation['scale'][self.train['route_index'][rows]]\n        y = cp.asarray(self.train['y'][rows] / scales, dtype=cp.float32)\n        weight = cp.asarray(scales / self.weight_scale, dtype=cp.float32)\n        self.device_batch = (x, y, weight)\n        input_data(data=x, label=y, weight=weight, feature_names=self.feature_names, feature_types=self.feature_types)\n        return True\n\ndef gpu_features(x, names):\n    values = cp.asarray(x, dtype=cp.float32)\n    route = values[:, names.index('rich_route_index')].reshape(-1, 1)\n    return cp.concatenate([values, route], axis=1)\n\ndef fit_model(train, segmentation, family, k, settings):\n    labels = segmentation['labels'][k][train['route_index']]\n    specification = PARAMETERS[ACTIVE['parameter_set']][str(ACTIVE_H)]\n    feature_names = list(train['names']) + ['route_code']\n    feature_types = ['float'] * len(train['names']) + ['c']\n    params = dict(specification['params'], objective='reg:absoluteerror', eval_metric='mae', device='cuda:0', tree_method='hist', max_bin=GPU_MAX_BIN, max_cat_to_onehot=1, nthread=THREADS, seed=SEED)\n    models = {}\n    for group in np.unique(labels):\n        indices = np.arange(len(labels)) if group == -1 else np.flatnonzero(labels == group)\n        iterator = GPUTrainingIterator(train, indices, segmentation)\n        iterator.weight_scale = float(segmentation['scale'][train['route_index']].mean())\n        iterator.feature_names, iterator.feature_types = (feature_names, feature_types)\n        started = time.monotonic()\n        try:\n            matrix = xgb.QuantileDMatrix(iterator, max_bin=GPU_MAX_BIN, max_quantile_batches=GPU_QUANTILE_BATCHES, enable_categorical=True)\n            model = xgb.train(params, matrix, num_boost_round=specification['trees'])\n        except Exception as error:\n            raise RuntimeError('Ошибка CUDA-обучения. При нехватке VRAM уменьшите GPU_BATCH_ROWS/GPU_MAX_BIN или оставьте MODEL_CLUSTERS=(8,). CPU fallback не используется.') from error\n        if not json.loads(model.save_config())['learner']['generic_param']['device'].startswith('cuda'):\n            raise RuntimeError('XGBoost переключился на CPU')\n        models[int(group)] = dict(model=model, scaler=None)\n        del matrix, iterator\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n        print('GPU h', ACTIVE_H, 'k', k, 'group', int(group), 'rows', len(indices), 'seconds', round(time.monotonic() - started, 1), flush=True)\n    return dict(models=models, k=k, residual=False, feature_names=train['names'], experiment=ACTIVE)\n\ndef predict_model(bundle, data, segmentation):\n    labels = segmentation['labels'][bundle['k']][data['route_index']]\n    prediction = np.empty(len(labels), dtype=np.float64)\n    for group, item in bundle['models'].items():\n        indices = np.flatnonzero(labels == group)\n        item['model'].set_param(dict(device='cuda:0'))\n        for start in range(0, len(indices), GPU_BATCH_ROWS):\n            rows = indices[start:start + GPU_BATCH_ROWS]\n            x = gpu_features(data['x'][rows], bundle['feature_names'])\n            result = item['model'].inplace_predict(x, validate_features=False)\n            if not isinstance(result, cp.ndarray):\n                raise RuntimeError('Предсказание не вернуло CUDA-массив')\n            prediction[rows] = cp.asnumpy(result)\n            del x, result\n    return np.maximum(prediction * segmentation['scale'][data['route_index']], 0)\ndef fit_rich_xgb_full():\n    folder = ARTIFACTS / 'xgb_rich_full'\n    folder.mkdir(exist_ok=True)\n    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]\n    if all((p.is_file() for p in outputs)):\n        return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)\n    with threadpool_limits(limits=THREADS):\n        segmentation = segment_routes(FULL_BOUNDARY, 'full_xgb')\n    manifest = json.loads((XGB_DATA / 'manifest.json').read_text())\n    parameters = manifest['cat_configs'][0]['parameters']\n    for h, path in zip(HORIZONS, outputs):\n        if path.is_file():\n            continue\n        print('Full XGB rich', h, flush=True)\n        origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < FULL_BOUNDARY)\n        matrix, names = rich_matrix(h, origins, segmentation)\n        query, query_names = rich_matrix(h, QUERY_ORIGINS, segmentation)\n        if names != query_names:\n            raise ValueError('Разные признаки XGB rich')\n        train = dict(x=matrix.reshape(-1, len(names)), names=names, route_index=np.tile(np.arange(len(ROUTES)), len(origins)), y=VALUES[TARGET][origins + h].ravel())\n        globals()['PARAMETERS'] = parameters\n        globals()['ACTIVE'] = dict(parameter_set='selected_xgb', residual=False)\n        globals()['ACTIVE_H'] = h\n        bundle = fit_model(train, segmentation, 'xgb', 8, {})\n        pred = predict_model(bundle, train, segmentation)\n        future = predict_model(bundle, dict(x=query.reshape(-1, len(names)), route_index=np.tile(np.arange(len(ROUTES)), len(QUERY_ORIGINS))), segmentation)\n        refit_report('xgb_rich', FULL_BOUNDARY, h, 'all', 'future', train['y'], pred, observed_query(h).ravel(), future)\n        for group, item in bundle['models'].items():\n            item['model'].save_model(str(folder / f'h{h}_group{group}.ubj'))\n        joblib.dump(segmentation, folder / 'segmentation.joblib')\n        save_parquet(path, pd.DataFrame(dict(cutoff=TIMES[QUERY_ORIGINS].repeat(len(ROUTES)), horizon=h, route_id=np.tile(ROUTES, len(QUERY_ORIGINS)), xgb_rich=future)))\n        del matrix, query, train, pred, bundle\n        gc.collect()\n        cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(p) for p in outputs], ignore_index=True)", 'rich_xgb_full', 'exec'), xgb_rich_runtime)
xgb_future = xgb_rich_runtime['fit_rich_xgb_full']()


In [ ]:
rf_runtime = dict(globals())
exec(compile('\n\nimport re\nimport cupy as cp\nfrom cuml.ensemble import RandomForestRegressor\nfrom sklearn.impute import SimpleImputer\n\ntimes, routes, values = TIMES, ROUTES, VALUES\nRUN = ARTIFACTS / \'rf\'\nRUN.mkdir(exist_ok=True)\n\n\nBASE_PARAMS_BY_HORIZON = {1: {\'n_estimators\': 150, \'max_depth\': 16, \'min_samples_leaf\': 44, \'max_features\': 0.9154095569322449, \'max_samples\': 0.7361074625809747}, 2: {\'n_estimators\': 300, \'max_depth\': 8, \'min_samples_leaf\': 37, \'max_features\': 0.6600327095074597, \'max_samples\': 0.5924272277627636}, 3: {\'n_estimators\': 300, \'max_depth\': 8, \'min_samples_leaf\': 37, \'max_features\': 0.6600327095074597, \'max_samples\': 0.5924272277627636}, 4: {\'n_estimators\': 150, \'max_depth\': 12, \'min_samples_leaf\': 37, \'max_features\': 0.6943109266465318, \'max_samples\': 0.5232252063599989}, 5: {\'n_estimators\': 150, \'max_depth\': 16, \'min_samples_leaf\': 44, \'max_features\': 0.9154095569322449, \'max_samples\': 0.7361074625809747}, 6: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 23, \'max_features\': 0.5045012539746527, \'max_samples\': 0.6831809216468459}, 7: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 18, \'max_features\': 0.7252813963310067, \'max_samples\': 0.5852620618436457}, 8: {\'n_estimators\': 300, \'max_depth\': 12, \'min_samples_leaf\': 18, \'max_features\': 0.7252813963310067, \'max_samples\': 0.5852620618436457}}\nN_STREAMS = 4\nPREDICTION_BATCH = 32768\nTRAIN_MINUTES = (570, 600, 630, 660, 690)\nWINDOW_OFFSETS = tuple(range(-4, 5))\nLAG_MAX_LOOKBACK_HOURS = 24\nLAG_MIN_PAIRS = 336\nLAG_MIN_CORRELATION = 0.05\nRESIDUAL = False\n\nbest_params = BASE_PARAMS_BY_HORIZON\nselected = pd.DataFrame(json.loads(\'[{"horizon":2,"trimmed":false,"route_median":true,"experiment":"thin_offsets","mean":0.3472783742,"std":0.0237257355,"max":0.3845543627},{"horizon":6,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":0.3505317232,"std":0.0160416261,"max":0.3713695576},{"horizon":1,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null},{"horizon":3,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":4,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":5,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":7,"trimmed":false,"route_median":false,"experiment":"baseline","mean":null,"std":null,"max":null},{"horizon":8,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null}]\')).set_index(\'horizon\')\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals[\'status_6_diff_hourly\']\nextra_signals[\'target_vs_mean_6h\'] = target - means[TARGET, 12]\nrecent = means[TARGET, 4]\nusual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])\nextra_signals[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n\ndef offset_name(base, offset):\n    if offset == 0:\n        return base\n    return f"{base}_offset_{(\'m\' if offset < 0 else \'p\')}{abs(offset)}"\n\ndef add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):\n    for offset in offsets:\n        common[offset_name(base, offset)] = past(array, anchor + offset)\n\ndef common_features(origins):\n    common = {}\n    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})\n    for lag in local_target_lags:\n        common[f\'target_back_{lag}\'] = past(target, origins - lag)\n    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))\n    for name, array in values.items():\n        for steps in (4, 12):\n            add_window(common, f\'{name}_mean_{steps}\', means[name, steps], origins, causal_offsets)\n        if name in STATUSES:\n            common[f\'{name}_latest\'] = array[origins]\n            for lag in range(1, 5):\n                common[f\'{name}_back_{lag}\'] = past(array, origins - lag)\n            if name in STATUSES[:3]:\n                change = array - past(array, positions - 2)\n                add_window(common, f\'{name}_change_1h\', change, origins, causal_offsets)\n    for name, array in extra_signals.items():\n        add_window(common, name, array, origins, causal_offsets)\n    for lag in (48, 96):\n        add_window(common, f\'target_1h_lag_{lag}\', target, origins - lag)\n    add_window(common, \'warehouse_total_load_lag_48\', extra_signals[\'warehouse_total_load\'], origins - 48)\n    add_window(common, \'status_6_diff_hourly_lag_48\', hub_signal, origins - 48)\n    return common\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\nfor name, series in values.items():\n    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]\n    for suffix, signal in [(\'latest\', network), (\'change_1h\', network - past(network, positions - 2))]:\n        extra_signals[f\'network_{name}_{suffix}\'] = np.broadcast_to(signal, target.shape)\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\n\ndef correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError(\'Некорректный горизонт или размер блока\')\n    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in (\'lagged_status6_signal\', \'lagged_status6_signal_available\')]\n    names = CORRELATION_COLUMNS + extra_columns\n    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)\n    if lag_table is None:\n        names = [prefix + name for name in names]\n        return pd.concat([frame.drop(columns=names, errors=\'ignore\'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError(\'Строка не сопоставлена с историей\')\n        selected = lag[route]\n        for j, name in enumerate(CORRELATION_COLUMNS[:4]):\n            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]\n        for offset in WINDOW_OFFSETS:\n            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset\n            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))\n            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n            available &= np.isfinite(signal)\n            result[start:finish, names.index(offset_name(\'lagged_status6_signal\', offset))] = np.where(available, signal, np.nan)\n            result[start:finish, names.index(offset_name(\'lagged_status6_signal_available\', offset))] = available.astype(np.float32)\n    names = [prefix + name for name in names]\n    frame = frame.drop(columns=names, errors=\'ignore\')\n    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)\n\ndef remove_seasonality(series):\n    typical = average_available([past(series, positions - 48 * d) for d in range(1, 8)])\n    return series - typical\nresidual_target = remove_seasonality(target)\nresidual_signal = remove_seasonality(hub_signal)\n\ndef fit_lags(boundary):\n    tables = []\n    for y, signal, prefix in ((target, hub_signal, \'\'), (residual_target, residual_signal, \'residual_\')):\n        table = calculate_dynamic_shoulder(y, signal, times, routes, fit_before=boundary, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)\n        tables.append(table.rename(columns={c: prefix + c for c in table.columns if c != \'route_id\'}))\n    return tables[0].merge(tables[1], on=\'route_id\', validate=\'one_to_one\')\n\ndef add_correlation_features(frame, h, lag_table):\n    frame = correlation_block(frame, h, lag_table, hub_signal, \'\')\n    if lag_table is None:\n        return correlation_block(frame, h, None, residual_signal, \'residual_\')\n    residual_table = lag_table[[\'route_id\', *[\'residual_\' + c for c in CORRELATION_COLUMNS[:4]]]].copy()\n    residual_table.columns = [\'route_id\', *CORRELATION_COLUMNS[:4]]\n    return correlation_block(frame, h, residual_table, residual_signal, \'residual_\')\n\ndef stable_ratio(numerator, denominator):\n    return numerator / np.maximum(np.abs(denominator), 1.0)\n\ndef available_statistics(window):\n    valid = np.isfinite(window)\n    count = valid.sum(axis=0)\n    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count, out=np.full(window.shape[1:], np.nan), where=count > 0)\n    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count, out=np.full_like(mean, np.nan), where=count > 0)\n    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)\n    return (mean, np.sqrt(variance), spread, count)\n\ndef add_experimental_features(frame, h, origins):\n    additions = {}\n    additions[\'x_time_cutoff_minutes\'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))\n    additions[\'x_profile_profile28\'] = frame.profile28.to_numpy()\n    additions[\'x_level_level24\'] = level24[origins].ravel()\n    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])\n    current = target[origins]\n    additions[\'x_anomaly_target_difference\'] = (current - typical).ravel()\n    additions[\'x_anomaly_target_ratio\'] = stable_ratio(current, typical).ravel()\n    for name in (TARGET, *STATUSES[:3]):\n        short, long = (means[name, 4][origins], means[name, 12][origins])\n        additions[f\'x_trend_{name}_difference\'] = (short - long).ravel()\n        additions[f\'x_trend_{name}_ratio\'] = stable_ratio(short, long).ravel()\n        for steps in (4, 12):\n            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])\n            mean, std, spread, count = available_statistics(window)\n            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]\n            additions[f\'x_volatility_{name}_std_{steps}\'] = std.ravel()\n            additions[f\'x_volatility_{name}_range_{steps}\'] = spread.ravel()\n            additions[f\'x_volatility_{name}_change_{steps}\'] = change.ravel()\n    for label, days in [(\'daily7\', range(1, 8)), (\'weekly4\', (7, 14, 21, 28))]:\n        window = np.stack([past(target, origins + h - day * 48) for day in days])\n        mean, std, spread, count = available_statistics(window)\n        additions[f\'x_dispersion_{label}_std\'] = std.ravel()\n        additions[f\'x_dispersion_{label}_relative_std\'] = stable_ratio(std, mean).ravel()\n        additions[f\'x_dispersion_{label}_count\'] = count.ravel()\n    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    common = common_features(origins)\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})\n    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, None)\n    seasonal_columns = {}\n    for offset in WINDOW_OFFSETS:\n        for day in (1, 2, 7):\n            seasonal_columns[offset_name(f\'slot_days_back_{day}\', offset)] = past(target, slot - day * 48 + offset).ravel()\n        for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\n            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()\n    for base in (\'slot_daily7\', \'slot_weekly4\'):\n        window = np.column_stack([seasonal_columns[offset_name(base, k)] for k in WINDOW_OFFSETS])\n        valid = np.isfinite(window)\n        maximum = np.where(valid, window, -np.inf).max(axis=1)\n        minimum = np.where(valid, window, np.inf).min(axis=1)\n        peak = valid & (window == maximum[:, None])\n        count = peak.sum(axis=1)\n        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window), np.nan), where=count > 0)\n        seasonal_columns[base + \'_peak_minutes\'] = shift * 30\n        seasonal_columns[base + \'_range\'] = np.where(count > 0, maximum - minimum, np.nan)\n        seasonal_columns[base + \'_right_minus_left\'] = window[:, 5:].mean(axis=1) - window[:, :4].mean(axis=1)\n    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame[\'overlap\'] = np.nan\n    raw_target_sources = []\n    for lag in range(9):\n        raw_target_sources.append((f\'target_back_{lag}\', -lag, lag in (0, 1, 2, 4)))\n    for lag in (48, 96):\n        for offset in WINDOW_OFFSETS:\n            raw_target_sources.append((offset_name(f\'target_1h_lag_{lag}\', offset), -lag + offset, offset == 0))\n    for day in (1, 2, 7):\n        for offset in WINDOW_OFFSETS:\n            raw_target_sources.append((offset_name(f\'slot_days_back_{day}\', offset), h - day * 48 + offset, offset == 0))\n    seen, redundant = (set(), [])\n    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):\n        if source in seen:\n            redundant.append(name)\n        else:\n            seen.add(source)\n    frame = frame.drop(columns=redundant)\n    frame = add_experimental_features(frame, h, origins)\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\n    return frame.astype({name: \'float32\' for name in numeric}).copy()\n\nTARGET_SCALE = 100000\n\ndef model_columns(data):\n    return data.columns.drop([\'timestamp\', \'cutoff\', \'y\', \'profile28\', \'overlap\']).tolist()\n\nEXPERIMENTS = {\'baseline\': dict(), \'drop_constant\': dict(drop_constant=True), \'compact_availability\': dict(compact_availability=True), \'calendar_dayofweek\': dict(calendar=True), \'thin_offsets\': dict(thin=True), \'drop_network\': dict(network=\'drop\'), \'thin_network\': dict(network=\'thin\'), \'add_time\': dict(add=[\'time\']), \'add_profile\': dict(add=[\'profile\']), \'add_level24\': dict(add=[\'level\']), \'add_anomaly\': dict(add=[\'anomaly\']), \'add_trends\': dict(add=[\'trend\']), \'add_volatility\': dict(add=[\'volatility\']), \'add_seasonal_dispersion\': dict(add=[\'dispersion\']), \'all_additions\': dict(add=[\'time\', \'profile\', \'level\', \'anomaly\', \'trend\', \'volatility\', \'dispersion\']), \'compact\': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network=\'thin\'), \'compact_with_additions\': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network=\'thin\', add=[\'time\', \'profile\', \'level\', \'anomaly\', \'trend\', \'volatility\', \'dispersion\'])}\n\ndef model_target(frame):\n    baseline = frame.profile28.to_numpy() if RESIDUAL else 0\n    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)\n\ndef transformed_frame(frame, columns):\n    result = frame[columns].drop(columns=\'route_id\').astype(np.float32)\n    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1\n    bits = {f\'route_bit_{i}\': (codes >> i & 1).astype(np.float32) for i in range(max(1, int(len(routes)).bit_length()))}\n    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()\n\ndef fit_forest(frame, matrix, params):\n    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type=\'numpy\', **params)\n    model.fit(matrix, model_target(frame))\n    cp.cuda.get_current_stream().synchronize()\n    return model\n\ndef predict_matrix(model, frame, matrix):\n    result = np.empty(len(frame), dtype=np.float64)\n    for start in range(0, len(frame), PREDICTION_BATCH):\n        stop = min(start + PREDICTION_BATCH, len(frame))\n        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)\n        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0\n        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)\n    if not np.isfinite(result).all():\n        raise ValueError(\'Некорректный прогноз\')\n    return result\n\ndef experiment_columns(names, spec):\n    result = []\n    for name in names:\n        if name.startswith(\'x_\'):\n            if any((name.startswith(f\'x_{group}_\') for group in spec.get(\'add\', []))):\n                result.append(name)\n            continue\n        if spec.get(\'drop_constant\') and name == \'selected_status6_pair_count\':\n            continue\n        if spec.get(\'compact_availability\') and \'signal_available\' in name:\n            continue\n        if spec.get(\'calendar\') and name in (\'weekday_sin\', \'weekday_cos\', \'is_weekend\'):\n            continue\n        if spec.get(\'network\') == \'drop\' and name.startswith(\'network_\'):\n            continue\n        if spec.get(\'network\') == \'thin\' and name.startswith(\'network_\') and (\'_offset_\' in name):\n            continue\n        if spec.get(\'thin\') and (not name.startswith(\'slot_\')) and re.search(\'_offset_m[134]$\', name):\n            continue\n        result.append(name)\n    if spec.get(\'compact_availability\'):\n        for name in (\'status6_lag_valid\', \'residual_status6_lag_valid\'):\n            if name in names and name not in result:\n                result.append(name)\n    return result\n\ndef rf_data(h, origins, lag_table, spec):\n    batches, parts = ([], [])\n    for start in range(0, len(origins), 24):\n        batch = origins[start:start + 24]\n        frame = horizon_dataset(h, batch)\n        frame[\'y\'] = past(target, batch + h).ravel()\n        frame = frame.loc[np.isfinite(frame.profile28)].reset_index(drop=True)\n        frame = add_correlation_features(frame, h, lag_table)\n        matrix = transformed_frame(frame, model_columns(frame))\n        columns = experiment_columns(list(matrix.columns), EXPERIMENTS[spec.experiment])\n        batches.append(matrix[columns].replace([np.inf, -np.inf], np.nan))\n        parts.append(frame[[\'cutoff\', \'timestamp\', \'route_id\', \'y\', \'profile28\']].copy())\n    return (pd.concat(parts, ignore_index=True), pd.concat(batches, ignore_index=True))\n\ndef fit_rf(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        lag_table = fit_lags(start)\n        minute = times.hour * 60 + times.minute\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            spec = selected.loc[h]\n            if bool(spec.trimmed):\n                raise ValueError(\'Для этого рецепта RF ожидается полная обучающая история\')\n            train_origins = np.flatnonzero(np.isin(minute, TRAIN_MINUTES) & (times + h * STEP < start))\n            query_origins = QUERY_ORIGINS\n            train, train_x = rf_data(h, train_origins, lag_table, spec)\n            query, query_x = rf_data(h, query_origins, lag_table, spec)\n            if bool(spec.route_median):\n                median = train_x.groupby(train.route_id).median()\n                train_x = train_x.fillna(median.reindex(train.route_id).set_axis(train_x.index))\n                query_x = query_x.fillna(median.reindex(query.route_id).set_axis(query_x.index))\n            imputer = SimpleImputer(strategy=\'median\', keep_empty_features=True)\n            train_matrix = imputer.fit_transform(train_x).astype(np.float32, copy=False)\n            query_matrix = imputer.transform(query_x).astype(np.float32, copy=False)\n            del train_x, query_x\n            model = fit_forest(train, train_matrix, best_params[h])\n            train_pred = predict_matrix(model, train, train_matrix)\n            query_pred = predict_matrix(model, query, query_matrix)\n            fit_report(\'rf\', start, h, \'all\', split, train.y, train_pred, query.y, query_pred)\n            save_parquet(path, query[[\'cutoff\', \'route_id\', \'y\']].assign(horizon=h, rf=query_pred))\n            joblib.dump(dict(model=model, imputer=imputer, median=median if bool(spec.route_median) else None, lag_table=lag_table, spec=spec), folder / f\'h{h}_model.joblib\')\n            del model, train_matrix, query_matrix, train, query, train_pred, query_pred\n            gc.collect()\n            cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n', 'rf_runtime', 'exec'), rf_runtime)


In [ ]:
rf_future = rf_runtime['fit_rf'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
clean_xgb_runtime = dict(globals())
exec(compile('\n\nimport xgboost as xgb\nimport cupy as cp\n\nRUN = ARTIFACTS / \'xgb_clean\'\nRUN.mkdir(exist_ok=True)\nCLEAN_XGB_CONTEXT = None\n\n\nTARGET_SCALE = 100000.0\nOBJECTIVE = \'reg:absoluteerror\'\nCLEANING = {\'window_days\': 28, \'min_days\': 14, \'z_threshold\': 4.0, \'log_scale_floor\': 0.25, \'mad_normalization\': 1.4826}\nLAG_MAX_LOOKBACK_HOURS = 24\nLAG_MIN_PAIRS = 336\nLAG_MIN_CORRELATION = 0.05\nBEST_PARAMS = {\'1\': {\'n_estimators\': 2530, \'max_depth\': 6, \'learning_rate\': 0.036760715868196514, \'min_child_weight\': 1.0809355847970479, \'subsample\': 0.9003563537429966, \'colsample_bytree\': 0.6639976846136139, \'reg_alpha\': 9.305271303855188e-05, \'reg_lambda\': 7.596418498005285}, \'2\': {\'n_estimators\': 1419, \'max_depth\': 4, \'learning_rate\': 0.10190438495585248, \'min_child_weight\': 1.8389812995413264, \'subsample\': 0.6817515181500764, \'colsample_bytree\': 0.6619351860505892, \'reg_alpha\': 0.0022965441587741537, \'reg_lambda\': 14.08596091970978}, \'3\': {\'n_estimators\': 1930, \'max_depth\': 3, \'learning_rate\': 0.07005359250881395, \'min_child_weight\': 2.2064897603579636, \'subsample\': 0.8779096005107611, \'colsample_bytree\': 0.7846752865583575, \'reg_alpha\': 0.018954349619526665, \'reg_lambda\': 1.043977626067855}, \'4\': {\'n_estimators\': 1827, \'max_depth\': 4, \'learning_rate\': 0.02925325002470326, \'min_child_weight\': 1.5841239345284504, \'subsample\': 0.8126045473410459, \'colsample_bytree\': 0.7066552862291448, \'reg_alpha\': 2.3192129195348187e-05, \'reg_lambda\': 0.22202924731915144}, \'5\': {\'n_estimators\': 1387, \'max_depth\': 3, \'learning_rate\': 0.03164978915749773, \'min_child_weight\': 1.9902815543275585, \'subsample\': 0.6844098448217815, \'colsample_bytree\': 0.7432566543439275, \'reg_alpha\': 0.010252186188681373, \'reg_lambda\': 0.5427730704771981}, \'6\': {\'n_estimators\': 1566, \'max_depth\': 3, \'learning_rate\': 0.023401679328514332, \'min_child_weight\': 3.1806793320017555, \'subsample\': 0.87035767368126, \'colsample_bytree\': 0.7980364720685608, \'reg_alpha\': 0.02123901077609503, \'reg_lambda\': 0.3328783942599951}, \'7\': {\'n_estimators\': 1136, \'max_depth\': 5, \'learning_rate\': 0.0330759240922086, \'min_child_weight\': 3.8692684642092345, \'subsample\': 0.6794525496919193, \'colsample_bytree\': 0.6799389555259473, \'reg_alpha\': 2.7314479492664567, \'reg_lambda\': 0.3796668381307971}, \'8\': {\'n_estimators\': 534, \'max_depth\': 4, \'learning_rate\': 0.09417045025581995, \'min_child_weight\': 13.418857993305474, \'subsample\': 0.689895840619612, \'colsample_bytree\': 0.6918995986906583, \'reg_alpha\': 0.3746751674810787, \'reg_lambda\': 1.8540896283895405}}\n\ndef clean_raw_array(array, timestamps, settings):\n    array = np.asarray(array, dtype=np.float32)\n    if not np.isfinite(array).all() or (array < 0).any():\n        raise ValueError(\'Для log1p нужны конечные неотрицательные исходные значения\')\n    cleaned = array.copy()\n    flagged = np.zeros(array.shape, dtype=bool)\n    slots = timestamps.hour * 2 + timestamps.minute // 30\n    for slot in range(48):\n        positions = np.flatnonzero(slots == slot)\n        logged = np.log1p(array[positions].astype(np.float64))\n        window_days = settings[\'window_days\']\n        min_days = settings[\'min_days\']\n        if not 1 <= min_days <= window_days:\n            raise ValueError(\'Нужно 1 <= min_days <= window_days\')\n        if len(positions) <= min_days:\n            continue\n        padded = np.pad(logged, ((window_days, 0), (0, 0)), constant_values=np.nan)\n        windows = np.lib.stride_tricks.sliding_window_view(padded, window_days, axis=0)[:-1]\n        eligible = np.arange(len(positions)) >= min_days\n        past = windows[eligible]\n        median = np.nanmedian(past, axis=-1)\n        mad = np.nanmedian(np.abs(past - median[..., None]), axis=-1)\n        scale = np.maximum(settings[\'mad_normalization\'] * mad, settings[\'log_scale_floor\'])\n        mask = np.abs(logged[eligible] - median) > settings[\'z_threshold\'] * scale\n        destination = positions[eligible]\n        cleaned[destination] = np.where(mask, np.expm1(median), array[destination]).astype(np.float32)\n        flagged[destination] = mask\n    return (cleaned, flagged)\n\nFEATURE_CODE = "import pyarrow as pa\\nimport pyarrow.parquet as pq\\ntarget = values[TARGET]\\n\\ndef past(array, positions):\\n    positions = np.asarray(positions, dtype=np.int64)\\n    valid = (positions >= 0) & (positions < len(array))\\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\\n    result[~valid] = np.nan\\n    return result\\n\\ndef average_available(matrices):\\n    total = np.zeros_like(matrices[0], dtype=np.float32)\\n    count = np.zeros_like(total)\\n    for matrix in matrices:\\n        valid = np.isfinite(matrix)\\n        total += np.where(valid, matrix, 0)\\n        count += valid\\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\\n\\ndef rolling(array, steps):\\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\\nlevel24 = rolling(target, 48)\\npositions = np.arange(len(times))\\nextra_signals = {}\\nfor name in STATUSES[:3]:\\n    extra_signals[name + \'_hourly\'] = values[name] + past(values[name], positions - 1)\\nextra_signals[\'warehouse_total_load\'] = sum((extra_signals[name + \'_hourly\'] for name in STATUSES[:3]))\\nextra_signals[\'warehouse_processing_rate\'] = extra_signals[\'status_3_hourly\'] / (extra_signals[\'status_1_hourly\'] + np.float32(1e-05))\\nfor name in STATUSES[3:]:\\n    extra_signals[name + \'_diff_hourly\'] = values[name] - past(values[name], positions - 2)\\nextra_signals[\'hub_buffer\'] = values[\'status_5\'] - values[\'status_4\']\\nextra_signals[\'hub_gate_bottleneck\'] = extra_signals[\'status_4_diff_hourly\'] - extra_signals[\'status_6_diff_hourly\']\\nextra_signals[\'target_acceleration\'] = past(target, positions - 1) - past(target, positions - 2)\\nhub_signal = extra_signals[\'status_6_diff_hourly\']\\n\\ndef common_features(origins):\\n    common = {}\\n    for k in (0, 1, 2, 4):\\n        common[f\'target_back_{k}\'] = past(target, origins - k)\\n    for name, array in values.items():\\n        for steps in (4, 12):\\n            common[f\'{name}_mean_{steps}\'] = means[name, steps][origins]\\n        if name in STATUSES:\\n            common[f\'{name}_latest\'] = array[origins]\\n            if name in STATUSES[:3]:\\n                common[f\'{name}_change_1h\'] = array[origins] - past(array, origins - 2)\\n    for name, array in extra_signals.items():\\n        common[name] = array[origins]\\n    for lag in (48, 96):\\n        common[f\'target_1h_lag_{lag}\'] = past(target, origins - lag)\\n    common[\'warehouse_total_load_lag_48\'] = past(extra_signals[\'warehouse_total_load\'], origins - 48)\\n    common[\'status_6_diff_hourly_lag_48\'] = past(hub_signal, origins - 48)\\n    common[\'target_vs_mean_6h\'] = target[origins] - means[TARGET, 12][origins]\\n    recent = means[TARGET, 4][origins]\\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\\n    common[\'morning_level_ratio\'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\\n    return common\\nCORRELATION_COLUMNS = [\'selected_status6_lag_steps\', \'selected_status6_correlation\', \'selected_status6_pair_count\', \'status6_lag_valid\', \'lagged_status6_signal\', \'lagged_status6_signal_available\']\\nFEATURE_GROUPS = {\'calendar\': [\'hour\', \'minute\', \'hour_float\', \'hour_sin\', \'hour_cos\', \'dayofweek\', \'is_weekend\', \'weekday_sin\', \'weekday_cos\'], \'local_statuses\': [\'status_1_hourly\', \'status_2_hourly\', \'status_3_hourly\', \'warehouse_total_load\', \'warehouse_processing_rate\'], \'hub_combinations\': [\'status_4_diff_hourly\', \'status_5_diff_hourly\', \'status_6_diff_hourly\', \'hub_buffer\', \'hub_gate_bottleneck\'], \'historical_lags\': [\'target_1h_lag_48\', \'target_1h_lag_96\', \'warehouse_total_load_lag_48\', \'status_6_diff_hourly_lag_48\'], \'target_change\': [\'target_acceleration\'], \'correlation_lag\': CORRELATION_COLUMNS, \'pilot_start_time\': [\'start_sin\', \'start_cos\'], \'pilot_peaks\': [\'peak7_mean_max\', \'peak7_mean_level\', \'peak7_share_-1\', \'peak7_share_0\', \'peak7_share_1\']}\\n\\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\\n    grid = pd.DatetimeIndex(timestamps)\\n    route_ids = np.asarray(route_ids)\\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\\n        raise ValueError(\'Ожидаются два массива [время, маршрут] одной формы\')\\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\\n        raise ValueError(\'Для выбора лага нужна упорядоченная получасовая сетка без дубликатов\')\\n    if len(np.unique(route_ids)) != len(route_ids):\\n        raise ValueError(\'Идентификаторы маршрутов повторяются\')\\n    steps = float(max_lookback_hours) * 2\\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\\n        raise ValueError(\'Граница поиска должна задавать целое число получасов, не меньше двух\')\\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\\n        raise ValueError(\'min_pairs должен быть целым числом >=3\')\\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\\n        raise ValueError(\'Некорректные параметры выбора корреляции\')\\n    max_lag = int(steps)\\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side=\'left\'))\\n    anchors = np.arange(max_lag + 2, stop)\\n    best_corr = np.full(len(route_ids), np.nan)\\n    best_lag = np.full(len(route_ids), np.nan)\\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\\n    for start in range(0, len(route_ids), route_block_size):\\n        finish = min(start + route_block_size, len(route_ids))\\n        x = target_values[anchors, start:finish].astype(np.float64)\\n        for lag in range(2, max_lag + 1):\\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\\n            valid = np.isfinite(x) & np.isfinite(z)\\n            count = valid.sum(axis=0)\\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\\n            divisor = np.maximum(count, 1)\\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\\n            denominator = np.sqrt(vx * vz)\\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\\n            corr = np.clip(corr, -1, 1)\\n            previous = best_corr[start:finish]\\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\\n            best_corr[start:finish][better] = corr[better]\\n            best_lag[start:finish][better] = lag\\n            best_pairs[start:finish][better] = count[better]\\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\\n    return pd.DataFrame({\'route_id\': route_ids, \'selected_status6_lag_steps\': np.where(accepted, best_lag, np.nan), \'selected_status6_correlation\': best_corr, \'selected_status6_pair_count\': best_pairs, \'status6_lag_valid\': accepted.astype(np.int8)})\\n\\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\\n    if h not in range(1, 9) or chunk_rows < 1:\\n        raise ValueError(\'Некорректный горизонт или размер блока\')\\n    by_route = lag_table.set_index(\'route_id\').reindex(routes)\\n    route_index = pd.Index(routes)\\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\\n    for name in CORRELATION_COLUMNS:\\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\\n    for start in range(0, len(frame), chunk_rows):\\n        finish = min(start + chunk_rows, len(frame))\\n        block = frame.iloc[start:finish]\\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\\n        route = route_index.get_indexer(block.route_id)\\n        if (origin < 0).any() or (route < 0).any():\\n            raise ValueError(\'Строка признаков не сопоставлена с историей\')\\n        selected = lag[route]\\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\\n        available &= np.isfinite(signal)\\n        for name, array in quality.items():\\n            frame.iloc[start:finish, column_positions[name]] = array[route]\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal\']] = np.where(available, signal, np.nan)\\n        frame.iloc[start:finish, column_positions[\'lagged_status6_signal_available\']] = available.astype(np.float32)\\n    return frame\\n\\ndef horizon_dataset(h, origins):\\n    slot = origins + h\\n    dates = times[origins] + h * STEP\\n    frame = pd.DataFrame({\'cutoff\': times[origins].repeat(len(routes)), \'timestamp\': dates.repeat(len(routes)), \'route_id\': np.tile(routes, len(origins))})\\n    for name, matrix in common_features(origins).items():\\n        frame[name] = matrix.ravel()\\n    calendar = {\'hour\': dates.hour, \'minute\': dates.minute, \'hour_float\': dates.hour + dates.minute / 60.0, \'dayofweek\': dates.dayofweek, \'is_weekend\': (dates.dayofweek >= 5).astype(np.int8)}\\n    calendar[\'hour_sin\'] = np.sin(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'hour_cos\'] = np.cos(2 * np.pi * calendar[\'hour_float\'] / 24)\\n    calendar[\'weekday_sin\'] = np.sin(2 * np.pi * dates.dayofweek / 7)\\n    calendar[\'weekday_cos\'] = np.cos(2 * np.pi * dates.dayofweek / 7)\\n    for name, array in calendar.items():\\n        frame[name] = np.repeat(array, len(routes))\\n    frame = add_correlation_features(frame, h, validation_lag_table)\\n    for days in (1, 2, 7):\\n        frame[f\'slot_days_back_{days}\'] = past(target, slot - days * 48).ravel()\\n    for name, days in [(\'slot_daily7\', range(1, 8)), (\'slot_weekly4\', (7, 14, 21, 28))]:\\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\\n    profile = np.zeros((len(origins), len(routes)))\\n    counts = np.zeros_like(profile)\\n    for day in range(1, 29):\\n        level = past(level24, origins - day * 48)\\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\\n        profile += np.nan_to_num(adjusted)\\n        counts += np.isfinite(adjusted)\\n    frame[\'profile28\'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\\n    if h == 1:\\n        lower = np.zeros_like(target[origins])\\n        upper = past(target, origins - 2)\\n        for position in (origins - 1, origins):\\n            total = past(target, position)\\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\\n        seasonal = frame[[\'slot_daily7\', \'slot_weekly4\']].mean(axis=1).to_numpy()\\n        frame[\'overlap\'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\\n    else:\\n        frame[\'overlap\'] = np.nan\\n    numeric = frame.columns.difference([\'cutoff\', \'timestamp\', \'route_id\'])\\n    return frame.astype({name: \'float32\' for name in numeric})\\nbase_horizon_dataset = horizon_dataset\\nnetwork_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}\\n\\ndef ratio_features(current, usual):\\n    denominator = np.maximum(usual, np.float32(1.0))\\n    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))\\n    return (ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32))\\n\\ndef horizon_dataset(h, origins):\\n    frame = base_horizon_dataset(h, origins)\\n    added = {}\\n    slot = origins + h\\n    profile = {}\\n    for offset in (-2, -1, 0, 1, 2):\\n        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])\\n        if offset != 0:\\n            added[f\'slot_daily7_neighbor_{offset:+d}\'] = profile[offset].ravel()\\n    added[\'slot_daily7_change_2h\'] = (profile[2] - profile[-2]).ravel()\\n    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])\\n    finite = np.isfinite(stack)\\n    masked = np.where(finite, stack, -np.inf)\\n    peak = masked.max(axis=0)\\n    available = finite.any(axis=0)\\n    added[\'slot_daily7_peak_value\'] = np.where(available, peak, np.nan).ravel()\\n    added[\'slot_daily7_peak_offset_minutes\'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()\\n    for name in STATUSES[:3]:\\n        for suffix, array in [(\'latest\', values[name]), (\'mean_4\', means[name, 4])]:\\n            current = array[origins]\\n            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\\n            ratio, small = ratio_features(current, usual)\\n            added[f\'{name}_{suffix}_vs_usual7\'] = (current - usual).ravel()\\n            added[f\'{name}_{suffix}_ratio_usual7\'] = ratio.ravel()\\n            added[f\'{name}_{suffix}_usual7_small\'] = small.ravel()\\n    for day in (14, 21, 28):\\n        added[f\'slot_days_back_{day}\'] = past(target, slot - day * 48).ravel()\\n    added[\'slot_week1_minus_week2\'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()\\n    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])\\n    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])\\n    added[\'slot_recent3_minus_previous4\'] = (recent - previous).ravel()\\n    for name, array in network_values.items():\\n        current = array[origins]\\n        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\\n        ratio, small = ratio_features(current, usual)\\n        features = {\'latest\': current, \'change_1h\': current - past(array, origins - 2), \'vs_usual7\': current - usual, \'ratio_usual7\': ratio, \'usual7_small\': small}\\n        for suffix, feature in features.items():\\n            added[f\'network_{name}_{suffix}\'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()\\n        local = values[name][origins]\\n        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)\\n        past_total = past(array, origins - 2)\\n        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)\\n        added[f\'network_{name}_route_share\'] = share.ravel()\\n        added[f\'network_{name}_route_share_change_1h\'] = (share - old_share).ravel()\\n    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)"\n\nclass CleanXgbBatches(xgb.DataIter):\n\n    def __init__(self, matrix, labels, columns):\n        self.matrix = matrix\n        self.labels = labels\n        self.columns = columns\n        self.types = [\'c\' if column == \'route_id\' else \'q\' for column in columns]\n        self.position = 0\n        super().__init__(release_data=True)\n\n    def reset(self):\n        self.position = 0\n\n    def next(self, input_data):\n        if self.position >= len(self.labels):\n            return False\n        start = self.position\n        stop = min(start + 65536, len(self.labels))\n        self.position = stop\n        input_data(data=cp.asarray(self.matrix[start:stop]), label=cp.asarray(self.labels[start:stop]), feature_names=self.columns, feature_types=self.types)\n        return True\n\ndef clean_xgb_matrix(h, origins, features):\n    route_count = len(ROUTES)\n    route_codes = np.arange(route_count, dtype=np.float32)\n    matrix = None\n    for start in range(0, len(origins), 24):\n        block = origins[start:start + 24]\n        frame = features(h, block)\n        columns = frame.columns.drop([\'cutoff\', \'timestamp\', \'overlap\']).tolist()\n        frame[\'route_id\'] = np.tile(route_codes, len(block))\n        if matrix is None:\n            names = columns\n            matrix = np.empty((len(origins) * route_count, len(names)), dtype=np.float32)\n        if columns != names:\n            raise ValueError(\'Расходятся колонки обычного XGBoost\')\n        matrix[start * route_count:(start + len(block)) * route_count] = frame[columns].to_numpy(dtype=np.float32)\n    if matrix is None:\n        raise ValueError(\'Пустая матрица обычного XGBoost\')\n    return (matrix, names)\n\ndef predict_clean_xgb(model, matrix):\n    result = np.empty(len(matrix), dtype=np.float64)\n    for start in range(0, len(matrix), 65536):\n        stop = min(start + 65536, len(matrix))\n        result[start:stop] = cp.asnumpy(model.inplace_predict(cp.asarray(matrix[start:stop]))) * TARGET_SCALE\n    return np.maximum(result, 0)\n\ndef clean_xgb_features():\n    global CLEAN_XGB_CONTEXT\n    if CLEAN_XGB_CONTEXT is None:\n        values = {}\n        rows = []\n        for name in [TARGET, *STATUSES]:\n            values[name], flagged = clean_raw_array(VALUES[name], TIMES, CLEANING)\n            rows.append(dict(column=name, replaced=int(flagged.sum()), rows=flagged.size))\n        pd.DataFrame(rows).to_csv(RUN / \'cleaning_counts.csv\', index=False)\n        CLEAN_XGB_CONTEXT = dict(np=np, pd=pd, TARGET=TARGET, STATUSES=STATUSES, STEP=STEP, values=values, times=TIMES, routes=ROUTES)\n        exec(FEATURE_CODE, CLEAN_XGB_CONTEXT)\n    return CLEAN_XGB_CONTEXT\n\ndef fit_clean_xgb(start, stop, split):\n    if start >= LOCAL_TEST_START and (not (ARTIFACTS / \'frozen_selection.json\').is_file()):\n        raise ValueError(\'Сначала зафиксируйте выбор по val\')\n    folder = RUN / start.strftime(\'%Y%m%d\')\n    folder.mkdir(parents=True, exist_ok=True)\n    outputs = [folder / f\'h{h}.parquet\' for h in HORIZONS]\n    if not all((path.is_file() for path in outputs)):\n        features = clean_xgb_features()\n        target = features[\'target\']\n        lag_table = features[\'calculate_dynamic_shoulder\'](target, features[\'hub_signal\'], TIMES, ROUTES, fit_before=start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)\n        features[\'validation_lag_table\'] = lag_table\n        lag_table.to_csv(folder / \'status6_lags.csv\', index=False)\n        for h, path in zip(HORIZONS, outputs):\n            if path.is_file():\n                continue\n            origins = np.arange(len(TIMES) - h)\n            train = origins[TIMES[origins + h] < start]\n            query = QUERY_ORIGINS\n            print(\'XGBoost clean\', split, start.date(), f\'h{h}\', flush=True)\n            matrix, columns = clean_xgb_matrix(h, train, features[\'horizon_dataset\'])\n            labels = (target[train + h].ravel() / TARGET_SCALE).astype(np.float32)\n            batches = CleanXgbBatches(matrix, labels, columns)\n            quantile = xgb.QuantileDMatrix(batches, max_bin=256, max_quantile_batches=8, enable_categorical=True)\n            params = BEST_PARAMS[str(h)].copy()\n            trees = params.pop(\'n_estimators\')\n            params.update(objective=OBJECTIVE, eval_metric=\'mae\', tree_method=\'hist\', device=\'cuda:0\', max_bin=256, max_cat_to_onehot=1, seed=SEED, nthread=THREADS)\n            model = xgb.train(params, quantile, num_boost_round=trees)\n            device = json.loads(model.save_config())[\'learner\'][\'generic_param\'][\'device\']\n            if not device.startswith(\'cuda\') or model.feature_types != batches.types:\n                raise RuntimeError(\'XGBoost clean потерял CUDA или категориальный route_id\')\n            del quantile, batches\n            cp.get_default_memory_pool().free_all_blocks()\n            train_prediction = predict_clean_xgb(model, matrix)\n            del matrix\n            query_matrix, query_columns = clean_xgb_matrix(h, query, features[\'horizon_dataset\'])\n            if columns != query_columns:\n                raise ValueError(\'Расходятся колонки train и проверки обычного XGBoost\')\n            prediction = predict_clean_xgb(model, query_matrix)\n            actual = observed_query(h).ravel()\n            fit_report(\'xgb_clean\', start, h, \'all\', split, VALUES[TARGET][train + h].ravel(), train_prediction, actual, prediction)\n            frame = pd.DataFrame(dict(cutoff=TIMES[query].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query)), horizon=h, y=actual, xgb_clean=prediction))\n            save_parquet(path, frame)\n            model.save_model(str(folder / f\'h{h}_model.ubj\'))\n            del model, query_matrix, train_prediction\n            gc.collect()\n            cp.get_default_memory_pool().free_all_blocks()\n    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)\n', 'clean_xgb_runtime', 'exec'), clean_xgb_runtime)


In [ ]:
clean_future = clean_xgb_runtime['fit_clean_xgb'](FULL_BOUNDARY, FULL_BOUNDARY + pd.Timedelta(days=1), 'future')


In [ ]:
import joblib
import torch
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
PAIRS = BASES[1:]
VOLUME_INPUTS = [name for name in PAIRS if name != 'xgb_clean']
SHAPES = ['session6', 'equal', 'joint', 'joint_without_xgb_clean', 'by_horizon', *[f'pair_{name}' for name in PAIRS], 'ridge_100', 'ridge_1000', 'cat_d2', 'cat_d4', 'xgb_d2', 'xgb_d3', 'gating_8', 'gating_16']
VOLUMES = ['fixed', 'ratio', 'ewma7', 'ridge']
META_DEVICE = 'cuda'

def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-05):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (0.2, 0.05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-09:
                        best = loss
                        choice = (source, destination, amount, trial)
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(log_level=np.log1p(level), current=frame.current.to_numpy() / level, delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level, log_network=np.log1p(frame.network_current.to_numpy()), horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(), shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return (context.clip(-50, 50).to_numpy(dtype=np.float32), level)

class GatingNetwork(torch.nn.Module):

    def __init__(self, width, experts, hidden, prior):
        super().__init__()
        self.layers = torch.nn.Sequential(torch.nn.Linear(width, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, experts))
        self.register_buffer('prior', torch.as_tensor(prior, dtype=torch.float32))
        torch.nn.init.zeros_(self.layers[-1].weight)
        torch.nn.init.zeros_(self.layers[-1].bias)

    def forward(self, context):
        return torch.softmax(self.prior.log() + self.layers(context), dim=1)

def fit_gating(x, experts, y, level, prior, hidden):
    torch.manual_seed(SEED)
    scaler = StandardScaler().fit(x)
    prior = (prior + 0.02) / (prior.sum() + 0.02 * len(prior))
    model = GatingNetwork(x.shape[1], experts.shape[1], hidden, prior).to(META_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    normalized = experts / level[:, None]
    target = np.asarray(y, dtype=np.float64) / level
    weights = level / level.mean()
    tensors = [torch.as_tensor(value, dtype=torch.float32, device=META_DEVICE) for value in (scaler.transform(x), normalized, target, weights)]
    tx, te, ty, tw = tensors
    for _ in range(15):
        permutation = torch.randperm(len(x), device=META_DEVICE)
        for rows in permutation.split(4096):
            gate = model(tx[rows])
            pred = (gate * te[rows]).sum(dim=1)
            loss = ((pred - ty[rows]).abs() * tw[rows]).mean() + 0.03 * (gate - model.prior).square().sum(dim=1).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return (scaler, model.cpu())

def fit_shape(name, frame, matrix, joint):
    if name == 'session6':
        return None
    if name == 'equal':
        return np.full(len(BASES), 1 / len(BASES))
    if name == 'joint':
        return joint
    if name == 'joint_without_xgb_clean':
        mask = np.array([base != 'xgb_clean' for base in BASES])
        weights = np.zeros(len(BASES))
        weights[mask] = fit_weights(matrix[:, mask], frame.y)
        return weights
    if name.startswith('pair_'):
        index = BASES.index(name.removeprefix('pair_'))
        weights = np.zeros(len(BASES))
        weights[[0, index]] = fit_weights(matrix[:, [0, index]], frame.y)
        return weights
    if name == 'by_horizon':
        weights = []
        for h in HORIZONS:
            mask = frame.horizon.eq(h).to_numpy()
            weights.append(0.5 * joint + 0.5 * fit_weights(matrix[mask], frame.y.to_numpy()[mask]))
        return np.stack(weights)
    baseline = matrix @ joint
    features, level = meta_features(frame, matrix, baseline)
    residual = (frame.y.to_numpy(dtype=np.float64) - baseline) / level
    sample_weight = level / level.mean()
    if name.startswith('gating_'):
        return fit_gating(features, matrix, frame.y, level, joint, int(name.split('_')[-1]))
    if name.startswith('ridge_'):
        prep = StandardScaler().fit(features)
        model = Ridge(alpha=float(name.split('_')[-1]))
        model.fit(prep.transform(features), residual, sample_weight=sample_weight)
        return (prep, model)
    depth = int(name[-1])
    if name.startswith('cat_'):
        model = CatBoostRegressor(depth=depth, iterations=220, loss_function='MAE', learning_rate=0.035, l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=THREADS)
    else:
        model = XGBRegressor(max_depth=depth, n_estimators=220, objective='reg:absoluteerror', tree_method='hist', learning_rate=0.035, reg_lambda=20, min_child_weight=30, random_state=SEED, n_jobs=THREADS)
    model.fit(features, residual, sample_weight=sample_weight)
    return (None, model)

def predict_shape(name, fitted, frame, matrix, joint):
    if name == 'session6':
        return matrix[:, 0]
    if isinstance(fitted, np.ndarray):
        if fitted.ndim == 1:
            return matrix @ fitted
        return np.sum(matrix * fitted[frame.horizon.to_numpy() - 1], axis=1)
    baseline = matrix @ joint
    prep, model = fitted
    features, level = meta_features(frame, matrix, baseline)
    if name.startswith('gating_'):
        weights = gate_weights(prep, model, features)
        return with_volume(frame, np.sum(matrix * weights, axis=1))
    encoded = features if prep is None else prep.transform(features)
    correction = np.clip(model.predict(encoded), -0.5, 0.5) * level
    return with_volume(frame, baseline + correction)

def gate_weights(prep, model, features):
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(prep.transform(features), dtype=torch.float32)).numpy()

def volume_table(frame, with_y=False):
    aggregations = {name: (name, 'sum') for name in BASES}
    aggregations['network_current'] = ('network_current', 'first')
    if with_y:
        aggregations['y'] = ('y', 'sum')
    return frame.groupby(GROUP_KEYS, as_index=False, sort=True).agg(**aggregations)

def volume_features(table):
    anchor = np.maximum(table.session6.to_numpy(), 1)
    features = np.column_stack([np.log1p(anchor), np.log1p(table.network_current), table.horizon, table.cutoff.dt.day_of_week, *[np.log((table[name].to_numpy() + 1) / (anchor + 1)) for name in VOLUME_INPUTS]])
    return features

def fit_volumes(frame):
    table = volume_table(frame, with_y=True)
    if table.session6.sum() <= 0:
        raise ValueError('Нулевой объём Session6')
    ages = (table.cutoff.max() - table.cutoff).dt.total_seconds().to_numpy() / 86400
    w = np.exp2(-ages / 7)
    ratio = table.y.sum() / table.session6.sum()
    recent_ratio = np.dot(table.y, w) / np.dot(table.session6, w)
    x = volume_features(table)
    prep = StandardScaler().fit(x)
    model = Ridge(alpha=100).fit(prep.transform(x), np.log((table.y + 1) / (table.session6 + 1)))
    return dict(fixed=1.0, ratio=float(1 + 0.5 * (ratio - 1)), ewma7=float(1 + 0.5 * (recent_ratio - 1)), ridge=(prep, model))

def volume_multipliers(frame, fitted):
    table = volume_table(frame)
    prep, model = fitted['ridge']
    fitted = dict(fitted, ridge=1 + 0.5 * np.expm1(model.predict(prep.transform(volume_features(table)))))
    return {name: frame[GROUP_KEYS].merge(table[GROUP_KEYS].assign(factor=np.clip(factor, 0.95, 1.05)), on=GROUP_KEYS, how='left', validate='many_to_one').factor.to_numpy() for name, factor in fitted.items()}

def fit_meta(train, query, phase, split):
    train_matrix, query_matrix = (shape_matrix(train), shape_matrix(query))
    joint = fit_weights(train_matrix, train.y)
    volume_models = fit_volumes(train)
    train_factors = volume_multipliers(train, volume_models)
    query_factors = volume_multipliers(query, volume_models)
    rows, fitted, predictions = ([], {}, {})
    for name in SHAPES:
        model = fit_shape(name, train, train_matrix, joint)
        fitted[name] = model
        train_prediction = predict_shape(name, model, train, train_matrix, joint)
        query_prediction = predict_shape(name, model, query, query_matrix, joint)
        local = []
        for volume in VOLUMES:
            label = f'{name}__{volume}'
            train_values = train_prediction * train_factors[volume]
            query_values = query_prediction * query_factors[volume]
            predictions[label] = query_values
            local.extend([dict(phase=phase, variant=label, shape=name, volume=volume, period=period, **metric(y, p)) for period, y, p in [('train_oof', train.y, train_values), (split, query.y, query_values)]])
        display(pd.DataFrame(local))
        rows.extend(local)
    return (pd.DataFrame(rows), dict(joint=joint, shapes=fitted, volumes=volume_models), predictions)

def fit_selection(oof):
    blocks = []
    for fold, (start, stop) in enumerate(OOF_WINDOWS[2:], 3):
        train = oof.loc[oof.cutoff.lt(start)]
        validation = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)]
        rows, _, _ = fit_meta(train, validation, f'val_week_{fold}', 'val')
        blocks.append(rows)
    metrics = pd.concat(blocks, ignore_index=True)
    metrics.to_csv(ARTIFACTS / 'selection_metrics.csv', index=False)
    ranking = metrics.loc[metrics.period.eq('val')].groupby(['variant', 'shape', 'volume'], as_index=False).agg(mean_val_score=('score', 'mean'), worst_val_score=('score', 'max'), weeks=('score', 'count'))
    ranking['order'] = ranking['shape'].map({name: i for i, name in enumerate(SHAPES)})
    ranking = ranking.sort_values(['mean_val_score', 'worst_val_score', 'order', 'variant'])
    if not ranking.weeks.eq(2).all():
        raise ValueError('В подборе должны присутствовать обе недели val')
    ranking.to_csv(ARTIFACTS / 'selection_ranking.csv', index=False)
    display(ranking.drop(columns='order').head(20))
    selected = ranking.iloc[0]
    frozen = dict(variant=selected.variant, shape=selected['shape'], volume=selected.volume, mean_val_score=float(selected.mean_val_score), test_used_for_selection=False, meta_fit_weeks=4, local_test_before_full_refit=True)
    write_json(ARTIFACTS / 'frozen_selection.json', frozen)
    return frozen

def evaluate(oof, test, frozen):
    rows, models, predictions = fit_meta(oof, test, 'frozen_local_test', 'test')
    base_rows = [dict(phase='base', variant=column, shape=column, volume='raw', period=period, **metric(frame.y, frame[column])) for period, frame in [('train_oof', oof), ('test', test)] for column in BASES]
    table = pd.concat([pd.DataFrame(base_rows), rows], ignore_index=True)
    table['selected_on_val'] = table.variant.eq(frozen['variant'])
    table.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
    predictions = pd.concat([test.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
    save_parquet(ARTIFACTS / 'test_predictions.parquet', predictions)
    columns = [*BASES, *[f'{name}__fixed' for name in SHAPES]]
    if frozen['variant'] not in columns:
        columns.append(frozen['variant'])
    slices = []
    for kind, key in [('day', predictions.cutoff), ('horizon', predictions.horizon), ('week', (predictions.cutoff.dt.normalize() - LOCAL_TEST_START).dt.days // 7 + 1)]:
        for value, group in predictions.groupby(key):
            slices.extend((dict(slice=kind, value=str(value), variant=name, **metric(group.y, group[name])) for name in columns))
    pd.DataFrame(slices).to_csv(ARTIFACTS / 'test_slices.csv', index=False)
    weights = []
    matrix = shape_matrix(test)
    features, _ = meta_features(test, matrix, matrix @ models['joint'])
    for name, model in models['shapes'].items():
        if isinstance(model, np.ndarray):
            if model.ndim == 1:
                weights.append(dict(variant=name, horizon='all', **dict(zip(BASES, model))))
            else:
                weights.extend((dict(variant=name, horizon=h, **dict(zip(BASES, row))) for h, row in zip(HORIZONS, model)))
        elif name.startswith('gating_'):
            prep, network = model
            frame = pd.DataFrame(gate_weights(prep, network, features), columns=BASES)
            means = frame.groupby(test.horizon.to_numpy()).mean()
            weights.extend((dict(variant=name, horizon=h, **row.to_dict()) for h, row in means.iterrows()))
            models['shapes'][name] = dict(prep=prep, state=network.state_dict(), width=features.shape[1], hidden=int(name.split('_')[-1]))
    pd.DataFrame(weights).to_csv(ARTIFACTS / 'blend_weights.csv', index=False)
    artifact = ARTIFACTS / 'evaluation_meta.joblib'
    temporary = artifact.with_suffix('.partial')
    joblib.dump(dict(frozen=frozen, bases=BASES, models=models), temporary)
    temporary.replace(artifact)
    comparison = ['joint__fixed', 'joint_without_xgb_clean__fixed', 'pair_xgb_clean__fixed']
    display(table.loc[table.selected_on_val | table.variant.isin([*BASES, *comparison])])
    note = 'Независимая от полного переобучения оценка test получена до переобучения. Локальный период 18–31 октября уже изучался. Выбор сделан на двух более ранних неделях. Новые метамодели обучены только на OOF до test. Session6 на OOF восстановлен по рецепту, а на test взят из архива решения 0.326. Полное переобучение и конкурсный сабмит не выполняются.'
    (ARTIFACTS / 'evaluation_note.txt').write_text(note)
    print(note)


In [ ]:
finish_submission(dict(session6=session_future, rf=rf_future, cat_rich=cat_future, xgb_rich=xgb_future, xgb_clean=clean_future))


In [ ]:
for name in ['session_runtime', 'cat_runtime', 'rf_runtime', 'clean_xgb_runtime', 'xgb_rich_runtime']:
    globals().pop(name, None)
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
